In [1]:
import matplotlib.pyplot as plt
import numpy as np
from astropy.io import fits
from astropy.table import Table
import os
import pyarrow.parquet as pq
import ipympl
from scipy.optimize import curve_fit
import warnings
from astropy.utils.exceptions import AstropyUserWarning
from astropy import units as u
import pyneb as pn
from astropy.stats import median_absolute_deviation as mad
# To resize the color bar same as the maps
from mpl_toolkits.axes_grid1 import make_axes_locatable
from scipy.interpolate import interp1d
from astropy.coordinates import SkyCoord

# Making x and y ticks more readable
from matplotlib.ticker import MaxNLocator   

from matplotlib.patches import Circle

In [2]:
#type = 'multi'
#data_dir = '/Users/amritasingh/amrita/LVM/LVM_lagoon_outputs/1.1.2dev0_outputs/resolved/'
#
#plotdir = f'{data_dir}july8_25/linefits/'
#rss = data_dir+f'july8_25/spaxel_weighted_rss_1.1.2dev0_jul8.fits'
#outfilename = f'{data_dir}july8_25/rss_obs_flux_table_1.1.2dev0_jul8_spax_weighted_velocity_corrected.fits'


In [3]:
warnings.filterwarnings('ignore', category=AstropyUserWarning)

In [4]:
os.environ["OMP_NUM_THREADS"] = "1"
os.environ["MKL_NUM_THREADS"] = "1"

In [5]:
'''
Single, double and triple Gaussian function.
inputs: wave: wavlength
flux: flux
mean: rest frame wavelength
sd: standard dev.
cont: continuum

Output: Gaussian fitting function calculated for given parameters
'''

# Gaussian + quadratic continuum
def gauss_plus_quad(wave, flux, mean, sd, c0, c1, c2):
    cont = c0 + c1 * (wave - mean) + c2 * (wave - mean)**2
    return cont + flux / (np.sqrt(2 * np.pi) * sd) * np.exp(-(wave - mean) ** 2 / (2 * sd ** 2))


#single gaussian function with 4 input params
def gaussian(wave, flux, mean, sd, cont):

    return cont + flux / (np.sqrt(2 * np.pi) * sd) * np.exp(-(wave - mean) ** 2 / (2 * sd ** 2))

# ==================================================== Double gaussians ===================================================================#
#Double gaussian function with 7 input params, linear continua
def double_gaussian(wave, flux1, mean1, sd1, flux2, mean2, cont):
    
    return (cont + gaussian(wave, flux1, mean1, sd1, 0) +
            gaussian(wave, flux2, mean2, sd1, 0))


# fixing 4363 vel shift wrt fe, quad continua, october 8th, 25

#Double gaussian function with 6 input params
def double_gaussian_quad_cont_vel_shift(wave, flux1, sd1, flux2, c0, c1, c2, vel_shift_hgm=60):
    c = 299792.458

    # 4363 shifted accordingly
    mean1 = 4359.340 * (1 + vel_shift_hgm / c)
    mean2 = 4363.209 * (1 + vel_shift_hgm / c)

    # --- continuum 
    wave_ref = mean2  
    cont = c0 + c1 * (wave - wave_ref) + c2 * (wave - wave_ref)**2

    
    return (cont +
             gaussian(wave, flux1, mean1, sd1, 0) +
             gaussian(wave, flux2, mean2, sd1, 0))


#Double gaussian function with 7 input params, quad continua
def double_gaussian_quad(wave, flux1, mean1, sd1, flux2, mean2, c0, c1, c2):
    cont = c0 + c1*(wave - mean1) + c2*(wave - mean1)**2

    return (cont + gaussian(wave, flux1, mean1, sd1, 0) +
            gaussian(wave, flux2, mean2, sd1, 0))



# ==================================================================================================================================================#

#Triple gaussian function with 10 input params
def triple_gaussian(wave, flux1, mean1, sd1, vel_shift, flux2, flux3, cont):

    lw = [0, 9.19, 21.74]
    c = 299792.458  # Speed of light in km/s
    observed_wavelengths = (mean1 + np.array(lw)) * (1 + vel_shift / c)
    
    return (cont +
            gaussian(wave, flux1, observed_wavelengths[0], sd1, 0) +
            gaussian(wave, flux2, observed_wavelengths[1], sd1, 0) +
            gaussian(wave, flux3, observed_wavelengths[2], sd1, 0) )

# Fitting 10 gaussians to OII RLs
def ten_gaussian(wave, flux1, line1, sd1, vel_shift, flux2, flux3, flux4, flux5, flux6, flux7, flux8, flux9, flux10, cont):
    
    lw = [0, 14.19, 23.38, 31.69, 34.65, 35.93, 41.97, 43.68, 51.01, 54.47, 69.07]
    
    c = 299792.458  # Speed of light in km/s
    observed_wavelengths = (line1 + np.array(lw)) * (1 + vel_shift / c)

    return (cont +
            gaussian(wave, flux1, observed_wavelengths[0], sd1, 0) +
            gaussian(wave, flux2, observed_wavelengths[1], sd1, 0) +
            gaussian(wave, flux3, observed_wavelengths[2], sd1, 0) +
            gaussian(wave, flux4, observed_wavelengths[3], sd1, 0) +
            gaussian(wave, flux5, observed_wavelengths[4], sd1, 0) +
            gaussian(wave, 1.37*flux1, observed_wavelengths[5], sd1, 0) +
            gaussian(wave, flux6, observed_wavelengths[6], sd1, 0) +
            gaussian(wave, flux7, observed_wavelengths[7], sd1, 0)+
            gaussian(wave, flux8, observed_wavelengths[8], sd1, 0) +
            gaussian(wave, flux9, observed_wavelengths[9], sd1, 0) +
            gaussian(wave, flux10, observed_wavelengths[10], sd1, 0))

# 10 gaussian fits with quadratic continua
def ten_gaussian_quad(wave, flux1, line1, sd1, vel_shift, flux2, flux3, flux4, flux5, flux6, flux7, flux8, flux9, flux10, c0, c1, c2):
    
    lw = [0, 14.19, 23.38, 31.69, 34.65, 35.93, 41.97, 43.68, 51.01, 54.47, 69.07]
    
    c = 299792.458  # Speed of light in km/s
    observed_wavelengths = (line1 + np.array(lw)) * (1 + vel_shift / c)

    cont = c0 + c1 * (wave - observed_wavelengths[6]) + c2 * (wave - observed_wavelengths[6])**2

    return (cont +
            gaussian(wave, flux1, observed_wavelengths[0], sd1, 0) +
            gaussian(wave, flux2, observed_wavelengths[1], sd1, 0) +
            gaussian(wave, flux3, observed_wavelengths[2], sd1, 0) +
            gaussian(wave, flux4, observed_wavelengths[3], sd1, 0) +
            gaussian(wave, flux5, observed_wavelengths[4], sd1, 0) +
            gaussian(wave, 1.37*flux1, observed_wavelengths[5], sd1, 0) +
            gaussian(wave, flux6, observed_wavelengths[6], sd1, 0) +
            gaussian(wave, flux7, observed_wavelengths[7], sd1, 0)+
            gaussian(wave, flux8, observed_wavelengths[8], sd1, 0) +
            gaussian(wave, flux9, observed_wavelengths[9], sd1, 0) +
            gaussian(wave, flux10, observed_wavelengths[10], sd1, 0))



In [6]:
'''
Function to fit single Gaussian to emission lines 

Inputs: 
wave: wavelength 
spectrum: flux 
error: error 
lwave: centroid wavlength 
continuum_rangea: an array of wavelengths to calculate continuum 
dwave: difference in wavelength
plot: True/False to produce plots of linefits(boolean) 
plotout: plot name

Outputs:
Returns optimal parameters and covariance matrix on fitting the emission lines
'''

def fit_gauss(wave, spectrum, error, lwave, dwave=6, plot=False, plotout='linefit'):

    # Making a selection mask on wavelength windows,finite flux and error values 
    sel = (wave > lwave - dwave / 2) * (wave < lwave + dwave / 2) * (np.isfinite(spectrum)) * (np.isfinite(error)) 
    sel_cont = (wave > lwave - dwave - 2 / 2) * (wave < lwave + dwave+2 / 2) * (np.isfinite(spectrum)) * (np.isfinite(error)) 

    sel2 = (wave > lwave - dwave / 2) & (wave < lwave + dwave / 2) & np.isfinite(spectrum) & np.isfinite(error)
    
    if not np.any(sel2):
        raise ValueError(f"Empty selection for Gaussian fit at line {lwave}")

    # Initial params for the Gaussian fit
    #init_flux = np.abs(spectrum[sel].max() - spectrum[sel].min()) * np.sqrt(2 * np.pi) * 0.7
    init_flux = np.abs(spectrum[sel].max() - spectrum[sel_cont].min()) * np.sqrt(2 * np.pi) * 0.7
    init_cont = spectrum[sel_cont].min()
    
    p0 = (init_flux, lwave, 0.7, init_cont)

    try:

        sel_goodpix = np.isfinite(spectrum)*np.isfinite(error)
        
        # calculating optimal parameters and covariance matrix
        popt, pcov = curve_fit(gaussian, wave[sel_goodpix], spectrum[sel_goodpix], sigma=error[sel_goodpix], p0=p0, absolute_sigma=True, bounds=([0, lwave - dwave, 0.5, -np.inf], (np.inf, lwave + dwave, 1.0, np.inf)))
        
    except RuntimeError:

        popt = np.array([-99, p0[1], p0[2], p0[3]])
        pcov = np.zeros((4, 4))
        pcov[0, 0] = np.sum(error** 2)

    nparam=len(popt) #no. of parameters
    xm = wave
    ym = gaussian(xm, *popt)
    
    # calculating weighted chi**2 value
    chi2=np.nansum((spectrum-ym)**2/error**2)/(len(spectrum)-nparam) 

    if plot==True:

        #xm=np.arange(wave[sel][0], wave[sel][-1], 0.01)
        #sigma = get_gaussian_ci(xm, gaussian, popt, pcov)
        fig = plt.figure(figsize = (10, 6))
        ax = fig.add_subplot(111)

        plt.rcParams.update({'axes.titlesize': 'Large',
                 'axes.labelsize':'Large',
                 'axes.linewidth':     '1.8' ,
                 'ytick.labelsize': 'Large',
                 'xtick.labelsize': 'Large',
                 'font.size': '12.0',
                 'legend.fontsize':'small'})

        # Plotting
        ax.errorbar(wave, spectrum, error, c='k', label='data')
        ax.scatter(wave, spectrum, c='b', label='data')
        ax.plot(xm, ym, c='r', label='model')
        #ax.plot(xm, gaussian(xm, *p0), c='orange', label='initial approx. fit')

        #ax.set_title(fr'F={popt[0]:.2f}e-17 ; $\lambda_0$={popt[1]:.2f} ; $\sigma=${popt[2]:.2f} ; C={popt[3]:.2f}e-17 ; $\chi^2=${chi2:.2f}')

        #ax.fill_between(xm, ym+1*sigma, np.max([ym-ym,ym-1*sigma], axis=0), alpha=0.7,         
        #            linewidth=3, label='error limits')
        
        #ax.set_ylim(spectrum[sel].min(),1.7*np.max([spectrum[sel].max(), (ym+3*sigma).max()]))
        
        ax.set_ylim(np.nanmin([0,(spectrum-error).min()]), 1.7*np.nanmax([spectrum.max(), ym.max()]))
        ax.set_xlim(lwave-dwave*3.5, lwave+dwave*4.5)       
        ax.set_xlabel(r'Wavelength $\AA$')
        ax.set_ylabel(r'Flux $(10^{-17} \times erg  s^{-1}  cm^{-2}  \AA^{-1}\, arcsec^{-2})$')   
        
        ax.legend()
        
        plt.savefig(f"{plotdir}{plotout}.png", dpi =100)
        

    return popt, pcov

In [7]:
def fit_gauss_quad(wave, spectrum, error, lwave, dwave=8, plot = True, plotout='linefit_quad', bin ='1'):
   
    # Making a selection mask on wavelength windows,finite flux and error values 
    sel = (wave > lwave - dwave / 2) * (wave < lwave + dwave / 2) * (np.isfinite(spectrum)) * (np.isfinite(error)) 

    # Check if the selection is valid
    if not np.any(sel):
        print(f"Warning: no valid data in window for line {lwave}")
        return np.array([-99, lwave, 0.7, 0.0]), np.zeros((4, 4))


    # Initial params for the Gaussian fit
    init_flux = np.abs(spectrum[sel].max() - spectrum[sel].min()) * np.sqrt(2 * np.pi) * 0.7
    c0, c1, c2 = np.nanmin(spectrum[sel]), 0, 0
    
    p0 = (init_flux, lwave, 0.7, c0, c1, c2)
    bounds = ([0, lwave-1, 0.5, -np.inf, -np.inf, -np.inf],[np.inf, lwave+1, 0.9, np.inf, np.inf, np.inf])

    try:

        sel_goodpix = np.isfinite(spectrum)*np.isfinite(error)*np.isfinite(wave)

    # calculating optimal parameters and covariance matrix
        popt, pcov = curve_fit(gauss_plus_quad, wave[sel_goodpix], spectrum[sel_goodpix], sigma=error[sel_goodpix], p0=p0, absolute_sigma=True,
                               bounds=bounds)
        
    except RuntimeError:
        popt = np.array([-99, p0[1], p0[2], p0[3]])
        pcov = np.zeros((4, 4))
    
        pcov[0, 0] = np.sum(error** 2)

    
    nparam=len(popt) #no. of parameters
    xm = wave
    ym = gauss_plus_quad(xm, *popt)
    
    # calculating weighted chi**2 value
    chi2=np.nansum((spectrum-ym)**2/error**2)/(len(spectrum)-nparam) 

    if plot==True:

        plt.rcParams.update({'axes.titlesize': 'Large',
                 'axes.labelsize':'Large',
                 'axes.linewidth':     '1.8' ,
                 'ytick.labelsize': '14.0',
                 'xtick.labelsize': '14.0',
                 'font.size': '14.0',
                 'legend.fontsize':'small'})
        
        fig = plt.figure(figsize = (8, 6))
        ax = fig.add_subplot(111)

        # Plotting
        #ax.step(wave, spectrum, alpha=0.5, label='steps')
        ax.errorbar(wave, spectrum, error, c='k', label='data')
        ax.scatter(wave, spectrum, c='b', label='data')
        ax.plot(xm, ym, c='r', label='model')

        #ax.set_ylim(spectrum[sel].min(),1.7*np.max([spectrum[sel].max(), (ym+3*sigma).max()]))
        ax.set_ylim(np.min([0,(spectrum[sel_goodpix]).min()]), 1.7*np.max([spectrum[sel_goodpix].max(), ym.max()]))
        ax.set_xlim(lwave-dwave*2, lwave+dwave*2)       
        ax.set_xlabel(r'Wavelength $\AA$')
        ax.set_ylabel(r' SB$(10^{-17} \times erg\, s^{-1} \,cm^{-2}\, \AA^{-1}\, arcsec^{-2})$')   
        ax.set_title(f'{lwave:.0f} line fit, {bin}')
        ax.legend()
        plt.tight_layout()

        plt.savefig(f'{plotdir}{plotout}_quad_cont_test.png', dpi =100)
        plt.close()

    return popt, pcov# calculating optimal parameters and covariance matrix


In [8]:
'''
Function to fit double Gaussian to emission lines (for [OII]3726, 3729, [OIII]4363 and [FeII]4360)

Inputs: 
wave: wavelength 
spectrum: flux 
error: error 
lwave1: centroid wavlength of first line
lwave2: centroid wavlength of second line
dwave: difference in wavelength
plot: True/False to produce plots of linefits(boolean) 
plotout: plot name

Outputs:
Returns optimal parameters and covariance matrix on fitting the emission lines
'''

def fit_double_gauss(wave, spectrum, error, lwave1, lwave2, dwave=6, plot=True, plotout='linefit'):
    
    # Making a selection masks on wavelength windows for each rest frame wavelength, finite flux and error values 
    sel1 = (wave > lwave1 - dwave / 2) * (wave < lwave1 + dwave / 2) * (np.isfinite(spectrum)) * (np.isfinite(error)) 
    sel2 = (wave > lwave2 - dwave / 2) * (wave < lwave2 + dwave / 2) * (np.isfinite(spectrum)) * (np.isfinite(error))
    sel_cont = (wave > lwave1 - dwave ) * (wave < lwave2 + dwave ) * (np.isfinite(spectrum)) * (np.isfinite(error))
    
    #if sel1.sum() == 0 or sel2.sum() == 0 or sel_cont.sum() == 0:
    #    print(f"[WARNING] Skipping fit: No data in one of the regions (sel1={sel1.sum()}, sel2={sel2.sum()}, sel_cont={sel_cont.sum()})")
    #    continue

    #initial parameters for the Gaussian fit
    p0 = (np.abs(spectrum[sel1].max()- spectrum[sel1].min()) * np.sqrt(2 * np.pi) * 0.7, lwave1, 0.7,  
          np.abs(spectrum[sel2].max()- spectrum[sel2].min()) * np.sqrt(2 * np.pi) * 0.7, lwave2, spectrum[sel_cont].min())
     
    try:

        sel_goodpix = np.isfinite(spectrum)*np.isfinite(error)
        
        # calculating optimal parameters and covariance matrix
        popt, pcov = curve_fit(double_gaussian, wave[sel_goodpix], spectrum[sel_goodpix], sigma=error[sel_goodpix], p0=p0, absolute_sigma=True,
                               bounds=((0, lwave1 - dwave, 0.5, -1, lwave2 - dwave,  -np.inf),
                                       (np.inf, lwave1 + dwave, 1.0, np.inf, lwave2 + dwave, np.inf)))
    except RuntimeError:
        print('Runtime error')
        popt = np.array([-1, lwave1, 0.7, -1, lwave2, spectrum[sel_cont].min()])
        pcov = np.zeros((7, 7))
        pcov[0, 0] = np.sum(error ** 2)

    
    nparam=len(popt) #no. of parameters
    xm = wave
    ym = double_gaussian(xm, *popt)

    # calculating weighted chi**2 value
    chi2=np.nansum((spectrum-ym)**2/error**2)/(len(spectrum)-nparam)

    if plot==True:

        xm = wave
        #sigma = get_gaussian_ci(xm, gaussian, popt, pcov)
        fig = plt.figure(figsize = (10, 6))
        ax = fig.add_subplot(111)
        
        plt.rcParams.update({'axes.titlesize': 'Large',
                 'axes.labelsize':'Large',
                 'axes.linewidth':     '1.8' ,
                 'ytick.labelsize': '20.0',
                 'xtick.labelsize': '20.0',
                 'font.size': '20.0',
                 'legend.fontsize':'small'})

        #Final plotting
        #Plotting final output
        ax.errorbar(wave, spectrum, error, c='k', label='data')
        ax.scatter(wave, spectrum, c='b', label='data')
        ax.plot(xm, ym, c='r', label='model')
        ax.step(wave, spectrum, alpha=0.5, label='steps')
        #ax.plot(xm, double_gaussian(xm, *p0), c='orange', label='initial approx. fit')

        #ax.set_title(fr'F0={popt[0]:.2f}e-13 ; $\lambda_0$={popt[1]:.2f} ; $\sigma_0=${popt[2]:.2f} ; F1={popt[3]:.2f}e-13 ; $\lambda_1$={popt[4]:.2f} ; $\sigma_1=${popt[5]:.2f} ; C={popt[6]:.2f}e-13 ; $\chi^2=${chi2:.2f}')

        ax.set_ylim(np.min([0,(spectrum-error)[sel1].min()]), 1.8*np.max([(spectrum-error)[sel1].max(), ym.max()]))
        ax.set_xlim(lwave1-dwave*2, lwave2+dwave*2)
        ax.set_xlabel(r'Wavelength $\AA$')
        ax.set_ylabel(r'Flux $(10^{-17} \times erg  s^{-1}  cm^{-2}  \AA^{-1} arcsec^{-2}$)')
        ax.legend()
       
        #Save the figure
        plt.savefig(f"{plotdir}{plotout}.png", dpi =100)
        
    return popt, pcov

In [9]:
'''
Function to fit double Gaussian to emission lines (for [OII]3726, 3729, [OIII]4363 and [FeII]4360)

Inputs: 
wave: wavelength 
spectrum: flux 
error: error 
lwave1: centroid wavlength of first line
lwave2: centroid wavlength of second line
dwave: difference in wavelength
plot: True/False to produce plots of linefits(boolean) 
plotout: plot name

Outputs:
Returns optimal parameters and covariance matrix on fitting the emission lines
'''


def fit_double_gauss_quad_vel_shift(wave, spectrum, error, lwave1, lwave2, vel = 60, plot=True, plotout='linefit'):

    dwave=4


    # Making a selection masks on wavelength windows for each rest frame wavelength, finite flux and error values 
    sel1 = (wave > lwave1 - dwave / 2) * (wave < lwave1 + dwave / 2) * (np.isfinite(spectrum)) * (np.isfinite(error)) 
    sel2 = (wave > lwave2 - dwave / 2) * (wave < lwave2 + dwave / 2) * (np.isfinite(spectrum)) * (np.isfinite(error))
    sel = (wave > lwave1 - dwave / 2) * (wave < lwave2 + dwave / 2) * (np.isfinite(spectrum)) * (np.isfinite(error))
    c0, c1, c2 = np.nanmin(spectrum[sel]), 0.0, 0.0

    #initial parameters for the Gaussian fit
    #p0 = (np.abs(spectrum[sel1].max()- spectrum[sel1].min()) * np.sqrt(2 * np.pi) * 0.7, lwave1, 0.7, 
    #      np.abs(spectrum[sel2].max()- spectrum[sel2].min()) * np.sqrt(2 * np.pi) * 0.7, c0, c1, c2)

    p0 = (0, lwave1, 0.7, 0, c0, c1, c2)
     
    p0 = (0, 0.7, 0, c0, c1, c2)
     
    try:
        sel_goodpix = np.isfinite(spectrum)*np.isfinite(error)*np.isfinite(wave)
        wave, spectrum, error = wave[sel_goodpix], spectrum[sel_goodpix], error[sel_goodpix]


        # defining bounds
        bounds = ([-np.inf, 0.5, -np.inf, -np.inf, -np.inf, -np.inf],
                [np.inf, 0.75, np.inf, np.inf, np.inf, np.inf])

        # calculating optimal parameters and covariance matrix
        popt, pcov, infodict, mesg, ier = curve_fit(double_gaussian_quad_cont_vel_shift, wave, spectrum, sigma=error, p0=p0, absolute_sigma=True,
            bounds=bounds, full_output=True, xtol=None)
        
        
    except RuntimeError as e:
        popt = np.array([-99, 0.7, -99,-99, c1, c2])
        pcov = np.zeros((7, 7))
        pcov[0, 0] = np.sum(error ** 2)
        infodict, mesg, ier = {}, str(e), -1

    
    nparam=len(popt) #no. of parameters
    xm = wave
    ym = double_gaussian_quad_cont_vel_shift(xm, *popt, vel)

    # calculating weighted chi**2 value
    chi2=np.nansum((spectrum-ym)**2/error**2)/(len(spectrum)-nparam)

    if plot==True:

        xm = wave

        plt.rcParams.update({'axes.titlesize': 'X-Large',
                 'axes.labelsize':'X-Large',
                 'axes.linewidth':     '1.8' ,
                 'ytick.labelsize': '16.0',
                 'xtick.labelsize': '16.0',
                 'font.size': '16.0',
                 'legend.fontsize':'small'})
        
        fig = plt.figure(figsize = (12, 8))
        ax = fig.add_subplot(111)
 
        #Final plotting
        ax.errorbar(wave, spectrum, error, c='k', label='Error bar')
        ax.scatter(wave, spectrum, c='b', label='Data')
        ax.plot(xm, ym, c='r', label='Model')

        ax.set_title(fr' F1={popt[2]:.2f}e-17; F1 err={np.sqrt(pcov[2,2]):.3f}e-17 ; $\sigma_1=${popt[1]:.2f} ; c0, c1, c2: {popt[3]:.2f}e-17, {popt[4]:.3f}e-17, {popt[5]:.3f}e-17', fontsize=12) #C={popt[6]:.2f}e-13 ; $\chi^2=${chi2:.2f}, 
        
        ax.set_ylim(np.min([0,(spectrum-error).min()]), 1.7*np.max([spectrum.max(), ym.max()]))
        ax.set_xlim(lwave1-dwave*3, lwave2+dwave*4)
        ax.set_xlabel(r'Wavelength $\AA$')
        ax.set_ylabel(r' SB$(10^{-17} \times erg \,s^{-1}\, cm^{-2}\, \AA^{-1} \, arcsec^{-2})$')
        ax.legend()
        plt.tight_layout()
        
        #Save the figure
        plt.savefig(f"{plotdir}{plotout}_quad_cont_vel_tied_to_5007.png", dpi =100)
        
    return popt, pcov, infodict, mesg, ier 

In [10]:
'''
Function to fit double Gaussian to emission lines (for [OII]3726, 3729, [OIII]4363 and [FeII]4360)

Inputs: 
wave: wavelength 
spectrum: flux 
error: error 
lwave1: centroid wavlength of first line
lwave2: centroid wavlength of second line
dwave: difference in wavelength
plot: True/False to produce plots of linefits(boolean) 
plotout: plot name

Outputs:
Returns optimal parameters and covariance matrix on fitting the emission lines
'''

def fit_double_gauss_quad(wave, spectrum, error, lwave1, lwave2, plot=True, plotout='linefit'):

    dwave=4

    # Making a selection masks on wavelength windows for each rest frame wavelength, finite flux and error values 
    sel1 = (wave > lwave1 - dwave / 2) * (wave < lwave1 + dwave / 2) * (np.isfinite(spectrum)) * (np.isfinite(error)) 
    sel2 = (wave > lwave2 - dwave / 2) * (wave < lwave2 + dwave / 2) * (np.isfinite(spectrum)) * (np.isfinite(error))
    sel = (wave > lwave1 - dwave / 2) * (wave < lwave2 + dwave / 2) * (np.isfinite(spectrum)) * (np.isfinite(error))
    c0, c1, c2 = np.nanmin(spectrum[sel]), 0.0, 0.0

    #initial parameters for the Gaussian fit
    p0 = (np.abs(spectrum[sel1].max()- spectrum[sel1].min()) * np.sqrt(2 * np.pi) * 0.7, lwave1, 0.7, 
          np.abs(spectrum[sel2].max()- spectrum[sel2].min()) * np.sqrt(2 * np.pi) * 0.7, lwave2, c0, c1, c2)
     
    try:
        
        sel_goodpix = np.isfinite(spectrum)*np.isfinite(error)*np.isfinite(wave)
        wave, spectrum, error = wave[sel_goodpix], spectrum[sel_goodpix], error[sel_goodpix]

        # defining bounds
        bounds = ([0, lwave1 - 1, 0.5, 0, lwave2 - 1, -np.inf, -np.inf, -np.inf],
                [np.inf, lwave1 + 1,  0.9, np.inf, lwave2 + 1, np.inf, np.inf, np.inf])

        # calculating optimal parameters and covariance matrix
        popt, pcov = curve_fit(double_gaussian_quad, wave, spectrum, sigma=error, p0=p0, absolute_sigma=True,
                               bounds=bounds)
        
    except RuntimeError:
        popt = np.array([-99, lwave1, -99, lwave2, 0.7, -99, c1, c2])
        pcov = np.zeros((7, 7))
        pcov[0, 0] = np.sum(error ** 2)

    
    nparam=len(popt) #no. of parameters
    xm = wave
    ym = double_gaussian_quad(xm, *popt)

    # calculating weighted chi**2 value
    chi2=np.nansum((spectrum-ym)**2/error**2)/(len(spectrum)-nparam)

    if plot==True:

        xm = wave

        plt.rcParams.update({'axes.titlesize': 'X-Large',
                 'axes.labelsize':'X-Large',
                 'axes.linewidth':     '1.8' ,
                 'ytick.labelsize': '16.0',
                 'xtick.labelsize': '16.0',
                 'font.size': '16.0',
                 'legend.fontsize':'small'})
        
        fig = plt.figure(figsize = (12, 8))
        ax = fig.add_subplot(111)
 
        #Final plotting
        ax.errorbar(wave, spectrum, error, c='k', label='Error bar')
        ax.scatter(wave, spectrum, c='b', label='Data')
        ax.plot(xm, ym, c='r', label='Model')

        ax.set_title(fr' F1={popt[2]:.2f}e-17; F1 err={np.sqrt(pcov[2, 2]):.3f}e-17 ; $\lambda_1$={popt[3]:.2f} ; $\sigma_1=${popt[4]:.2f} ; resi:{np.nanmedian(ym-spectrum):.3f}', fontsize=12) #C={popt[6]:.2f}e-13 ; $\chi^2=${chi2:.2f}, 
        ax.set_ylim(np.min([0,(spectrum-error).min()]), 1.7*np.max([spectrum.max(), ym.max()]))
        ax.set_xlim(lwave1-dwave*3, lwave2+dwave*4)
        ax.set_xlabel(r'Wavelength $\AA$')
        ax.set_ylabel(r' SB$(10^{-17} \times erg \,s^{-1}\, cm^{-2}\, \AA^{-1} \, arcsec^{-2})$')
        ax.legend()
        plt.tight_layout()
        
        #Save the figure
        plt.savefig(f'{plotdir}{plotout}_quad_cont_test.png', dpi =100)

    return popt, pcov

In [11]:
'''
Function to fit 10 Gaussian to emission lines

Inputs: 
wave: wavelength 
spectrum: flux 
error: error 
lwave1: centroid wavlength of first line
lwave2: centroid wavlength of second line
lwave3: centroid wavlength of third line
dwave: difference in wavelength
plot: True/False to produce plots of linefits(boolean) 
plotout: plot name

Outputs:
Returns optimal parameters and covariance matrix on fitting the emission lines
'''

def fit_ten_gauss(wave, spectrum, error, lwave1, dwave=4, plot=False, plotout='linefit'):
    
    lw = [0, 14.19, 23.38, 31.69, 34.65, 35.93, 41.97, 43.68, 51.01, 54.47, 69.07]

    sel_goodpix = np.isfinite(spectrum)*np.isfinite(error)
    spectrum = spectrum[sel_goodpix]
    error = error[sel_goodpix]
    wave = wave[sel_goodpix]

    #cond = spectra>0
    #spectrum = spectra[cond]
    #wave = wav[cond]
    #error = err[cond]
    #print(f"spectrum.shape: {spectrum.shape}")

    #making a selection on wavelength range, and finite flux and error values
    sel1 = (wave > (lwave1+lw[0]) - dwave / 2) * (wave < (lwave1+lw[0])+ dwave / 2) * (np.isfinite(spectrum)) * (np.isfinite(error))  
    sel2 = (wave > (lwave1+lw[1]) - dwave / 2) * (wave < (lwave1+lw[1])+ dwave / 2) * (np.isfinite(spectrum)) * (np.isfinite(error))
    sel3 = (wave > (lwave1+lw[2]) - dwave / 2) * (wave < (lwave1+lw[2])+ dwave / 2) * (np.isfinite(spectrum)) * (np.isfinite(error))
    sel4 = (wave > (lwave1+lw[3]) - dwave / 2) * (wave < (lwave1+lw[3])+ dwave / 2) * (np.isfinite(spectrum)) * (np.isfinite(error))
    sel5 = (wave > (lwave1+lw[4]) - dwave / 2) * (wave < (lwave1+lw[4])+ dwave / 2) * (np.isfinite(spectrum)) * (np.isfinite(error))
    sel6 = (wave > (lwave1+lw[6]) - dwave / 2) * (wave < (lwave1+lw[6])+ dwave / 2) * (np.isfinite(spectrum)) * (np.isfinite(error))
    sel7 = (wave > (lwave1+lw[7]) - dwave / 2) * (wave < (lwave1+lw[7])+ dwave / 2) * (np.isfinite(spectrum)) * (np.isfinite(error))
    sel8 = (wave > (lwave1+lw[8]) - dwave / 2) * (wave < (lwave1+lw[8])+ dwave / 2) * (np.isfinite(spectrum)) * (np.isfinite(error))
    sel9 = (wave > (lwave1+lw[9]) - dwave / 2) * (wave < (lwave1+lw[9])+ dwave / 2) * (np.isfinite(spectrum)) * (np.isfinite(error))
    sel10 = (wave > (lwave1+lw[10]) - dwave / 2) * (wave < (lwave1+lw[10])+ dwave / 2) * (np.isfinite(spectrum)) * (np.isfinite(error))

    #sel_cont = (wave > lwave1 - dwave / 2) * (wave < lwave1+lw[10] + dwave ) * (np.isfinite(spectrum)) * (np.isfinite(error))

    # Initial paramets for the Gaussian fitting
    p0 = (np.abs(spectrum[sel1].max()- spectrum[sel1].min()) * np.sqrt(2 * np.pi) * 0.7,  lwave1, 0.7,  40, 
          np.abs(spectrum[sel2].max()- spectrum[sel2].min()) * np.sqrt(2 * np.pi) * 0.7,  
          np.abs(spectrum[sel3].max()- spectrum[sel3].min()) * np.sqrt(2 * np.pi) * 0.7,  
          np.abs(spectrum[sel4].max()- spectrum[sel4].min()) * np.sqrt(2 * np.pi) * 0.7,  
          np.abs(spectrum[sel5].max()- spectrum[sel5].min()) * np.sqrt(2 * np.pi) * 0.7,  
          np.abs(spectrum[sel6].max()- spectrum[sel6].min()) * np.sqrt(2 * np.pi) * 0.7,  
          np.abs(spectrum[sel7].max()- spectrum[sel7].min()) * np.sqrt(2 * np.pi) * 0.7,
          np.abs(spectrum[sel8].max()- spectrum[sel8].min()) * np.sqrt(2 * np.pi) * 0.7,
          np.abs(spectrum[sel9].max()- spectrum[sel9].min()) * np.sqrt(2 * np.pi) * 0.7,
          np.abs(spectrum[sel10].max()- spectrum[sel10].min()) * np.sqrt(2 * np.pi) * 0.7,  spectrum.min())

    bounds = ((0, lwave1 - dwave, 0.5, -np.inf, 0, 0, 0, 0, 0, 0, 0, 0, 0, -np.inf), 
            (np.inf, lwave1+ dwave, 0.9, np.inf, np.inf, np.inf, np.inf, np.inf, np.inf, np.inf,
              np.inf, np.inf, np.inf, np.inf))
    
    try:

        # Calculating optimal parmeters and covariance matrix
        popt, pcov = curve_fit(ten_gaussian, wave, spectrum, sigma=error, p0=p0, absolute_sigma=True, 
                               bounds=bounds)
        
    except RuntimeError:
        print('Calculating popt and pcov in RuntimeError condition')
        popt = np.array([-99, lwave1, 0.7, -99, -99, -99, -99, -99, -99, -99, -99, -99, -99, spectrum.min()])
        pcov = np.zeros((14, 14))
        pcov[0, 0] = np.sum(error ** 2)

    nparam=len(popt)  # no. of parameters
    xm = wave
    ym = ten_gaussian(xm, *popt)

    #Calculating weighted Chi**2 value
    chi2=np.nansum((spectrum-ym)**2/error**2)/(len(spectrum)-nparam)

    if plot==True:

        xm = wave
        #sigma = get_gaussian_ci(xm, gaussian, popt, pcov)

        plt.rcParams.update({'axes.titlesize': 'Large',
                 'axes.labelsize':'16.0',
                 'axes.linewidth':     '1.8' ,
                 'ytick.labelsize': '16.0',
                 'xtick.labelsize': '16.0',
                 'font.size': '10.0',
                 'legend.fontsize':'small'})
        
        fig = plt.figure(figsize = (15, 6))
        ax = fig.add_subplot()

        lim = 1.2*(spectrum-error).max()

        #Plotting final output
        #ax.step(wave, spectrum*1e-17, alpha=0.5, label='steps')
        ax.errorbar(wave, spectrum, error, c='k', label='Error bar')
        ax.plot(xm, ym, c='r', linestyle = '-', linewidth = 1, label='Model')  
        ax.scatter(wave, spectrum, c='b', alpha=0.8, s = 10, label='Data points')  

        #colors = plt.cm.viridis(np.linspace(0, 1, 10))
#
        #for i, (color, lw_shift) in enumerate(zip(colors, lw)):
        #    single_gaussian = gaussian(wave, popt[i + 4], popt[1] + lw_shift, popt[2], popt[3])
        #    ax.plot(wave, single_gaussian * 1e-17, color=color, linestyle='--', label=f'Gaussian {i + 1}')

        rest_positions1 = [4638.86, 4641.81, 4649.13, 4650.84,4661.63, 4676.23]  
        rest_positions2 = [4607.16, 4621.35, 4630.54, 4643.09, 4658.5]  

        # Assuming constant velocity shift
        line_positions1 = rest_positions1   #*np.array(np.ones(len(rest_positions1)))*(1+np.divide(50, 299792.458)) # velocity shift calculated using 5007 line
        line_positions2 = rest_positions2   #*np.array(np.ones(len(rest_positions2)))*(1+np.divide(50, 299792.458)) # velocity shift calculated using 5007 line

        line_labels1 = ['OII 4638.86', 'OII 4641.81', 'OII 4649.13', 'OII 4650.84', 'OII 4661.63', 'OII 4676.23']

        line_labels2 = ['NII4607.16', 'NII4621.35', 'NII4630.54', 'NII4643.09', '[Fe III] 4658.5']
    
        # Annotate OII lines
        for pos1, label1 in zip(line_positions1, line_labels1):

            ax.axvline(x=pos1, color='r', linestyle='--', alpha=0.6, linewidth=2.8)
            ax.annotate(label1, xy=(pos1, lim), xytext=(pos1, lim),
                        rotation=90, verticalalignment='bottom', horizontalalignment='center')

        # Annotate NII and Fe III lines
        for pos2, label2 in zip(line_positions2, line_labels2):
            
            ax.axvline(x=pos2, color='deepskyblue', linestyle='--', linewidth=1.0)
            ax.annotate(label2, xy=(pos2, lim), xytext=(pos2, lim),
                        rotation=90, verticalalignment='bottom', horizontalalignment='center')
                            
        ax.set_xlim(lwave1-dwave*2, lwave1+lw[9]+dwave*6)
        ax.set_ylim(np.min([popt[12], (spectrum-error).min()]), 1.5*np.max([popt[9], ym.max()]))
        #ax.set_ylim(-1, 20)

        ax.set_xlabel(r'Wavelength $(\AA)$')
        ax.set_ylabel(r'Intensity $(10^{-17} \times erg\, s^{-1}\, cm^{-2} \,\AA^{-1} arcsec^{-2})$', fontsize = 'large')

        ax.set_title(fr'NII and OII RL line fit')
        ax.legend()
        plt.tight_layout()

        #Save the figure
        #print(f'Plotting fig in /Volumes/amrita/linefitplots/'+plotout+'.png')
        
        plt.savefig(f"{plotdir}{plotout}.png", dpi =100)
        
        
    return popt, pcov 

In [12]:
'''
Function to fit 10 Gaussian to emission lines

Inputs: 
wave: wavelength 
spectrum: flux 
error: error 
lwave1: centroid wavlength of first line
lwave2: centroid wavlength of second line
lwave3: centroid wavlength of third line
dwave: difference in wavelength
plot: True/False to produce plots of linefits(boolean) 
plotout: plot name

Outputs:
Returns optimal parameters and covariance matrix on fitting the emission lines
'''

def fit_ten_gauss_quad(wave, spectrum, error, lwave1, dwave=6, plot=False, plotout='linefit'):
    
    lw = [0, 14.19, 23.38, 31.69, 34.65, 35.93, 41.97, 43.68, 51.01, 54.47, 69.07]

    #making a selection on wavelength range, and finite flux and error values
    sel1 = (wave > (lwave1+lw[0]) - dwave / 2) * (wave < (lwave1+lw[0])+ dwave / 2) * (np.isfinite(spectrum)) * (np.isfinite(error))  
    sel2 = (wave > (lwave1+lw[1]) - dwave / 2) * (wave < (lwave1+lw[1])+ dwave / 2) * (np.isfinite(spectrum)) * (np.isfinite(error))
    sel3 = (wave > (lwave1+lw[2]) - dwave / 2) * (wave < (lwave1+lw[2])+ dwave / 2) * (np.isfinite(spectrum)) * (np.isfinite(error))
    sel4 = (wave > (lwave1+lw[3]) - dwave / 2) * (wave < (lwave1+lw[3])+ dwave / 2) * (np.isfinite(spectrum)) * (np.isfinite(error))
    sel5 = (wave > (lwave1+lw[4]) - dwave / 2) * (wave < (lwave1+lw[4])+ dwave / 2) * (np.isfinite(spectrum)) * (np.isfinite(error))
    sel6 = (wave > (lwave1+lw[6]) - dwave / 2) * (wave < (lwave1+lw[6])+ dwave / 2) * (np.isfinite(spectrum)) * (np.isfinite(error))
    sel7 = (wave > (lwave1+lw[7]) - dwave / 2) * (wave < (lwave1+lw[7])+ dwave / 2) * (np.isfinite(spectrum)) * (np.isfinite(error))
    sel8 = (wave > (lwave1+lw[8]) - dwave / 2) * (wave < (lwave1+lw[8])+ dwave / 2) * (np.isfinite(spectrum)) * (np.isfinite(error))
    sel9 = (wave > (lwave1+lw[9]) - dwave / 2) * (wave < (lwave1+lw[9])+ dwave / 2) * (np.isfinite(spectrum)) * (np.isfinite(error))
    sel10 = (wave > (lwave1+lw[10]) - dwave / 2) * (wave < (lwave1+lw[10])+ dwave / 2) * (np.isfinite(spectrum)) * (np.isfinite(error))

    sel = (wave > lwave1 - dwave / 2) * (wave < lwave1 + dwave ) * (np.isfinite(spectrum)) * (np.isfinite(error))
    
    c0, c1, c2 = np.nanmin(spectrum[sel]), 0.0, 0.0

    bounds = ((0, lwave1 - 3, 0.5, -np.inf, 0, 0, 0, 0, 0, 0, 0, 0, 0, -np.inf, -np.inf, -np.inf), 
              (np.inf, lwave1+ 3, 0.75, np.inf, np.inf, np.inf, np.inf, np.inf, np.inf, np.inf, np.inf, np.inf, np.inf, np.inf, np.inf, np.inf))
    
    # Initial paramets for the Gaussian fitting
    p0 = (np.abs(np.nanmax(spectrum[sel1])- np.nanmin(spectrum[sel1])) * np.sqrt(2 * np.pi) * 0.7,  lwave1, 0.7, 30, 
          np.abs(np.nanmax(spectrum[sel2])- np.nanmin(spectrum[sel2])) * np.sqrt(2 * np.pi) * 0.7,  
          np.abs(np.nanmax(spectrum[sel3])- np.nanmin(spectrum[sel3])) * np.sqrt(2 * np.pi) * 0.7,  
          np.abs(np.nanmax(spectrum[sel4])- np.nanmin(spectrum[sel4])) * np.sqrt(2 * np.pi) * 0.7,  
          np.abs(np.nanmax(spectrum[sel5])- np.nanmin(spectrum[sel5])) * np.sqrt(2 * np.pi) * 0.7,  
          np.abs(np.nanmax(spectrum[sel6])- np.nanmin(spectrum[sel6])) * np.sqrt(2 * np.pi) * 0.7,  
          np.abs(np.nanmax(spectrum[sel7])- np.nanmin(spectrum[sel7])) * np.sqrt(2 * np.pi) * 0.7,
          np.abs(np.nanmax(spectrum[sel8])- np.nanmin(spectrum[sel8])) * np.sqrt(2 * np.pi) * 0.7,
          np.abs(np.nanmax(spectrum[sel9])- np.nanmin(spectrum[sel9])) * np.sqrt(2 * np.pi) * 0.7,
          np.abs(np.nanmax(spectrum[sel10])- np.nanmin(spectrum[sel10])) * np.sqrt(2 * np.pi) * 0.7,  c0, c1, c2)

    try:
        
        sel_goodpix = np.isfinite(spectrum)*np.isfinite(error)*np.isfinite(wave)
        
        # Calculating optimal parmeters and covariance matrix
        popt, pcov = curve_fit(ten_gaussian_quad, wave[sel_goodpix], spectrum[sel_goodpix], sigma=error[sel_goodpix], p0=p0, 
        absolute_sigma=True, bounds=bounds)
        
    except RuntimeError:
        popt = np.array([-99, lwave1, 0.7, -99, -99, -99, -99, -99, -99, -99, -99, -99, -99, np.nanmin(spectrum), -99, -999])
        pcov = np.zeros((10, 10))
        pcov[0, 0] = np.sum(error ** 2)

    nparam=len(popt)  # no. of parameters
    xm = wave
    ym = ten_gaussian_quad(xm, *popt)

    #Calculating weighted Chi**2 value
    chi2=np.nansum((spectrum-ym)**2/error**2)/(len(spectrum)-nparam)

    if plot==True:

        xm = wave
        #sigma = get_gaussian_ci(xm, gaussian, popt, pcov)

        plt.rcParams.update({'axes.titlesize': '24',
                 'axes.labelsize':'22.0',
                 'axes.linewidth':     '1.8' ,
                 'ytick.labelsize': '22.0',
                 'xtick.labelsize': '22.0',
                 'font.size': '20.0',
                 'legend.fontsize':'22.0'})
        
        fig = plt.figure(figsize = (15, 7.5))
        ax = fig.add_subplot()

        #lim = 12.5
        lim = 1*np.nanmax((spectrum-error))

        #Plotting final output
        ax.errorbar(wave, spectrum, error, c='k', label='Error bar')
        ax.plot(xm, ym, c='r', linestyle = '-', linewidth = 1, label='model')  
        ax.scatter(wave, spectrum, c='b', alpha=0.8, s = 10, label='data points')  


        rest_positions1 = [4638.86, 4641.81, 4649.13, 4650.84, 4661.63, 4676.23]  
        rest_positions2 = [4607.16, 4621.35, 4630.54, 4643.09, 4658.5]   

        # Assuming constant velocity shift
        line_positions1 = rest_positions1   #*np.array(np.ones(len(rest_positions1)))*(1+np.divide(50, 299792.458)) # velocity shift calculated using 5007 line
        line_positions2 = rest_positions2         
        
        line_labels1 = ['OII 4638.86\n', 'OII 4641.81\n', 'OII 4649.13\n', '\nOII 4650.84', '\nOII 4661.63', '\nOII 4676.23']

        line_labels2 = ['NII4607.16', 'NII4621.35', 'NII4630.54', '\nNII4643.09', '[Fe III] 4658.5']
    
        # Annotate OII lines
        for pos1, label1 in zip(line_positions1, line_labels1):

            ax.axvline(x=pos1, color='r', linestyle='--', alpha=0.6, linewidth=2.8)
            ax.annotate(label1, xy=(pos1, lim), xytext=(pos1, lim),
                        rotation=90, verticalalignment='bottom', horizontalalignment='center')

        # Annotate NII and Fe III lines
        for pos2, label2 in zip(line_positions2, line_labels2):
            
            ax.axvline(x=pos2, color='deepskyblue', linestyle='--', linewidth=1.8)
            ax.annotate(label2, xy=(pos2, lim), xytext=(pos2, lim),
                        rotation=90, verticalalignment='bottom', horizontalalignment='center') 
                            
        ax.set_xlim(lwave1-dwave*2, lwave1+lw[9]+dwave*6)
        ax.set_ylim(np.nanmin([popt[13], np.nanmin((spectrum-error))]), 1.21*np.nanmax([popt[10], np.nanmax(ym)]))
        #ax.set_ylim(-1, 20)

        #ax.text(4608, 1.1*popt[10], s='Integrated spectrum')
        ax.set_xlabel(r'Wavelength $(\AA)$')
        ax.set_ylabel(r'Intensity [$\times 10^{-17}$ erg s$^{-1}$ cm$^{-2}$ arcsec$^{-2}$]')

        #ax.set_title(fr'OII RL, [FeIII] line fit')
        ax.legend()
        plt.tight_layout()

        #Save the figure
        plt.savefig(f'{plotdir}{plotout}_quad_cont_test.png', dpi =100)

    #return  popt, pcov, mean_flux6, std_flux6
    return popt, pcov


In [13]:
plt.close('all')

In [14]:
def mklinemask(wave, line_mask_arr, left_mask_arr, right_mask_arr, line):
    
    # masking array of length same as wave with False value
    mask = np.repeat(False, len(wave))
    
    # Creating individual masks for line, and left and right continuum regions
    masking_line_range = (wave >= line_mask_arr[0]) & (wave <= line_mask_arr[1])
    masking_left_range = (wave >= left_mask_arr[0]) & (wave <= left_mask_arr[1])
    masking_right_range = (wave >= right_mask_arr[0]) & (wave <= right_mask_arr[1])

    # continuum mask by combining left and right ranges
    continuum_mask = masking_left_range | masking_right_range

    # Total mask
    total_mask = masking_line_range | continuum_mask

    return total_mask

In [15]:
plt.close('all')

In [16]:

##################################################### SFRAME combined FITTING OF LINES IN EACH FIBER SPECTRUM #####################################################

# Initializing linefit dictinory to store rest frame wavelengths, line range and the underlying left and right continua

line_fitting = {

            'HI3750': [3750.15, [3747.5, 3753.5], [3738, 3743], [3758, 3765]], # H_kappa           
            'HI3771': [3770.63, [3768.5, 3773.5], [3758, 3764], [3776, 3786]], # H_iota
            'HI3835': [3835.38, [3833, 3839], [3824, 3832], [3842, 3852]], # H_eta
            'HI4102': [4101.74, [4100, 4105], [4085, 4095], [4110, 4120]], # H_delta
            'HI8863': [8862.78, [8860,8865.5], [8852,8859], [8870,8880]], # P_11        
            'HI9015': [9014.91, [9011.5, 9019], [9005, 9010], [9022, 9032]], # P_10
            'HI9229': [9229.01, [9226.5, 9232.5], [9212, 9225], [9240, 9250]], # P_9
            'Ha6563': [6562.8, [6560, 6566], [6535, 6545], [6590, 6605]],
            'Hb4861' : [4861.32, [4858, 4866], [4820, 4850], [4870, 4900]],
            'Hgm4340': [4340.463, [4338, 4344], [4310, 4335], [4345, 4355]],
            #'NII_OII' : [4607.16, [4606.4, 4678], [4600, 4605], [4680, 4694]],
            #'CII4267':[4267.09, [4265, 4270], [4240, 4260], [4275, 4285]],
            ###'[SII]4069': [4068.6, [4067, 4080], [4055, 4065],[4083, 4098]],  
            '[SII]6717': [6716.44, [6714, 6720], [6680, 6700], [6735, 6750]],
            '[SII]6731': [6730.82, [6728, 6733], [6684, 6702], [6735, 6750]],
            #'[NII]5755': [5754.64, [5752, 5758], [5720, 5745], [5760, 5775]],
            #'[NII]6548': [6548.04, [6546, 6551], [6525, 6540], [6590, 6605]],  
            #'[NII]6584': [6583.46, [6581, 6586], [6535, 6545], [6590, 6605]],
            #'[OIII]4363': [4363.2,[4359, 4367], [4346, 4355],  [4370, 4385]],
            '[OIII]4959': [4958.91,[4955, 4962], [4830, 4850],  [5020, 5035]],
            '[OIII]5007': [5006.84,[5004, 5010], [4830, 4850],  [5020, 5035]],
            #'[SIII]6312': [6312.1, [6310, 6315], [6260, 6290], [6320, 6330]],
            #'[SIII]9069': [9068.6, [9066, 9072], [9025, 9035], [9080, 9100]],
            #'[SIII]9531': [9530.6, [9528, 9535], [9480, 9510], [9555, 9580]],  
            #'[OII]3726': [3726.03, [3723, 3732], [3715, 3720], [3740, 3760]],
            #'[OII]7320': [7319.99, [7317, 7324], [7300, 7315], [7345, 7360]],
            #'[OII]7330': [7330, [7327, 7335], [7300, 7315], [7345, 7360]],
            #'[ClIII]5517':[5517.71, [5515, 5520], [5490, 5510], [5522, 5535]],
            #'[ClIII]5537':[5537.88, [5536, 5541], [5520, 5535], [5546, 5570]],

            
            
              }

linename = []
lines = []
line_mask_arr = []
left_mask_arr = []
right_mask_arr = []

#Iterating over the above linefitdict keys and values to append values to above lists
for key, values in zip(line_fitting.keys(), line_fitting.values()):

    linename.append(key)
    lines.append(values[0])
    line_mask_arr.append(values[1]) 
    left_mask_arr.append(values[2]) 
    right_mask_arr.append(values[3]) 

# Dictionary to store data for each line in each fiber from each tile
linefitdict = {'RA': [], 'Dec': [], 'tileid': [], 'fiberid':[], }  

# Initialize the dictionary to store fitting results
num = np.linspace(1, 10, 10, dtype = int)

for i, j in enumerate(linename):

    if j == '[OII]3726'  or j  == '[SII]4069' or j == '[OIII]4363':
       linefitdict[f'{j}_flux0'] = []
       linefitdict[f'{j}_flux0_err'] = []
       linefitdict[f'{j}_wave0'] = []
       linefitdict[f'{j}_wave0_err'] = []
       linefitdict[f'{j}_sigma0'] = []
       linefitdict[f'{j}_flux1'] = []
       linefitdict[f'{j}_flux1_err'] = []
       linefitdict[f'{j}_wave1'] = []
       linefitdict[f'{j}_wave1_err'] = []
       linefitdict[f'{j}_sigma1'] = []
       linefitdict[f'{j}_cont0'] = []
       linefitdict[f'{j}_cont0_err'] = []
       linefitdict[f'{j}_cont1'] = []
       linefitdict[f'{j}_cont1_err'] = []
       linefitdict[f'{j}_cont2'] = []
       linefitdict[f'{j}_cont2_err'] = []


    #elif  j == '[OIII]4363':                               # quad continuum+vel shift of [OIII]4363 fix  wrt 4360 [feIII] line
    #    linefitdict[f'{j}_flux0'] = []
    #    linefitdict[f'{j}_flux0_err'] = []
    #    linefitdict[f'{j}_sigma0'] = []
    #    linefitdict[f'{j}_flux1'] = []
    #    linefitdict[f'{j}_flux1_err'] = []
    #    linefitdict[f'{j}_cont0'] = []
    #    linefitdict[f'{j}_cont0_err'] = []
    #    linefitdict[f'{j}_cont1'] = []
    #    linefitdict[f'{j}_cont1_err'] = []
    #    linefitdict[f'{j}_cont2'] = []
    #    linefitdict[f'{j}_cont2_err'] = []


    elif j  == 'NII_OII':

        for k in num:  # Assuming up to 10 components 
            linefitdict[f'{j}_flux{k-1}'] = []
            linefitdict[f'{j}_flux{k-1}_err'] = []
        linefitdict[f'{j}_wave0'] = []
        linefitdict[f'{j}_wave0_err'] = []
        linefitdict[f'{j}_sigma0'] = []
        linefitdict[f'{j}_cont0'] = []
        linefitdict[f'{j}_cont0_err'] = []
        linefitdict[f'{j}_cont1'] = []
        linefitdict[f'{j}_cont1_err'] = []
        linefitdict[f'{j}_cont2'] = []
        linefitdict[f'{j}_cont2_err'] = []

    elif j == '[NII]5755'  or j  == 'CII4267' or j=='[SIII]6312' or j=='[OII]7320' or j=='[OII]7330':
        linefitdict[f'{j}_flux0'] = []
        linefitdict[f'{j}_flux0_err'] = []
        linefitdict[f'{j}_wave0'] = []
        linefitdict[f'{j}_wave0_err'] = []
        linefitdict[f'{j}_sigma0'] = []
        linefitdict[f'{j}_cont0'] = []
        linefitdict[f'{j}_cont0_err'] = []
        linefitdict[f'{j}_cont1'] = []
        linefitdict[f'{j}_cont1_err'] = []
        linefitdict[f'{j}_cont2'] = []
        linefitdict[f'{j}_cont2_err'] = []
    
    else:
        linefitdict[f'{j}_flux0'] = []
        linefitdict[f'{j}_flux0_err'] = []
        linefitdict[f'{j}_wave0'] = []
        linefitdict[f'{j}_wave0_err'] = []
        linefitdict[f'{j}_sigma0'] = []
        linefitdict[f'{j}_cont'] = []
        linefitdict[f'{j}_cont_err'] = []


plot = False

plotdir = '/Users/amritasingh/amrita/LVM/LVM_M17_output/linefits/'

# reading velocity of 5007 line, to use it constrain 4363 velocity shift

#hdu = fits.open(outfilename)
#data = hdu[1].data
#wave_5007 = data['[OIII]5007_wave0']
#sigma_5007 = data['[OIII]5007_sigma0']
#vel_shift_5007 = (wave_5007 - 5006.8)/5006.8 * 299792.458
#vel_shift_5007 = vel_shift_5007+60  # km/s

# Reading fits file

rss = '/Users/amritasingh/amrita/LVM/LVM_M17_output/M17_rss.fits'
with fits.open(rss) as hdu:

    data = hdu[1].data

    print(data.shape)

    wave = data['wave']
    #flux = data['flux_early']* 1e17
    final_flux = data['flux']* 1e17
    error = data['error']* 1e17
    #sym_err = 1e-19*1e17
    #error = np.sqrt(error**2 + sym_err**2)

    fiberid = data['fiberid']
    tileid = data['tileid']
    RA = data['ra']
    Dec = data['dec']


    # sel = (tileid == 3) & (fiberid==733) # Selecting tileid 3 for fitting ########################
    # final_flux = final_flux[sel]
    # error = error[sel]
    # RA = RA[sel]
    # Dec = Dec[sel]
    # fiberid = fiberid[sel]
    # tileid = tileid[sel]


    # Reading flux and error in each fiber
    for i, j in enumerate(fiberid):

        #spaxel_flux = flux[i]
        spaxel_final_flux = final_flux[i]
        spaxel_error = error[i]
        RA_fib = RA[i]
        Dec_fib = Dec[i]
        fibid_fib = fiberid[i]
        tileid_fib = tileid[i]
        spaxel_wave = wave[i]

        #print(f"Processing fiber {j} in tile {tileid_fib}")

        #print(f"fiberid {fibid_fib}, tileid {tileid_fib},flux in each spaxel: {np.nanmean(spaxel_flux)}, err in each spaxel: {np.nanmean(spaxel_error)}")

        # Store the fiber number and tile id for each iteration
        linefitdict['RA'].append(RA_fib)
        linefitdict['Dec'].append(Dec_fib)
        linefitdict['fiberid'].append(fibid_fib)
        linefitdict['tileid'].append(tileid_fib)
        #linefitdict['flux_early'].append(spaxel_flux * 1e-17)
        #linefitdict['flux'].append(spaxel_final_flux * 1e-17)
        #linefitdict['error'].append(spaxel_error* 1e-17)
        #linefitdict['wave'].append(spaxel_wave)


        # Iterating over each line and masks to select emission line ranges with their underlying left and right continuum masks
        for lineid, line_data in line_fitting.items():
            
            # Extracting data for line fitting
            line, line_mask, left_mask, right_mask = line_data

            
            if lineid == '[OII]3726':

                total_mask = mklinemask(spaxel_wave, line_mask, left_mask, right_mask, line)
                #Fit 2 gaussians to [OII] doublets 3726 and 3729 \lambda (3726 is written as line+2.8)

                popt, pcov = fit_double_gauss_quad(spaxel_wave[total_mask], spaxel_final_flux[total_mask], spaxel_error[total_mask], line, line+2.8, plot = plot, plotout=f'individual_fib_{lineid}_fiber{j}_tile{tileid_fib}')

                # Store data for 1st component
                linefitdict[lineid+'_flux0'].append(popt[0]*1e-17)                 
                linefitdict[lineid+'_flux0_err'].append(np.sqrt(pcov[0,0])*1e-17)
                linefitdict[lineid+'_wave0'].append(popt[1])
                linefitdict[lineid+'_wave0_err'].append(np.sqrt(pcov[1,1]))
                linefitdict[lineid+'_sigma0'].append(popt[2])
                # Store data for 2nd component
                linefitdict[lineid+'_flux1'].append(popt[3]*1e-17)
                linefitdict[lineid+'_flux1_err'].append(np.sqrt(pcov[3,3])*1e-17)
                linefitdict[lineid+'_wave1'].append(popt[4])
                linefitdict[lineid+'_wave1_err'].append(np.sqrt(pcov[4,4]))
                # Store continuum data
                linefitdict[lineid+'_cont0'].append(popt[5]*1e-17)
                linefitdict[lineid+'_cont0_err'].append(np.sqrt(pcov[5,5])*1e-17)
                linefitdict[lineid+'_cont1'].append(popt[6]*1e-17)
                linefitdict[lineid+'_cont1_err'].append(np.sqrt(pcov[6,6])*1e-17)
                linefitdict[lineid+'_cont2'].append(popt[7]*1e-17)
                linefitdict[lineid+'_cont2_err'].append(np.sqrt(pcov[7,7])*1e-17)


            elif lineid == '[OIII]4363':

                total_mask = mklinemask(spaxel_wave, line_mask, left_mask, right_mask, line)

                #Fit 2 gaussians to [FeII]4360 and [OIII]4363 \lambda lines (4360 \lambda is written as line-3.2)
                popt, pcov  = fit_double_gauss_quad(spaxel_wave[total_mask], spaxel_final_flux[total_mask], spaxel_error[total_mask], line-3.2, line, plot = True, plotout=f'individual_fib_{lineid}_{j}')

                # Store data for 1st component
                linefitdict[lineid+'_flux0'].append(popt[0]*1e-17)
                linefitdict[lineid+'_flux0_err'].append(np.sqrt(pcov[0,0])*1e-17)
                linefitdict[lineid+'_wave0'].append(popt[1])
                linefitdict[lineid+'_wave0_err'].append(np.sqrt(pcov[1,1]))
                linefitdict[lineid+'_sigma0'].append(popt[2])

                # Store data for 2nd component
                linefitdict[lineid+'_flux1'].append(popt[3]*1e-17)
                linefitdict[lineid+'_flux1_err'].append(np.sqrt(pcov[3,3])*1e-17)
                linefitdict[lineid+'_wave1'].append(popt[4])
                linefitdict[lineid+'_wave1_err'].append(np.sqrt(pcov[4,4]))
                #cont
                linefitdict[lineid+'_cont0'].append(popt[5]*1e-17)
                linefitdict[lineid+'_cont0_err'].append(np.sqrt(pcov[5,5])*1e-17)
                linefitdict[lineid+'_cont1'].append(popt[6]*1e-17)
                linefitdict[lineid+'_cont1_err'].append(np.sqrt(pcov[6,6])*1e-17)
                linefitdict[lineid+'_cont2'].append(popt[7]*1e-17)
                linefitdict[lineid+'_cont2_err'].append(np.sqrt(pcov[7,7])*1e-17)


            elif lineid == '[SII]4069':
                
                total_mask = mklinemask(spaxel_wave, line_mask, left_mask, right_mask, line)

                #Fit 3 Gaussians to 4069, 4073 and 4076 \lambda lines, the rest frame lambda of other two lines are defined as line+c (c is a const)
                popt, pcov = fit_double_gauss_quad(spaxel_wave[total_mask], spaxel_final_flux[total_mask], spaxel_error[total_mask], line, line+5, line+7.5, plot = plot, plotout=f'individual_fib_{lineid}_{j}')

                # Store data for 1st component
                linefitdict[lineid+'_flux0'].append(popt[0]*1e-17)
                linefitdict[lineid+'_flux0_err'].append(np.sqrt(pcov[0,0])*1e-17)
                linefitdict[lineid+'_wave0'].append(popt[1])
                linefitdict[lineid+'_wave0_err'].append(np.sqrt(pcov[1,1]))
                linefitdict[lineid+'_sigma0'].append(popt[2])

                # Store data for 2nd component
                linefitdict[lineid+'_flux1'].append(popt[3]*1e-17)
                linefitdict[lineid+'_flux1_err'].append(np.sqrt(pcov[3,3])*1e-17)
                linefitdict[lineid+'_wave1'].append(popt[4])
                linefitdict[lineid+'_wave1_err'].append(np.sqrt(pcov[4,4]))
                #cont
                linefitdict[lineid+'_cont0'].append(popt[5]*1e-17)
                linefitdict[lineid+'_cont0_err'].append(np.sqrt(pcov[5,5])*1e-17)
                linefitdict[lineid+'_cont1'].append(popt[6]*1e-17)
                linefitdict[lineid+'_cont1_err'].append(np.sqrt(pcov[6,6])*1e-17)
                linefitdict[lineid+'_cont2'].append(popt[7]*1e-17)
                linefitdict[lineid+'_cont2_err'].append(np.sqrt(pcov[7,7])*1e-17)

            
            elif lineid == 'NII_OII':

                total_mask = mklinemask(spaxel_wave, line_mask, left_mask, right_mask, line)

                #print(f"fiber no. {fibid_fib}, tile id: {tileid_fib}")

                popt, pcov = fit_ten_gauss_quad(spaxel_wave[total_mask], spaxel_final_flux[total_mask], spaxel_error[total_mask], line, plot = plot, plotout=f'individual_fib_{lineid}_fiber{j}_tile{tileid_fib}')

                for i in range(16):

                    if popt[i]<0:
                        popt[i] = np.nan
       
                #print(f"fiber no. {fibid_fib}, tile id: {tileid_fib}, popt shape: {popt.shape}, popt 10: {popt[10]}")
                
                # Store data for 1st component
                linefitdict[lineid+'_flux0'].append(popt[0]*1e-17)
                linefitdict[lineid+'_flux0_err'].append(np.sqrt(pcov[0,0])*1e-17)
                linefitdict[lineid+'_wave0'].append(popt[1])
                linefitdict[lineid+'_wave0_err'].append(np.sqrt(pcov[1,1]))
                linefitdict[lineid+'_sigma0'].append(popt[2])
                # Store data for 2nd component
                linefitdict[lineid+'_flux1'].append(popt[4]*1e-17)
                linefitdict[lineid+'_flux1_err'].append(np.sqrt(pcov[4, 4])*1e-17)
                # Store data for 3rd component
                linefitdict[lineid+'_flux2'].append(popt[5]*1e-17)
                linefitdict[lineid+'_flux2_err'].append(np.sqrt(pcov[5, 5])*1e-17)
                # Store data for 4th component
                linefitdict[lineid+'_flux3'].append(popt[6]*1e-17)
                linefitdict[lineid+'_flux3_err'].append(np.sqrt(pcov[6, 6])*1e-17)
                # Store data for 5th component
                linefitdict[lineid+'_flux4'].append(popt[7]*1e-17)
                linefitdict[lineid+'_flux4_err'].append(np.sqrt(pcov[7, 7])*1e-17)
                # Store data for 6th component
                linefitdict[lineid+'_flux5'].append(popt[8]*1e-17)
                linefitdict[lineid+'_flux5_err'].append(np.sqrt(pcov[8, 8])*1e-17)
                # Store data for 7th component
                linefitdict[lineid+'_flux6'].append(popt[9]*1e-17)
                linefitdict[lineid+'_flux6_err'].append(np.sqrt(pcov[9, 9])*1e-17)
                # Store data for 8th component
                linefitdict[lineid+'_flux7'].append(popt[10]*1e-17)
                linefitdict[lineid+'_flux7_err'].append(np.sqrt(pcov[10, 10])*1e-17)
                # Store data for 9th component
                linefitdict[lineid+'_flux8'].append(popt[11]*1e-17)
                linefitdict[lineid+'_flux8_err'].append(np.sqrt(pcov[11, 11])*1e-17)
                # Store data for 10th component
                linefitdict[lineid+'_flux9'].append(popt[12]*1e-17)
                linefitdict[lineid+'_flux9_err'].append(np.sqrt(pcov[12, 12])*1e-17)
                # Store continuum data
                linefitdict[lineid+'_cont0'].append(popt[13]*1e-17)
                linefitdict[lineid+'_cont0_err'].append(np.sqrt(pcov[13, 13])*1e-17)
                linefitdict[lineid+'_cont1'].append(popt[14]*1e-17)
                linefitdict[lineid+'_cont1_err'].append(np.sqrt(pcov[14,14])*1e-17)
                linefitdict[lineid+'_cont2'].append(popt[15]*1e-17)
                linefitdict[lineid+'_cont2_err'].append(np.sqrt(pcov[15, 15])*1e-17)

            elif lineid == '[NII]5755'  or lineid  == 'CII4267' or lineid =='[SIII]6312' or lineid =='[OII]7320' or lineid =='[OII]7330':

                total_mask = mklinemask(spaxel_wave, line_mask, left_mask, right_mask, line)

                # Single Gaussian fitting
                popt, pcov = fit_gauss_quad(spaxel_wave[total_mask], spaxel_final_flux[total_mask], spaxel_error[total_mask], line, plot = plot, plotout=f'individual_fib_{lineid}_{j}_retrieve_central_spaxels')

                
                linefitdict[lineid+'_flux0'].append(popt[0]*1e-17)
                linefitdict[lineid+'_flux0_err'].append(np.sqrt(pcov[0, 0])*1e-17)
                linefitdict[lineid+'_wave0'].append(popt[1])
                linefitdict[lineid+'_wave0_err'].append(np.sqrt(pcov[1, 1]))
                linefitdict[lineid+'_sigma0'].append(popt[2])
                linefitdict[lineid+'_cont0'].append(popt[3]*1e-17)
                linefitdict[lineid+'_cont0_err'].append(np.sqrt(pcov[3, 3])*1e-17)
                linefitdict[lineid+'_cont1'].append(popt[4]*1e-17)
                linefitdict[lineid+'_cont1_err'].append(np.sqrt(pcov[4,4])*1e-17)
                linefitdict[lineid+'_cont2'].append(popt[5]*1e-17)
                linefitdict[lineid+'_cont2_err'].append(np.sqrt(pcov[5, 5])*1e-17)


            else:

                total_mask = mklinemask(spaxel_wave, line_mask, left_mask, right_mask, line)

                # Single Gaussian fitting
                popt, pcov = fit_gauss(spaxel_wave[total_mask], spaxel_final_flux[total_mask], spaxel_error[total_mask], line, plot = plot, plotout=f'individual_fib_{lineid}_{j}_retrieve_central_spaxels')
                
                linefitdict[lineid+'_flux0'].append(popt[0]*1e-17)
                linefitdict[lineid+'_flux0_err'].append(np.sqrt(pcov[0, 0])*1e-17)
                linefitdict[lineid+'_wave0'].append(popt[1])
                linefitdict[lineid+'_wave0_err'].append(np.sqrt(pcov[1, 1]))
                linefitdict[lineid+'_sigma0'].append(popt[2])
                linefitdict[lineid+'_cont'].append(popt[3]*1e-17)
                linefitdict[lineid+'_cont_err'].append(np.sqrt(pcov[3, 3])*1e-17)
                # linefitdict[lineid+'_cont1'].append(popt[4]*1e-17)
                # linefitdict[lineid+'_cont1_err'].append(np.sqrt(pcov[4,4])*1e-17)
                # linefitdict[lineid+'_cont2'].append(popt[5]*1e-17)
                # linefitdict[lineid+'_cont2_err'].append(np.sqrt(pcov[5, 5])*1e-17)


# Save the table to a FITS file
max_length = max(len(lst) for lst in linefitdict.values())

for key in linefitdict:
    if len(linefitdict[key]) < max_length:
        linefitdict[key].extend([0] * (max_length - len(linefitdict[key])))

linefitdict = {key: np.array(value) for key, value in linefitdict.items()}
table = Table(linefitdict)

outfilename = '/Users/amritasingh/amrita/LVM/LVM_M17_output/M17_obs_flux.fits'
table.write(outfilename, overwrite=True)



(3508,)


In [ ]:
# plot the line intensities' map

In [33]:
#def plotmap(c, RA_list, Dec_list, data, pick_fib, vmin=None, vmax=None, showexpnum=True, title='M8 Haplha cont Map', ctitle = r'Flux (erg $s^{-1} cm^{-2} \AA^{-1}$)'):
def plotmap(c, RA_list, Dec_list, vmin=None, vmax=None, title='M8 Haplha cont Map', \
            ctitle = r'log Flux (erg $s^{-1} cm^{-2}$)', figname='Electron temperature map of [OIII]', folder='ne'):

    fig, ax = plt.subplots(figsize=(8,8))

    plt.rcParams.update({'axes.titlesize': 'Large',
                 'axes.labelsize':'14.0',
                 'axes.linewidth':     '1.8' ,
                 'ytick.labelsize': '12.0',
                 'xtick.labelsize': '12.0',
                 'font.size': '12.0',
                 'legend.fontsize':'small'})


    if vmin is None:
        vmin=np.min(c)
    if vmax is None:
        vmax=np.max(c)

    
    ra_hms = '18 03 40.3201232304'
    dec_dms = '-24 22 42.857540472'
    coord = SkyCoord(ra=ra_hms, dec=dec_dms, unit=(u.hourangle, u.deg), frame='icrs')
    
    ra_center_deg = coord.ra.deg
    dec_center_deg = coord.dec.deg

    # central spaxel
    ra, dec = 270.91779, -24.37442 #
    ra1, dec1 = 270.92372608, -24.37629402 # bright spaxel

    
    sc=ax.scatter(RA_list, Dec_list, c=c ,s=8, cmap ='plasma', vmin = vmin, vmax = vmax)
    #ax.scatter(ra1, dec1, s=8, edgecolor='magenta', marker='o', facecolor='None', label='Brightest spaxel')
    #ax.scatter(ra, dec, s=8, edgecolor='cyan', marker='o', facecolor='None', label='Central spaxel')
    ax.scatter(ra_center_deg, dec_center_deg, s=8, edgecolor='w', marker='*', facecolor='None', label='Her 36')


    # ---- Draw circular spaxel outlines ----
    spaxel_diam_arcsec = 35.3
    spaxel_radius_deg = (spaxel_diam_arcsec / 2) / 3600.0  # convert arcsec → degrees

    circ_central = Circle((ra, dec), spaxel_radius_deg, edgecolor='cyan', facecolor='none', lw=1.5, label='Central spaxel')
    circ_bright  = Circle((ra1, dec1), spaxel_radius_deg, edgecolor='magenta', facecolor='none', lw=1.5, linestyle='--', label='Bright spaxel')

    ax.add_patch(circ_central)
    ax.add_patch(circ_bright)
    
    ax.set_aspect('equal', adjustable='box')
    
    ax.set_xlabel("RA [deg]")
    ax.set_ylabel("DEC [deg]")
    ax.set_title(title)
    ax.tick_params(axis='x')
    ax.tick_params(axis='y')
    ax.set_xlim(270.4, 271.8)
    ax.set_ylim(-25, -23.6)

    ax.invert_xaxis()

    # Create a divider for existing axes instance
    divider = make_axes_locatable(ax)

    # Append axes to the right of ax, with width equal to 5% of ax, and padding equal to 0.05 inch
    cax = divider.append_axes("right", size="5%", pad=1e-4)
    cbar = fig.colorbar(sc, cax=cax)
    cbar.set_label(ctitle)
    
    ax.set_facecolor('black')
    plt.tight_layout()
    plt.savefig(f'/Users/amritasingh/amrita/LVM/dec16_25_1.2.dev0/{title}.png')
    plt.close()


with fits.open('/Users/amritasingh/amrita/LVM/dec16_25_1.2.dev0/lagoon_combined_multi_normalized_sframe_rss_hb_flux_1.1.2dev.fits') as hdu:

    data = hdu[1].data

c_km = 299792.458

lambda_fit = data['HI9229_wave0']#data['[OIII]4363_wave0'] #data['[OIII]5007_wave0']
lambda_rest = 9229.01 #4340.463#4363.2 #5006.8
vp9 = c_km * (lambda_fit - lambda_rest) / lambda_rest


plotmap(vp9, data['RA'], data['Dec'],  vmin=-10, vmax=15,\
                title=fr'P9 velocity', ctitle = r'km/s')

#========================================================================================================================

#with fits.open(outfilename) as hdu:
#
#    data = hdu[1].data
#
#sn = data['[OIII]4363_flux1']/data['[OIII]4363_flux1_err']  
#print(np.nanmax(sn))
#
#sel = sn>3
#plotmap(np.log10(data['[OIII]4363_flux1'])[sel], data['RA'][sel], data['Dec'][sel],  vmin=-19, vmax=-15.6,\
#                title=fr'[OIII] 4363 flux', ctitle = r'log [OIII](erg $s^{-1} cm^{-2} \AA^{-1})$')
#    
#sn = (data['NII_OII_flux5']+data['NII_OII_flux6'])/(data['NII_OII_flux5_err']+data['NII_OII_flux6_err'])
#
#oiisum = data['NII_OII_flux3']+data['NII_OII_flux4']+data['NII_OII_flux5']+data['NII_OII_flux6']+data['NII_OII_flux8']
#sel = sn>3
#plotmap((oiisum)[sel], data['RA'][sel], data['Dec'][sel],  vmin=1e-20, vmax=1e-16,\
#                title=fr'OII sum flux', ctitle = r'log CII (erg $s^{-1} cm^{-2} \AA^{-1})$')
    

In [15]:
plt.close()

In [9]:
# Date : Dec 9th, 2025, Tuesday; I'm going to correct the velocity of the RSS based on [OIII]5007 line
# I see tile to tile variation in velocity map of 5007 line, so I'll correct for it
with fits.open('/Users/amritasingh/amrita/LVM/LVM_lagoon_outputs/1.1.2dev0_outputs/resolved/july8_25/rss_obs_flux_table_1.1.2dev0_jul8.fits') as hdu:

    table = hdu[1].data
#=======================================================================================#

# read tileids
tileid = table['tileid']             
tiles = np.unique(tileid)

# creating masking for each tile
tile_masks = {t: (tileid == t) for t in tiles}
print(tile_masks.keys())

#=======================================================================================#
# Calculating v_5007 for each spaxel
c_km = 299792.458

lambda_fit = table['[OIII]5007_wave0']
lambda_rest = 5006.8
v5007 = c_km * (lambda_fit - lambda_rest) / lambda_rest

# Plotted it, saw jump between different tiles
#=======================================================================================#

#compute tile medians
tile_v = {}
for t in tiles:
    mask = tile_masks[t]
    tile_v[t] = np.nanmedian(v5007[mask])

print("Tile medians:", tile_v)

#=======================================================================================#
# assuming a global reference, 5007 is one of the brightest, let's take its median as global

v_ref = np.nanmedian(v5007)
print("Global reference v_ref =", v_ref)

#=======================================================================================#
# calculate tile to yile offset

tile_offset = {t: tile_v[t] - v_ref for t in tiles}

print("Tile offsets (km/s) relative to ref:")
for t in tiles:
    print(f"Tile {t}: {tile_offset[t]:.3f} km/s")

#=======================================================================================#
with fits.open(rss) as hdu:

    data = hdu[1].data
    #print(repr(hdu[1].header))

    wave = data['wave']
    flux_cube = data['flux']
    error_cube = data['error']

#correct per-tile Wavelength Zeropoint in the Spectra

flux_corrected = np.full_like(flux_cube, np.nan)
error_corrected = np.full_like(error_cube, np.nan)

for t in tiles:
    mask = tile_masks[t]
    dv = tile_offset[t]          # km/s
    factor = 1/ (1 + (dv / c_km))

    for i in np.where(mask)[0]:
        wave_shifted_i = wave[i] * factor   # per-spaxel wave grid

        f_interp = interp1d(wave_shifted_i, flux_cube[i],
                            bounds_error=False, fill_value=np.nan)
        e_interp = interp1d(wave_shifted_i, error_cube[i],
                            bounds_error=False, fill_value=np.nan)

        flux_corrected[i] = f_interp(wave[i])      # interpolate onto original wave grid
        error_corrected[i] = e_interp(wave[i])


output_file = f"{data_dir}july8_25/rss_spax_weighted_velocity_corrected.fits"

with fits.open(rss, mode="readonly") as hdul:

    hdr0 = hdul[0].header.copy()
    data_old = hdul[1].data

    # build new columns list
    new_cols = []

    for name in data_old.columns.names:
        if name.lower() == "flux":
            new_cols.append(fits.Column(name="flux",   format=data_old.columns[name].format,
                                        array=flux_corrected))
        elif name.lower() == "error":
            new_cols.append(fits.Column(name="error",  format=data_old.columns[name].format,
                                        array=error_corrected))
            
        else:
            # keep original column unchanged
            new_cols.append(fits.Column(name=name,
                                        format=data_old.columns[name].format,
                                        array=data_old[name]))

    # Make new HDU
    hdu1 = fits.BinTableHDU.from_columns(new_cols, header=hdul[1].header)

    hdul_new = fits.HDUList([fits.PrimaryHDU(header=hdr0), hdu1])
    #hdul_new.writeto(output_file, overwrite=True)

print("Saved:", output_file)


plotmap(tile_offset, table['RA'], table['Dec'],  vmin=-50, vmax=50,
               title=fr'velocity', ctitle = r'km/s')
    

dict_keys([1, 2, 3, 4, 5, 6, 7, 8, 9, 10])
Tile medians: {1: 8.738161185141307, 2: 1.2349143916693417, 3: 17.0308845091201, 4: 15.729414619308436, 5: 2.0925343455481453, 6: 14.638558440968218, 7: 18.00398467728001, 8: 15.213669675929234, 9: 12.968758821306174, 10: -8.195958784079128}
Global reference v_ref = 12.284896971320642
Tile offsets (km/s) relative to ref:
Tile 1: -3.547 km/s
Tile 2: -11.050 km/s
Tile 3: 4.746 km/s
Tile 4: 3.445 km/s
Tile 5: -10.192 km/s
Tile 6: 2.354 km/s
Tile 7: 5.719 km/s
Tile 8: 2.929 km/s
Tile 9: 0.684 km/s
Tile 10: -20.481 km/s
Saved: /Users/amritasingh/amrita/LVM/LVM_lagoon_outputs/1.1.2dev0_outputs/resolved/july8_25/rss_spax_weighted_velocity_corrected.fits


TypeError: float() argument must be a string or a real number, not 'dict'

In [9]:
plt.close()

In [6]:

# Date : Dec 10th, 2025, Wenesday; I'm shifting all flux and error in the rss file to reference zero wrt vel5007 shift
with fits.open('/Users/amritasingh/amrita/LVM/LVM_lagoon_outputs/1.1.2dev0_outputs/resolved/july8_25/rss_obs_flux_table_1.1.2dev0_jul8.fits') as hdu:

    table = hdu[1].data

# Calculating v_5007 for each spaxel
c_km = 299792.458

lambda_fit = table['[OIII]5007_wave0']
lambda_rest = 5006.8
v5007 = c_km * (lambda_fit - lambda_rest) / lambda_rest

v_ref =np.nanmedian(v5007)

# opening the rss file: contains weights, flux, error, wave, ra, dec, fiberid, tileid for each spaxel
with fits.open(rss) as hdu:

    data = hdu[1].data

    wave = data['wave'][0]
    flux_arr = data['flux']
    error_arr = data['error']

flux_shifted = np.full_like(flux_arr, np.nan)
error_shifted = np.full_like(error_arr, np.nan)


# correcting vel shift for each spaxel wrt v5007 
for i in range(len(v5007)):
    dv = v5007[i] - v_ref
    factor = 1/ (1 + dv / c_km)
    wave_shifted = wave * factor

    flux_shifted[i]  = np.interp(wave, wave_shifted, flux_arr[i],  left=np.nan, right=np.nan)
    error_shifted[i] = np.interp(wave, wave_shifted, error_arr[i], left=np.nan, right=np.nan)

output_file = f"{data_dir}july8_25/rss_spax_weighted_velocity_corrected.fits"

with fits.open(rss, mode="readonly") as hdul:

    hdr0 = hdul[0].header.copy()
    data_old = hdul[1].data

    # build new columns list
    new_cols = []

    for name in data_old.columns.names:
        if name.lower() == "flux":
            new_cols.append(fits.Column(name="flux",   format=data_old.columns[name].format,
                                        array=flux_shifted))
        elif name.lower() == "error":
            new_cols.append(fits.Column(name="error",  format=data_old.columns[name].format,
                                        array=error_shifted))
            
        else:
            # keep original column unchanged
            new_cols.append(fits.Column(name=name,
                                        format=data_old.columns[name].format,
                                        array=data_old[name]))

    # Make new HDU
    hdu1 = fits.BinTableHDU.from_columns(new_cols, header=hdul[1].header)

    hdul_new = fits.HDUList([fits.PrimaryHDU(header=hdr0), hdu1])
    hdul_new.writeto(output_file, overwrite=True)

print("Saved:", output_file)



Saved: /Users/amritasingh/amrita/LVM/LVM_lagoon_outputs/1.1.2dev0_outputs/resolved/july8_25/rss_spax_weighted_velocity_corrected.fits


In [21]:
plt.close()

In [ ]:
if:

In [ ]:
def dust_corr_all_ratios(filename, lines_to_correct):
    """
    Correct fluxes using multiple Balmer and Paschen line ratios, saving separate files for each correction.
    """
    with fits.open(filename) as hdul:
        table = hdul[1].data

    extinction_model = F99(Rv=3.1)
    
    # Prepare new columns
    new_columns = []
    
    for key in table.columns.names:
        if "flux" in key or "RA" in key or "Dec" in key:
            obs_col_name = f"obs_{key}"
            new_columns.append(fits.Column(name=obs_col_name, format="D", array=table[key]))

    for line_label in lines_to_correct:

        for i in range(10):  # Correcting up to 10 components

            flux_column = f"{line_label}_flux{i}"
            flux_err_column = f"{line_label}_flux{i}_err"
            wave_column = f"{line_label}_wave{i}"

            if flux_column in table.columns.names:

                line_wavelength = (
                    table[wave_column][0])
                
                if line_wavelength == 0:
                    int_flux_col_name = f"int_{flux_column}"
                    int_flux_err_col_name = f"int_{flux_err_column}"
                    new_columns.append(fits.Column(name=int_flux_col_name, format="D", array=table[flux_column]))
                    new_columns.append(fits.Column(name=int_flux_err_col_name, format="D", array=table[flux_err_column]))

                else:
                    f_line = extinction_model(line_wavelength * u.AA)
                    A_line = data['E(B-V)'] * f_line
                    A_line_err = data['E(B-V)_err'] * f_line
                    observed_flux = table[flux_column]
                    observed_flux_err = table[flux_err_column]
                    intrinsic_flux = observed_flux * 10 ** (0.4 * A_line)
                    intrinsic_flux_err = intrinsic_flux * np.sqrt(
                        (observed_flux_err / observed_flux) ** 2
                        + (0.4 * A_line_err * np.log(10)) ** 2
                    )

                    int_flux_col_name = f"int_{flux_column}"
                    int_flux_err_col_name = f"int_{flux_err_column}"
                    new_columns.append(fits.Column(name=int_flux_col_name, format="D", array=intrinsic_flux))
                    new_columns.append(fits.Column(name=int_flux_err_col_name, format="D", array=intrinsic_flux_err))

    # Add E(B-V) columns
    new_columns.append(fits.Column(name="E_B_V", format="D", array=data['E(B-V)'] * np.ones(len(table))))
    new_columns.append(fits.Column(name="E_B_V_err", format="D", array=data['E(B-V)_err'] * np.ones(len(table))))
    print('mean , median:', np.nanmean(data['E(B-V)']), np.nanmedian(data['E(B-V)']), 'err:', np.nanmean(data['E(B-V)_err']), np.nanmedian(data['E(B-V)_err']))

    # Write to a new FITS file
    cols = fits.ColDefs(new_columns)
    hdu = fits.BinTableHDU.from_columns(cols)
    output_filename = f"/Users/amritasingh/LVM_lagoon_outputs/condition2_cont/lagoon_sframe_obs_corr_flux_table_dec11_newdrp_using_ebv_map.fits"
    hdu.writeto(output_filename, overwrite=True)
    print(f"File saved: {output_filename}")

# Input
filename = "/Users/amritasingh/LVM_lagoon_outputs/condition2_cont/lagoon_sframe_obs_flux_table_dec11_newdrp.fits"

lines_to_correct = {

    'NII_OII' : [4607.16],
    #'[SII]4069': [4068.6], 
    '[SII]6717': [6716.44],
    '[SII]6731': [6730.82],

    '[OIII]4363': [4363.2],
    '[OIII]4959': [4958.91],
    '[OIII]5007': [5006.84],

    '[OII]3726': [3726.03],
    '[OII]7320': [7319.99],
    '[OII]7330': [7330],

    'Ha6563': [6562.8],
    'Hb4861': [4861.32],
    'Hgm4340': [4340.46],
    'HI3750': [3750.15],
    'HI3835': [3835.38],
    'HI4102': [4101.74],
    'HI8863': [8862.78],
    'HI9015': [9014.91],
    'HI9229': [9229.01],
    ##'OII4638': [4638.86],
    #'[ClIII]5517':[5517.71],
    #'[ClIII]5537':[5537.88] 

    }

dust_corr_all_ratios(filename, lines_to_correct)


/var/folders/01/0zj4nff14gx4hd80g0nl5gbr0000gn/T/ipykernel_901/2606685435.py:45: RuntimeWarning: divide by zero encountered in divide
  (observed_flux_err / observed_flux) ** 2
/var/folders/01/0zj4nff14gx4hd80g0nl5gbr0000gn/T/ipykernel_901/2606685435.py:45: RuntimeWarning: overflow encountered in square
  (observed_flux_err / observed_flux) ** 2
/var/folders/01/0zj4nff14gx4hd80g0nl5gbr0000gn/T/ipykernel_901/2606685435.py:44: RuntimeWarning: invalid value encountered in multiply
  intrinsic_flux_err = intrinsic_flux * np.sqrt(


mean , median: 1.2671464494264586 1.1651680812278076 err: 2.3262631258764847e+23 0.30632132642290877
File saved: /Users/amritasingh/LVM_lagoon_outputs/condition2_cont/lagoon_sframe_obs_corr_flux_table_dec11_newdrp_using_ebv_map.fits


In [ ]:
## Steps after fiting the combined rss sframes:
#apply the condition to remove the bright spaxels with star in them
# I shall have ~16800 fibers left after this cutoff

In [ ]:
'''
##################################################### FITTING OF LINES IN UNMASKED REGION #####################################################

# Initializing linefit dictinory to store rest frame wavelengths, line range and the underlying left and right continua

line_fitting = {
            'Ha6563': [6562.8, [6560, 6566], [6535, 6545], [6590, 6605]],

            'NII_OII' : [4607.16, [4606.4, 4678], [4600, 4605], [4680, 4694]],

            #'[SII]4069': [4068.6, [4067, 4080], [4055, 4065],[4083, 4098]],  
            '[SII]6717': [6716.44, [6714, 6720], [6680, 6700], [6735, 6750]],
            '[SII]6731': [6730.82, [6728, 6733], [6684, 6702], [6735, 6750]],

            #'[NII]5755': [5754.64, [5752, 5758], [5720, 5745], [5760, 5775]],
            #'[NII]6548': [6548.04, [6546, 6551], [6525, 6540], [6590, 6605]],  
            '[NII]6584': [6583.46, [6581, 6586], [6535, 6545], [6590, 6605]],

            '[OIII]4363': [4363.2,[4359, 4367], [4346, 4355],  [4370, 4385]],
            '[OIII]4959': [4958.91,[4955, 4962], [4830, 4850],  [5020, 5035]],
            '[OIII]5007': [5006.84,[5004, 5010], [4830, 4850],  [5020, 5035]],

            #'[SIII]6312': [6312.1, [6310, 6315], [6260, 6290], [6320, 6330]],
            #'[SIII]9069': [9068.6, [9066, 9072], [9025, 9035], [9080, 9100]],
            #'[SIII]9531': [9530.6, [9528, 9535], [9480, 9510], [9555, 9580]],  

            '[OII]3726': [3726.03, [3723, 3734], [3650, 3675], [3755, 3765]],
            '[OII]7320': [7319.99, [7317, 7324], [7300, 7315], [7345, 7360]],
            '[OII]7330': [7330, [7327, 7335], [7300, 7315], [7345, 7360]],

            #'[ClIII]5517':[5517.71, [5515, 5520], [5490, 5510], [5522, 5535]],
            #'[ClIII]5537':[5537.88, [5536, 5541], [5520, 5535], [5546, 5570]],

            'Hb4861': [4861.32, [4858, 4866], [4820, 4850], [4870, 4900]],
            #'Hgm4340': [4340.463, [4338, 4344], [4310, 4335], [4345, 4355]]
            
            
              }

linename = []
lines = []
line_mask_arr = []
left_mask_arr = []
right_mask_arr = []

#Iterating over the above linefitdict keys and values to append values to above lists
for key, values in zip(line_fitting.keys(), line_fitting.values()):

    linename.append(key)
    lines.append(values[0])
    line_mask_arr.append(values[1]) 
    left_mask_arr.append(values[2]) 
    right_mask_arr.append(values[3]) 


#linefitdict = {'RA': [], 'Dec': []}  # Dictionary to store data for each line in each fiber from each tile

linefitdict = {'RA': [], 'Dec': []}  # Dictionary to store data for each line in each fiber from each tile

# Initialize the dictionary to store fitting results
num = np.linspace(1, 10, 10, dtype = int)

for i, j in enumerate(linename):
    for k in num:  # Assuming up to 10 components 
        linefitdict[f'{j}_flux{k-1}'] = []
        linefitdict[f'{j}_flux{k-1}_err'] = []
        linefitdict[f'{j}_wave{k-1}'] = []
        linefitdict[f'{j}_wave{k-1}_err'] = []
        linefitdict[f'{j}_sigma{k-1}'] = []
    linefitdict[f'{j}_cont'] = []
    linefitdict[f'{j}_cont_err'] = []


# Reading fits file
with fits.open('/Users/amritasingh/LVM_lagoon_outputs/condition2_cont/annular_binned/region_masks/v1_unmasked_lagoon_spectra.fits') as hdul:  # reading the fits file

    flux = hdul['flux'].data*1e17
    error = hdul['error'].data*1e17
    wave = hdul['wave'].data
    table = hdul['table'].data
    RA =  table['RA']
    Dec = table['Dec']

    # Reading flux and error in each fiber
    for idx in range(len(flux)):  
        spaxel_flux = flux[idx]
        spaxel_error = error[idx]
        spaxel_wave = wave
        RA_fib = RA[idx]
        Dec_fib = Dec[idx]


        # Store the fiber number and tile id for each iteration
        
        linefitdict['RA'].append(RA_fib)
        linefitdict['Dec'].append(Dec_fib)
        
        # Iterating over each line and masks to select emission line ranges with their underlying left and right continuum masks
        for lineid, line_data in line_fitting.items():
            
            # Extracting data for line fitting
            line, line_mask, left_mask, right_mask = line_data

            if lineid == '[OII]3726':

                total_mask = mklinemask(spaxel_wave, line_mask, left_mask, right_mask, line)
                #Fit 2 gaussians to [OII] doublets 3726 and 3729 \lambda (3726 is written as line+2.8)

                popt, pcov = fit_double_gauss(spaxel_wave[total_mask], spaxel_flux[total_mask], spaxel_error[total_mask], line, line+2.8, plot = False, plotout=f'masked_individual_fib_{lineid}_{idx}')


                # Store data for 1st component
                linefitdict[lineid+'_flux0'].append(popt[0]*1e-17)                 
                linefitdict[lineid+'_flux0_err'].append(np.sqrt(pcov[0,0])*1e-17)
                linefitdict[lineid+'_wave0'].append(popt[1])
                linefitdict[lineid+'_wave0_err'].append(np.sqrt(pcov[1,1]))
                linefitdict[lineid+'_sigma0'].append(popt[2])


                # Store data for 2nd component
                linefitdict[lineid+'_flux1'].append(popt[3]*1e-17)
                linefitdict[lineid+'_flux1_err'].append(np.sqrt(pcov[3,3])*1e-17)
                linefitdict[lineid+'_wave1'].append(popt[4])
                linefitdict[lineid+'_wave1_err'].append(np.sqrt(pcov[4,4]))
                linefitdict[lineid+'_sigma1'].append(popt[5])


                # Store continuum data
                linefitdict[lineid+'_cont'].append(popt[6]*1e-17)
                linefitdict[lineid+'_cont_err'].append(np.sqrt(pcov[6,6]*1e-17))

                num = np.linspace(2, 9, 8, dtype =int)
                for i in num:
                    i = str(i)

                    linefitdict[lineid+'_flux'+i].append(0)
                    linefitdict[lineid+'_flux'+i+'_err'].append(0)
                    linefitdict[lineid+'_wave'+i].append(0)
                    linefitdict[lineid+'_wave'+i+'_err'].append(0)
                    linefitdict[lineid+'_sigma'+i].append(0)

            elif lineid == '[OIII]4363':

                total_mask = mklinemask(spaxel_wave, line_mask, left_mask, right_mask, line)

                #Fit 2 gaussians to [FeII]4360 and [OIII]4363 \lambda lines (4360 \lambda is written as line-3.2)
                popt, pcov = fit_double_gauss(spaxel_wave[total_mask], spaxel_flux[total_mask], spaxel_error[total_mask], line-3.2, line, plot = False, plotout=f'masked_individual_fib_{lineid}_{idx}')


                # Store data for 1st component       
                linefitdict[lineid+'_flux0'].append(popt[0]*1e-17)
                linefitdict[lineid+'_flux0_err'].append(np.sqrt(pcov[0,0])*1e-17)
                linefitdict[lineid+'_wave0'].append(popt[1])
                linefitdict[lineid+'_wave0_err'].append(np.sqrt(pcov[1,1]))
                linefitdict[lineid+'_sigma0'].append(popt[2])

                # Store data for 2nd component
                linefitdict[lineid+'_flux1'].append(popt[3]*1e-17)
                linefitdict[lineid+'_flux1_err'].append(np.sqrt(pcov[3,3])*1e-17)
                linefitdict[lineid+'_wave1'].append(popt[4])
                linefitdict[lineid+'_wave1_err'].append(np.sqrt(pcov[4,4]))
                linefitdict[lineid+'_sigma1'].append(popt[5])

                # Store continuum data
                linefitdict[lineid+'_cont'].append(popt[6]*1e-17)
                linefitdict[lineid+'_cont_err'].append(np.sqrt(pcov[6,6])*1e-17)

                num = np.linspace(2, 9, 8, dtype =int)
                for i in num:
                    i = str(i)

                    linefitdict[lineid+'_flux'+i].append(0)
                    linefitdict[lineid+'_flux'+i+'_err'].append(0)
                    linefitdict[lineid+'_wave'+i].append(0)
                    linefitdict[lineid+'_wave'+i+'_err'].append(0)
                    linefitdict[lineid+'_sigma'+i].append(0)

            elif lineid == '[SII]4069':
                
                total_mask = mklinemask(spaxel_wave, line_mask, left_mask, right_mask, line)

                #Fit 3 Gaussians to 4069, 4073 and 4076 \lambda lines, the rest frame lambda of other two lines are defined as line+c (c is a const)
                popt, pcov = fit_triple_gauss(spaxel_wave[total_mask], spaxel_flux[total_mask], spaxel_error[total_mask], line, line+5, line+7.5, plot = False, plotout=f'masked_individual_fib_{lineid}_{idx}')

                for i in range(14):

                    if popt[i]<0:
                        popt[i] = np.nan
                        pcov[i, i] = np.nan

                # Store data for 1st component
                linefitdict[lineid+'_flux0'].append(popt[0]* 1e-17)
                linefitdict[lineid+'_flux0_err'].append(np.sqrt(pcov[0,0])* 1e-17)
                linefitdict[lineid+'_wave0'].append(popt[1])
                linefitdict[lineid+'_wave0_err'].append(np.sqrt(pcov[1,1]))
                linefitdict[lineid+'_sigma0'].append(popt[2])

                # Store data for 2nd component
                linefitdict[lineid+'_flux1'].append(popt[3]* 1e-17)
                linefitdict[lineid+'_flux1_err'].append(np.sqrt(pcov[3,3])* 1e-17)
                linefitdict[lineid+'_wave1'].append(popt[4])
                linefitdict[lineid+'_wave1_err'].append(np.sqrt(pcov[4,4]))
                linefitdict[lineid+'_sigma1'].append(popt[5])

                # Store data for 3rd component
                linefitdict[lineid+'_flux2'].append(popt[6]* 1e-17)
                linefitdict[lineid+'_flux2_err'].append(np.sqrt(pcov[6, 6])* 1e-17)
                linefitdict[lineid+'_wave2'].append(popt[7])
                linefitdict[lineid+'_wave2_err'].append(np.sqrt(pcov[7,7]))     ############## 
                linefitdict[lineid+'_sigma2'].append(popt[8])


                # Store continuum data
                linefitdict[lineid+'_cont'].append(popt[9]* 1e-17)
                linefitdict[lineid+'_cont_err'].append(np.sqrt(pcov[9,9])* 1e-17)

                num = np.linspace(3, 9, 7, dtype =int)
                for i in num:
                    i = str(i)

                    linefitdict[lineid+'_flux'+i].append(0)
                    linefitdict[lineid+'_flux'+i+'_err'].append(0)
                    linefitdict[lineid+'_wave'+i].append(0)
                    linefitdict[lineid+'_wave'+i+'_err'].append(0)
                    linefitdict[lineid+'_sigma'+i].append(0)

            elif lineid == 'NII_OII':

                total_mask = mklinemask(spaxel_wave, line_mask, left_mask, right_mask, line)

                popt, pcov = fit_ten_gauss(spaxel_wave[total_mask], spaxel_flux[total_mask], spaxel_error[total_mask], line, plot = True, plotout=f'masked_individual_fib_{lineid}_{idx}')

                for i in range(14):

                    if popt[i]<0:
                        popt[i] = np.nan
                        pcov[i, i] = np.nan

                #print(f"fiber no. {fibid_fib}, tile id: {tileid_fib}, popt shape: {popt.shape}, popt 10: {popt[10]}")

                #print(f)
                
                # Store data for 1st component
                linefitdict[lineid+'_flux0'].append(popt[0]*1e-17)
                linefitdict[lineid+'_flux0_err'].append(np.sqrt(pcov[0,0])*1e-17)
                linefitdict[lineid+'_wave0'].append(popt[1])
                linefitdict[lineid+'_wave0_err'].append(np.sqrt(pcov[1,1]))
                linefitdict[lineid+'_sigma0'].append(popt[2])

                ############ 3rd paramter is velocity #############

                # Store data for 2nd component
                linefitdict[lineid+'_flux1'].append(popt[4]*1e-17)
                linefitdict[lineid+'_flux1_err'].append(np.sqrt(pcov[4, 4])*1e-17)
                linefitdict[lineid+'_wave1'].append(0)
                linefitdict[lineid+'_wave1_err'].append(0)
                linefitdict[lineid+'_sigma1'].append(0)

                # Store data for 3rd component
                linefitdict[lineid+'_flux2'].append(popt[5]*1e-17)
                linefitdict[lineid+'_flux2_err'].append(np.sqrt(pcov[5, 5])*1e-17)
                linefitdict[lineid+'_wave2'].append(0)
                linefitdict[lineid+'_wave2_err'].append(0)     ############## 
                linefitdict[lineid+'_sigma2'].append(0)

                # Store data for 4th component
                linefitdict[lineid+'_flux3'].append(popt[6]*1e-17)
                linefitdict[lineid+'_flux3_err'].append(np.sqrt(pcov[6, 6])*1e-17)
                linefitdict[lineid+'_wave3'].append(0)
                linefitdict[lineid+'_wave3_err'].append(np.sqrt(0))     ############## 
                linefitdict[lineid+'_sigma3'].append(0)

                # Store data for 5th component
                linefitdict[lineid+'_flux4'].append(popt[7]*1e-17)
                linefitdict[lineid+'_flux4_err'].append(np.sqrt(pcov[7, 7])*1e-17)
                linefitdict[lineid+'_wave4'].append(0)
                linefitdict[lineid+'_wave4_err'].append(0)     ############## 
                linefitdict[lineid+'_sigma4'].append(0)

                # Store data for 6th component
                linefitdict[lineid+'_flux5'].append(popt[8]*1e-17)
                linefitdict[lineid+'_flux5_err'].append(np.sqrt(pcov[8, 8])*1e-17)
                linefitdict[lineid+'_wave5'].append(0)
                linefitdict[lineid+'_wave5_err'].append(0)     ############## 
                linefitdict[lineid+'_sigma5'].append(0)

                # Store data for 7th component
                linefitdict[lineid+'_flux6'].append(popt[9]*1e-17)
                linefitdict[lineid+'_flux6_err'].append(np.sqrt(pcov[9, 9])*1e-17)
                linefitdict[lineid+'_wave6'].append(0)
                linefitdict[lineid+'_wave6_err'].append(0)     ############## 
                linefitdict[lineid+'_sigma6'].append(0)

                # Store data for 8th component
                linefitdict[lineid+'_flux7'].append(popt[10]*1e-17)
                linefitdict[lineid+'_flux7_err'].append(np.sqrt(pcov[10, 10])*1e-17)
                linefitdict[lineid+'_wave7'].append(0)
                linefitdict[lineid+'_wave7_err'].append(0)     ############## 
                linefitdict[lineid+'_sigma7'].append(0)

                # Store data for 9th component
                linefitdict[lineid+'_flux8'].append(popt[11]*1e-17)
                linefitdict[lineid+'_flux8_err'].append(np.sqrt(pcov[11, 11])*1e-17)
                linefitdict[lineid+'_wave8'].append(0)
                linefitdict[lineid+'_wave8_err'].append(0)     ############## 
                linefitdict[lineid+'_sigma8'].append(0)

                # Store data for 10th component
                linefitdict[lineid+'_flux9'].append(popt[12]*1e-17)
                linefitdict[lineid+'_flux9_err'].append(np.sqrt(pcov[12, 12])*1e-17)
                linefitdict[lineid+'_wave9'].append(0)
                linefitdict[lineid+'_wave9_err'].append(0)     ############## 
                linefitdict[lineid+'_sigma9'].append(0)

                # Store continuum data
                linefitdict[lineid+'_cont'].append(popt[13]*1e-17)
                linefitdict[lineid+'_cont_err'].append(np.sqrt(pcov[13, 13])*1e-17)

            else:

                total_mask = mklinemask(spaxel_wave, line_mask, left_mask, right_mask, line)

                # Single Gaussian fitting
                popt, pcov = fit_gauss(spaxel_wave[total_mask], spaxel_flux[total_mask], spaxel_error[total_mask], line, plot = False, plotout=f'masked_individual_fib_{lineid}_{idx}')

                # Store fitting parameters
                linefitdict[lineid+'_flux0'].append(popt[0]*1e-17)
                linefitdict[lineid+'_flux0_err'].append(np.sqrt(pcov[0, 0])*1e-17)
                linefitdict[lineid+'_wave0'].append(popt[1])
                linefitdict[lineid+'_wave0_err'].append(np.sqrt(pcov[1, 1]))
                linefitdict[lineid+'_sigma0'].append(popt[2])

                linefitdict[lineid+'_cont'].append(popt[3]*1e-17)
                linefitdict[lineid+'_cont_err'].append(np.sqrt(pcov[3, 3])*1e-17)

                # Other componets store 0 for single gaussian fit

                num = np.linspace(1, 9, 9, dtype =int)
                for i in num:
                    i = str(i)

                    linefitdict[lineid+'_flux'+i].append(0)
                    linefitdict[lineid+'_flux'+i+'_err'].append(0)
                    linefitdict[lineid+'_wave'+i].append(0)
                    linefitdict[lineid+'_wave'+i+'_err'].append(0)
                    linefitdict[lineid+'_sigma'+i].append(0)
                    

# Ensuring all lists have the same length by filling missing values with zeros
max_length = max(len(lst) for lst in linefitdict.values())
for key in linefitdict:
    if len(linefitdict[key]) < max_length:
        linefitdict[key].extend([0] * (max_length - len(linefitdict[key])))
 
linefitdict = {key: np.array(value) for key, value in linefitdict.items()}

# Create an Astropy Table
#table = Table(linefitdict)

# Write the table to a FITS file
#table.write('/Users/amritasingh/LVM_lagoon_outputs/condition2_cont/annular_binned/region_masks/v1_unmasked_lagoon_obs_table.fits', overwrite=True)
'''

'\n##################################################### FITTING OF LINES IN UNMASKED REGION #####################################################\n\n# Initializing linefit dictinory to store rest frame wavelengths, line range and the underlying left and right continua\n\nline_fitting = {\n            \'Ha6563\': [6562.8, [6560, 6566], [6535, 6545], [6590, 6605]],\n\n            \'NII_OII\' : [4607.16, [4606.4, 4678], [4600, 4605], [4680, 4694]],\n\n            #\'[SII]4069\': [4068.6, [4067, 4080], [4055, 4065],[4083, 4098]],  \n            \'[SII]6717\': [6716.44, [6714, 6720], [6680, 6700], [6735, 6750]],\n            \'[SII]6731\': [6730.82, [6728, 6733], [6684, 6702], [6735, 6750]],\n\n            #\'[NII]5755\': [5754.64, [5752, 5758], [5720, 5745], [5760, 5775]],\n            #\'[NII]6548\': [6548.04, [6546, 6551], [6525, 6540], [6590, 6605]],  \n            \'[NII]6584\': [6583.46, [6581, 6586], [6535, 6545], [6590, 6605]],\n\n            \'[OIII]4363\': [4363.2,[4359, 4367], [

In [ ]:

##################################################### FITTING OF LINES IN EACH FIBER SPECTRUM #####################################################

# Initializing linefit dictinory to store rest frame wavelengths, line range and the underlying left and right continua
'''
line_fitting = {

            'HI3750': [3750.15 , [3747.5,3753.5], [3738,3743], [3758,3765]], # H_kappa           
            'HI3771': [3770.63 , [3768.5,3753.5], [3758,3764], [3776,3786]], # H_iota
            'HI3835': [3835.38 , [3833,3839], [3824,3832], [3842,3852]], # H_eta
            'HI4102': [4101.74 , [4100,4105], [4085,4095], [4110,4120]], # H_delta
            #'HI8863': [ , [,], [,], [,]], # P_11        
            #'HI9015': [ , [,], [,], [,]], # P_10
            #'HI9229': [ , [,], [,], [,]], # P_9

            'Ha6563': [6562.8, [6560, 6566], [6535, 6545], [6590, 6605]],

            #'NII_OII' : [4607.16, [4606.4, 4678], [4600, 4605], [4680, 4694]],
#
            ##'[SII]4069': [4068.6, [4067, 4080], [4055, 4065],[4083, 4098]],  
            #'[SII]6717': [6716.44, [6714, 6720], [6680, 6700], [6735, 6750]],
            #'[SII]6731': [6730.82, [6728, 6733], [6684, 6702], [6735, 6750]],
#
            #'[NII]5755': [5754.64, [5752, 5758], [5720, 5745], [5760, 5775]],
            ##'[NII]6548': [6548.04, [6546, 6551], [6525, 6540], [6590, 6605]],  
            #'[NII]6584': [6583.46, [6581, 6586], [6535, 6545], [6590, 6605]],
#
            #'[OIII]4363': [4363.2,[4359, 4367], [4346, 4355],  [4370, 4385]],
            #'[OIII]4959': [4958.91,[4955, 4962], [4830, 4850],  [5020, 5035]],
            #'[OIII]5007': [5006.84,[5004, 5010], [4830, 4850],  [5020, 5035]],
#
            ##'[SIII]6312': [6312.1, [6310, 6315], [6260, 6290], [6320, 6330]],
            ##'[SIII]9069': [9068.6, [9066, 9072], [9025, 9035], [9080, 9100]],
            ##'[SIII]9531': [9530.6, [9528, 9535], [9480, 9510], [9555, 9580]],  
#
            #'[OII]3726': [3726.03, [3723, 3734], [3650, 3675], [3755, 3765]],
            #'[OII]7320': [7319.99, [7317, 7324], [7300, 7315], [7345, 7360]],
            #'[OII]7330': [7330, [7327, 7335], [7300, 7315], [7345, 7360]],

            #'[ClIII]5517':[5517.71, [5515, 5520], [5490, 5510], [5522, 5535]],
            #'[ClIII]5537':[5537.88, [5536, 5541], [5520, 5535], [5546, 5570]],

            'Hb4861': [4861.32, [4858, 4866], [4820, 4850], [4870, 4900]],
            'Hgm4340': [4340.463, [4338, 4344], [4310, 4335], [4345, 4355]]
            
            
              }

linename = []
lines = []
line_mask_arr = []
left_mask_arr = []
right_mask_arr = []

#Iterating over the above linefitdict keys and values to append values to above lists
for key, values in zip(line_fitting.keys(), line_fitting.values()):

    linename.append(key)
    lines.append(values[0])
    line_mask_arr.append(values[1]) 
    left_mask_arr.append(values[2]) 
    right_mask_arr.append(values[3]) 


#linefitdict = {'RA': [], 'Dec': []}  # Dictionary to store data for each line in each fiber from each tile

linefitdict = {'fiberid': [], 'tileid': [], 'RA': [], 'Dec': []}  # Dictionary to store data for each line in each fiber from each tile

# Initialize the dictionary to store fitting results
num = np.linspace(1, 10, 10, dtype = int)

for i, j in enumerate(linename):
    for k in num:  # Assuming up to 10 components 
        linefitdict[f'{j}_flux{k-1}'] = []
        linefitdict[f'{j}_flux{k-1}_err'] = []
        linefitdict[f'{j}_wave{k-1}'] = []
        linefitdict[f'{j}_wave{k-1}_err'] = []
        linefitdict[f'{j}_sigma{k-1}'] = []
    linefitdict[f'{j}_cont'] = []
    linefitdict[f'{j}_cont_err'] = []


# Reading fits file
with fits.open('/Volumes/amrita/LVM/lagoon_outputs/condition2_cont.fits') as hdu:  # reading the fits file

    data = hdu[1].data

    wave = data['wave']
    flux = data['flux']* 1e17
    error = data['error']* 1e17

    fiberid = data['fiberid']
    tileid = data['tileid']
    RA = data['RA']
    Dec = data['Dec']
    
    print(flux.shape, np.nanmedian(data['flux']))

    # Reading flux and error in each fiber
    for i, j in enumerate(fiberid):

        spaxel_flux = flux[i]/978.18
        spaxel_error = error[i]/978.18
        RA_fib = RA[i]
        Dec_fib = Dec[i]
        fibid_fib = fiberid[i]
        tileid_fib = tileid[i]
        spaxel_wave = wave[i]

        # Store the fiber number and tile id for each iteration
        
        linefitdict['RA'].append(RA_fib)
        linefitdict['Dec'].append(Dec_fib)
        linefitdict['fiberid'].append(fibid_fib)
        linefitdict['tileid'].append(tileid_fib)


        
        # Iterating over each line and masks to select emission line ranges with their underlying left and right continuum masks
        for lineid, line_data in line_fitting.items():
            
            # Extracting data for line fitting
            line, line_mask, left_mask, right_mask = line_data

            if lineid == '[OII]3726':

                total_mask = mklinemask(spaxel_wave, line_mask, left_mask, right_mask, line)
                #Fit 2 gaussians to [OII] doublets 3726 and 3729 \lambda (3726 is written as line+2.8)

                popt, pcov = fit_double_gauss(spaxel_wave[total_mask], spaxel_flux[total_mask], spaxel_error[total_mask], line, line+2.8, plot = False, plotout=f'individual_fib_{lineid}_{j}')


                # Store data for 1st component
                linefitdict[lineid+'_flux0'].append(popt[0]*1e-17)                 
                linefitdict[lineid+'_flux0_err'].append(np.sqrt(pcov[0,0])*1e-17)
                linefitdict[lineid+'_wave0'].append(popt[1])
                linefitdict[lineid+'_wave0_err'].append(np.sqrt(pcov[1,1]))
                linefitdict[lineid+'_sigma0'].append(popt[2])


                # Store data for 2nd component
                linefitdict[lineid+'_flux1'].append(popt[3]*1e-17)
                linefitdict[lineid+'_flux1_err'].append(np.sqrt(pcov[3,3])*1e-17)
                linefitdict[lineid+'_wave1'].append(popt[4])
                linefitdict[lineid+'_wave1_err'].append(np.sqrt(pcov[4,4]))
                linefitdict[lineid+'_sigma1'].append(popt[5])


                # Store continuum data
                linefitdict[lineid+'_cont'].append(popt[6]*1e-17)
                linefitdict[lineid+'_cont_err'].append(np.sqrt(pcov[6,6]*1e-17))

                num = np.linspace(2, 9, 8, dtype =int)
                for i in num:
                    i = str(i)

                    linefitdict[lineid+'_flux'+i].append(0)
                    linefitdict[lineid+'_flux'+i+'_err'].append(0)
                    linefitdict[lineid+'_wave'+i].append(0)
                    linefitdict[lineid+'_wave'+i+'_err'].append(0)
                    linefitdict[lineid+'_sigma'+i].append(0)

            elif lineid == '[OIII]4363':

                total_mask = mklinemask(spaxel_wave, line_mask, left_mask, right_mask, line)

                #Fit 2 gaussians to [FeII]4360 and [OIII]4363 \lambda lines (4360 \lambda is written as line-3.2)
                popt, pcov = fit_double_gauss(spaxel_wave[total_mask], spaxel_flux[total_mask], spaxel_error[total_mask], line-3.2, line, plot = False, plotout=f'individual_fib_{lineid}_{j}')


                # Store data for 1st component       
                linefitdict[lineid+'_flux0'].append(popt[0]*1e-17)
                linefitdict[lineid+'_flux0_err'].append(np.sqrt(pcov[0,0])*1e-17)
                linefitdict[lineid+'_wave0'].append(popt[1])
                linefitdict[lineid+'_wave0_err'].append(np.sqrt(pcov[1,1]))
                linefitdict[lineid+'_sigma0'].append(popt[2])

                # Store data for 2nd component
                linefitdict[lineid+'_flux1'].append(popt[3]*1e-17)
                linefitdict[lineid+'_flux1_err'].append(np.sqrt(pcov[3,3])*1e-17)
                linefitdict[lineid+'_wave1'].append(popt[4])
                linefitdict[lineid+'_wave1_err'].append(np.sqrt(pcov[4,4]))
                linefitdict[lineid+'_sigma1'].append(popt[5])

                # Store continuum data
                linefitdict[lineid+'_cont'].append(popt[6]*1e-17)
                linefitdict[lineid+'_cont_err'].append(np.sqrt(pcov[6,6])*1e-17)

                num = np.linspace(2, 9, 8, dtype =int)
                for i in num:
                    i = str(i)

                    linefitdict[lineid+'_flux'+i].append(0)
                    linefitdict[lineid+'_flux'+i+'_err'].append(0)
                    linefitdict[lineid+'_wave'+i].append(0)
                    linefitdict[lineid+'_wave'+i+'_err'].append(0)
                    linefitdict[lineid+'_sigma'+i].append(0)

            elif lineid == '[SII]4069':
                
                total_mask = mklinemask(spaxel_wave, line_mask, left_mask, right_mask, line)

                #Fit 3 Gaussians to 4069, 4073 and 4076 \lambda lines, the rest frame lambda of other two lines are defined as line+c (c is a const)
                popt, pcov = fit_triple_gauss(spaxel_wave[total_mask], spaxel_flux[total_mask], spaxel_error[total_mask], line, line+5, line+7.5, plot = False, plotout=f'individual_fib_{lineid}_{j}')

                for i in range(14):

                    if popt[i]<0:
                        popt[i] = np.nan
                        pcov[i, i] = np.nan

                # Store data for 1st component
                linefitdict[lineid+'_flux0'].append(popt[0]* 1e-17)
                linefitdict[lineid+'_flux0_err'].append(np.sqrt(pcov[0,0])* 1e-17)
                linefitdict[lineid+'_wave0'].append(popt[1])
                linefitdict[lineid+'_wave0_err'].append(np.sqrt(pcov[1,1]))
                linefitdict[lineid+'_sigma0'].append(popt[2])

                # Store data for 2nd component
                linefitdict[lineid+'_flux1'].append(popt[3]* 1e-17)
                linefitdict[lineid+'_flux1_err'].append(np.sqrt(pcov[3,3])* 1e-17)
                linefitdict[lineid+'_wave1'].append(popt[4])
                linefitdict[lineid+'_wave1_err'].append(np.sqrt(pcov[4,4]))
                linefitdict[lineid+'_sigma1'].append(popt[5])

                # Store data for 3rd component
                linefitdict[lineid+'_flux2'].append(popt[6]* 1e-17)
                linefitdict[lineid+'_flux2_err'].append(np.sqrt(pcov[6, 6])* 1e-17)
                linefitdict[lineid+'_wave2'].append(popt[7])
                linefitdict[lineid+'_wave2_err'].append(np.sqrt(pcov[7,7]))     ############## 
                linefitdict[lineid+'_sigma2'].append(popt[8])


                # Store continuum data
                linefitdict[lineid+'_cont'].append(popt[9]* 1e-17)
                linefitdict[lineid+'_cont_err'].append(np.sqrt(pcov[9,9])* 1e-17)

                num = np.linspace(3, 9, 7, dtype =int)
                for i in num:
                    i = str(i)

                    linefitdict[lineid+'_flux'+i].append(0)
                    linefitdict[lineid+'_flux'+i+'_err'].append(0)
                    linefitdict[lineid+'_wave'+i].append(0)
                    linefitdict[lineid+'_wave'+i+'_err'].append(0)
                    linefitdict[lineid+'_sigma'+i].append(0)

            elif lineid == 'NII_OII':

                total_mask = mklinemask(spaxel_wave, line_mask, left_mask, right_mask, line)

                popt, pcov = fit_ten_gauss(spaxel_wave[total_mask], spaxel_flux[total_mask], spaxel_error[total_mask], line, plot = False, plotout=f'individual_fib_{lineid}_{j}')

                for i in range(14):

                    if popt[i]<0:
                        popt[i] = np.nan
       

                #print(f"fiber no. {fibid_fib}, tile id: {tileid_fib}, popt shape: {popt.shape}, popt 10: {popt[10]}")
                
                # Store data for 1st component
                linefitdict[lineid+'_flux0'].append(popt[0]*1e-17)
                linefitdict[lineid+'_flux0_err'].append(np.sqrt(pcov[0,0])*1e-17)
                linefitdict[lineid+'_wave0'].append(popt[1])
                linefitdict[lineid+'_wave0_err'].append(np.sqrt(pcov[1,1]))
                linefitdict[lineid+'_sigma0'].append(popt[2])

                ############ 3rd paramter is velocity #############

                # Store data for 2nd component
                linefitdict[lineid+'_flux1'].append(popt[4]*1e-17)
                linefitdict[lineid+'_flux1_err'].append(np.sqrt(pcov[4, 4])*1e-17)
                linefitdict[lineid+'_wave1'].append(0)
                linefitdict[lineid+'_wave1_err'].append(0)
                linefitdict[lineid+'_sigma1'].append(0)

                # Store data for 3rd component
                linefitdict[lineid+'_flux2'].append(popt[5]*1e-17)
                linefitdict[lineid+'_flux2_err'].append(np.sqrt(pcov[5, 5])*1e-17)
                linefitdict[lineid+'_wave2'].append(0)
                linefitdict[lineid+'_wave2_err'].append(0)     ############## 
                linefitdict[lineid+'_sigma2'].append(0)

                # Store data for 4th component
                linefitdict[lineid+'_flux3'].append(popt[6]*1e-17)
                linefitdict[lineid+'_flux3_err'].append(np.sqrt(pcov[6, 6])*1e-17)
                linefitdict[lineid+'_wave3'].append(0)
                linefitdict[lineid+'_wave3_err'].append(np.sqrt(0))     ############## 
                linefitdict[lineid+'_sigma3'].append(0)

                # Store data for 5th component
                linefitdict[lineid+'_flux4'].append(popt[7]*1e-17)
                linefitdict[lineid+'_flux4_err'].append(np.sqrt(pcov[7, 7])*1e-17)
                linefitdict[lineid+'_wave4'].append(0)
                linefitdict[lineid+'_wave4_err'].append(0)     ############## 
                linefitdict[lineid+'_sigma4'].append(0)

                # Store data for 6th component
                linefitdict[lineid+'_flux5'].append(popt[8]*1e-17)
                linefitdict[lineid+'_flux5_err'].append(np.sqrt(pcov[8, 8])*1e-17)
                linefitdict[lineid+'_wave5'].append(0)
                linefitdict[lineid+'_wave5_err'].append(0)     ############## 
                linefitdict[lineid+'_sigma5'].append(0)

                # Store data for 7th component
                linefitdict[lineid+'_flux6'].append(popt[9]*1e-17)
                linefitdict[lineid+'_flux6_err'].append(np.sqrt(pcov[9, 9])*1e-17)
                linefitdict[lineid+'_wave6'].append(0)
                linefitdict[lineid+'_wave6_err'].append(0)     ############## 
                linefitdict[lineid+'_sigma6'].append(0)

                # Store data for 8th component
                linefitdict[lineid+'_flux7'].append(popt[10]*1e-17)
                linefitdict[lineid+'_flux7_err'].append(np.sqrt(pcov[10, 10])*1e-17)
                linefitdict[lineid+'_wave7'].append(0)
                linefitdict[lineid+'_wave7_err'].append(0)     ############## 
                linefitdict[lineid+'_sigma7'].append(0)

                # Store data for 9th component
                linefitdict[lineid+'_flux8'].append(popt[11]*1e-17)
                linefitdict[lineid+'_flux8_err'].append(np.sqrt(pcov[11, 11])*1e-17)
                linefitdict[lineid+'_wave8'].append(0)
                linefitdict[lineid+'_wave8_err'].append(0)     ############## 
                linefitdict[lineid+'_sigma8'].append(0)

                # Store data for 10th component
                linefitdict[lineid+'_flux9'].append(popt[12]*1e-17)
                linefitdict[lineid+'_flux9_err'].append(np.sqrt(pcov[12, 12])*1e-17)
                linefitdict[lineid+'_wave9'].append(0)
                linefitdict[lineid+'_wave9_err'].append(0)     ############## 
                linefitdict[lineid+'_sigma9'].append(0)

                # Store continuum data
                linefitdict[lineid+'_cont'].append(popt[13]*1e-17)
                linefitdict[lineid+'_cont_err'].append(np.sqrt(pcov[13, 13])*1e-17)

            else:

                total_mask = mklinemask(spaxel_wave, line_mask, left_mask, right_mask, line)

                # Single Gaussian fitting
                popt, pcov = fit_gauss(spaxel_wave[total_mask], spaxel_flux[total_mask], spaxel_error[total_mask], line, plot = False, plotout=f'individual_fib_{lineid}_{j}')

                print(popt)
                
                # Store fitting parameters
                linefitdict[lineid+'_flux0'].append(popt[0]*1e-17)
                linefitdict[lineid+'_flux0_err'].append(np.sqrt(pcov[0, 0])*1e-17)
                linefitdict[lineid+'_wave0'].append(popt[1])
                linefitdict[lineid+'_wave0_err'].append(np.sqrt(pcov[1, 1]))
                linefitdict[lineid+'_sigma0'].append(popt[2])

                linefitdict[lineid+'_cont'].append(popt[3]*1e-17)
                linefitdict[lineid+'_cont_err'].append(np.sqrt(pcov[3, 3])*1e-17)

                # Other componets store 0 for single gaussian fit

                num = np.linspace(1, 9, 9, dtype =int)
                for i in num:
                    i = str(i)

                    linefitdict[lineid+'_flux'+i].append(0)
                    linefitdict[lineid+'_flux'+i+'_err'].append(0)
                    linefitdict[lineid+'_wave'+i].append(0)
                    linefitdict[lineid+'_wave'+i+'_err'].append(0)
                    linefitdict[lineid+'_sigma'+i].append(0)
                    
max_length = max(len(lst) for lst in linefitdict.values())
for key in linefitdict:
    if len(linefitdict[key]) < max_length:
        linefitdict[key].extend([0] * (max_length - len(linefitdict[key])))
 
linefitdict = {key: np.array(value) for key, value in linefitdict.items()}

table = Table(linefitdict)

#table.write('/Users/amritasingh/LVM_lagoon_outputs/condition2_cont/Oct_3_lagoon_FOV_obs_flux_table.fits', overwrite=True)
'''

KeyboardInterrupt: 

ValueError: Residuals are not finite in the initial point.

In [ ]:
with fits.open('/Users/amritasingh/LVM_lagoon_outputs/condition2_cont/v1_unmasked_lagoon_obs_table.fits') as hdu:

    header = hdu[1].header
    data = hdu[1].data

oiisum = data['NII_OII_flux3']+data['NII_OII_flux4']+data['NII_OII_flux5']+data['NII_OII_flux6']+data['NII_OII_flux8']+data['NII_OII_flux9']
#print(repr(header))

XTENSION= 'BINTABLE'           / binary table extension                         
BITPIX  =                    8 / array data type                                
NAXIS   =                    2 / number of array dimensions                     
NAXIS1  =                 5008 / length of dimension 1                          
NAXIS2  =                 3591 / length of dimension 2                          
PCOUNT  =                    0 / number of group parameters                     
GCOUNT  =                    1 / number of groups                               
TFIELDS =                  626 / number of table fields                         
TTYPE1  = 'RA      '                                                            
TFORM1  = 'D       '                                                            
TTYPE2  = 'Dec     '                                                            
TFORM2  = 'D       '                                                            
TTYPE3  = 'Ha6563_flux0'    

In [ ]:
#Fitting 10 gaussians to OII RLs
def ten_gaussian(wave, flux1, line1, sd1, vel_shift, flux2, flux3, flux4, flux5, flux6, flux7, flux8, flux9, flux10, cont):
    
    lw = [0, 14.19, 23.38, 31.69, 34.65, 35.93, 41.97, 43.68, 51.01, 54.47, 69.07]
    
    c = 299792.458  # Speed of light in km/s
    observed_wavelengths = (line1 + np.array(lw)) * (1 + vel_shift / c)

    return (cont +
            gaussian(wave, flux1, observed_wavelengths[0], sd1, 0) +
            gaussian(wave, flux2, observed_wavelengths[1], sd1, 0) +
            gaussian(wave, flux3, observed_wavelengths[2], sd1, 0) +
            gaussian(wave, flux4, observed_wavelengths[3], sd1, 0) +
            gaussian(wave, flux5, observed_wavelengths[4], sd1, 0) +
            gaussian(wave, 1.37*flux1, observed_wavelengths[5], sd1, 0) +
            gaussian(wave, flux6, observed_wavelengths[6], sd1, 0) +
            gaussian(wave, flux7, observed_wavelengths[7], sd1, 0)+
            gaussian(wave, flux8, observed_wavelengths[8], sd1, 0) +
            gaussian(wave, flux9, observed_wavelengths[9], sd1, 0) +
            gaussian(wave, flux10, observed_wavelengths[10], sd1, 0))

In [ ]:
'''
Function to fit 10 Gaussian to emission lines

Inputs: 
wave: wavelength 
spectrum: flux 
error: error 
lwave1: centroid wavlength of first line
lwave2: centroid wavlength of second line
lwave3: centroid wavlength of third line
dwave: difference in wavelength
plot: True/False to produce plots of linefits(boolean) 
plotout: plot name

Outputs:
Returns optimal parameters and covariance matrix on fitting the emission lines
'''

def fit_ten_gauss(wave, spectrum, error, lwave1, dwave=6, plot=False, plotout='linefit', bin='4', n_iter = 5):
    
    lw = [0, 14.19, 23.38, 31.69, 34.65, 35.93, 41.97, 43.68, 51.01, 54.47, 69.07]

    #making a selection on wavelength range, and finite flux and error values
    sel1 = (wave > (lwave1+lw[0]) - dwave / 2) * (wave < (lwave1+lw[0])+ dwave / 2) * (np.isfinite(spectrum)) * (np.isfinite(error))  
    sel2 = (wave > (lwave1+lw[1]) - dwave / 2) * (wave < (lwave1+lw[1])+ dwave / 2) * (np.isfinite(spectrum)) * (np.isfinite(error))
    sel3 = (wave > (lwave1+lw[2]) - dwave / 2) * (wave < (lwave1+lw[2])+ dwave / 2) * (np.isfinite(spectrum)) * (np.isfinite(error))
    sel4 = (wave > (lwave1+lw[3]) - dwave / 2) * (wave < (lwave1+lw[3])+ dwave / 2) * (np.isfinite(spectrum)) * (np.isfinite(error))
    sel5 = (wave > (lwave1+lw[4]) - dwave / 2) * (wave < (lwave1+lw[4])+ dwave / 2) * (np.isfinite(spectrum)) * (np.isfinite(error))
    sel6 = (wave > (lwave1+lw[6]) - dwave / 2) * (wave < (lwave1+lw[6])+ dwave / 2) * (np.isfinite(spectrum)) * (np.isfinite(error))
    sel7 = (wave > (lwave1+lw[7]) - dwave / 2) * (wave < (lwave1+lw[7])+ dwave / 2) * (np.isfinite(spectrum)) * (np.isfinite(error))
    sel8 = (wave > (lwave1+lw[8]) - dwave / 2) * (wave < (lwave1+lw[8])+ dwave / 2) * (np.isfinite(spectrum)) * (np.isfinite(error))
    sel9 = (wave > (lwave1+lw[9]) - dwave / 2) * (wave < (lwave1+lw[9])+ dwave / 2) * (np.isfinite(spectrum)) * (np.isfinite(error))
    sel10 = (wave > (lwave1+lw[10]) - dwave / 2) * (wave < (lwave1+lw[10])+ dwave / 2) * (np.isfinite(spectrum)) * (np.isfinite(error))

    sel = (wave > lwave1 - dwave / 2) * (wave < lwave1 + dwave ) * (np.isfinite(spectrum)) * (np.isfinite(error))
    
    # Initial paramets for the Gaussian fitting
    p0 = (np.abs(np.nanmax(spectrum[sel1])- np.nanmin(spectrum[sel1])) * np.sqrt(2 * np.pi) * 0.7,  lwave1, 0.7, 30, 
          np.abs(np.nanmax(spectrum[sel2])- np.nanmin(spectrum[sel2])) * np.sqrt(2 * np.pi) * 0.7,  
          np.abs(np.nanmax(spectrum[sel3])- np.nanmin(spectrum[sel3])) * np.sqrt(2 * np.pi) * 0.7,  
          np.abs(np.nanmax(spectrum[sel4])- np.nanmin(spectrum[sel4])) * np.sqrt(2 * np.pi) * 0.7,  
          np.abs(np.nanmax(spectrum[sel5])- np.nanmin(spectrum[sel5])) * np.sqrt(2 * np.pi) * 0.7,  
          np.abs(np.nanmax(spectrum[sel6])- np.nanmin(spectrum[sel6])) * np.sqrt(2 * np.pi) * 0.7,  
          np.abs(np.nanmax(spectrum[sel7])- np.nanmin(spectrum[sel7])) * np.sqrt(2 * np.pi) * 0.7,
          np.abs(np.nanmax(spectrum[sel8])- np.nanmin(spectrum[sel8])) * np.sqrt(2 * np.pi) * 0.7,
          np.abs(np.nanmax(spectrum[sel9])- np.nanmin(spectrum[sel9])) * np.sqrt(2 * np.pi) * 0.7,
          np.abs(np.nanmax(spectrum[sel10])- np.nanmin(spectrum[sel10])) * np.sqrt(2 * np.pi) * 0.7,  np.nanmin(spectrum[sel]))

    
    try:
        
        sel_goodpix = np.isfinite(spectrum)*np.isfinite(error)*np.isfinite(wave)
        
        # Calculating optimal parmeters and covariance matrix
        popt, pcov = curve_fit(ten_gaussian, wave[sel_goodpix], spectrum[sel_goodpix], sigma=error[sel_goodpix], p0=p0, absolute_sigma=True, bounds=((0, lwave1 - 3, 0.5, -np.inf, 0, 0, 0, 0, 0, 0, 0, 0, 0, -np.inf), (np.inf, lwave1+ 3, 0.9, np.inf, np.inf, np.inf, np.inf, np.inf, np.inf, np.inf, np.inf, np.inf, np.inf, np.inf)))
        
    except RuntimeError:
        
        popt = np.array([-99, lwave1, 0.7, -99, -99, -99, -99, -99, -99, -99, -99, -99, -99, np.nanmin(spectrum)])
        pcov = np.zeros((14, 14))
        pcov[0, 0] = np.sum(error ** 2)

    nparam=len(popt)  # no. of parameters
    xm = wave
    ym = ten_gaussian(xm, *popt)

    #Calculating weighted Chi**2 value
    chi2=np.nansum((spectrum-ym)**2/error**2)/(len(spectrum)-nparam)

    if plot==True:

        xm = wave
        #sigma = get_gaussian_ci(xm, gaussian, popt, pcov)

        plt.rcParams.update({'axes.titlesize': 'Large',
                 'axes.labelsize':'16.0',
                 'axes.linewidth':     '1.8' ,
                 'ytick.labelsize': '16.0',
                 'xtick.labelsize': '16.0',
                 'font.size': '10.0',
                 'legend.fontsize':'small'})
        
        fig = plt.figure(figsize = (15, 6))
        ax = fig.add_subplot()

        lim = 1.2*np.nanmax((spectrum-error))*1e-17

        #Plotting final output
        ##ax.step(wave, spectrum*1e-17, alpha=0.5, label='steps')
        ax.errorbar(wave, spectrum*1e-17, error*1e-17, c='k', label='data')
        ax.plot(xm, ym*1e-17, c='r', linestyle = '-', linewidth = 2, label='model')  
        ax.scatter(wave, spectrum*1e-17, c='b', alpha=0.5, s = 10, label='data points')  

        rest_positions1 = [4638.86, 4641.81, 4649.13, 4650.84, 4661.63, 4676.23]  
        rest_positions2 = [4607.16, 4621.35, 4630.54, 4643.09, 4658.5]  

        # Assuming constant velocity shift
        line_positions1 = rest_positions1   #*np.array(np.ones(len(rest_positions1)))*(1+np.divide(50, 299792.458)) # velocity shift calculated using 5007 line
        line_positions2 = rest_positions2   #*np.array(np.ones(len(rest_positions2)))*(1+np.divide(50, 299792.458)) # velocity shift calculated using 5007 line

        line_labels1 = ['OII 4638.86', 'OII 4641.81', 'OII 4649.13', 'OII 4650.84', 'OII 4661.63', 'OII 4676.23']

        line_labels2 = ['NII4607.16', 'NII4621.35', 'NII4630.54', 'NII4643.09', '[Fe III] 4658.5']
    
        # Annotate OII lines
        for pos1, label1 in zip(line_positions1, line_labels1):

            ax.axvline(x=pos1, color='r', linestyle='--', alpha=0.6, linewidth=2.8)
            ax.annotate(label1, xy=(pos1, lim), xytext=(pos1, lim),
                        rotation=90, verticalalignment='bottom', horizontalalignment='center')

        # Annotate NII and Fe III lines
        for pos2, label2 in zip(line_positions2, line_labels2):
            
            ax.axvline(x=pos2, color='deepskyblue', linestyle='--', linewidth=1.0)
            ax.annotate(label2, xy=(pos2, lim), xytext=(pos2, lim),
                        rotation=90, verticalalignment='bottom', horizontalalignment='center') 
                            
        ax.set_xlim(lwave1-dwave*2, lwave1+lw[9]+dwave*6)
        ax.set_ylim(np.nanmin([popt[13]*1e-17, np.nanmin((spectrum-error))*1e-17]), 1.5*np.nanmax([popt[10]*1e-17, np.nanmax(ym)*1e-17]))
        #ax.set_ylim(-1, 20)

        ax.set_xlabel(r'Wavelength $(\AA)$')
        ax.set_ylabel(r'SB $( erg\, s^{-1}\, cm^{-2} \,\AA^{-1} arcsec^{-2})$', fontsize = 'large')

        ax.set_title(fr'OII RL line fit')
        ax.legend()
        plt.tight_layout()

        #Save the figure
        #plt.savefig(f'/Volumes/amrita/'+plotout+'_apr9_central_spaxel_20501.png', dpi =100)
        plt.savefig(f'/Volumes/amrita/spectra_products/linefits/'+plotout+'_indidual_spaxels_3501.png', dpi =100)


    #return  popt, pcov, mean_flux6, std_flux6
    return popt, pcov #mean_flux1, std_flux1, mean_flux2, std_flux2, mean_flux3, std_flux3, mean_flux4, std_flux4, \
    #mean_flux5, std_flux5, mean_flux6, std_flux6, mean_flux7, std_flux7, mean_flux8, std_flux8, mean_flux9, std_flux9, mean_flux10, std_flux10 

In [16]:

########################### Testing Sframe emission line fittings ########################################

# Define your line fitting dictionary and other variables as before
line_fitting = {
    
            'HI3750': [3750.15, [3747.5, 3753.5], [3738, 3743], [3758, 3765]], # H_kappa           
            'HI3771': [3770.63, [3768.5, 3773.5], [3758, 3764], [3776, 3786]], # H_iota
            'HI3835': [3835.38, [3833, 3839], [3824, 3832], [3842, 3852]], # H_eta
            'HI4102': [4101.74, [4100, 4105], [4085, 4095], [4110, 4120]], # H_delta
            'HI8863': [8862.78, [8860,8865.5], [8852,8859], [8870,8880]], # P_11        
            'HI9015': [9014.91, [9011.5, 9019], [9005, 9010], [9022, 9032]], # P_10
            #'HI9229': [9229.01, [9226.5, 9232.5], [9212, 9225], [9240, 9250]], # P_9
            'Ha6563': [6562.8, [6560, 6566], [6535, 6545], [6590, 6605]],
            'Hb4861': [4861.32, [4858, 4866], [4820, 4850], [4870, 4900]],
            'Hgm4340': [4340.463, [4338, 4344], [4310, 4335], [4345, 4355]],

            #'NII_OII' : [4607.16, [4606.4, 4678], [4600, 4605], [4680, 4694]],
#
            #'[SII]4069': [4068.6, [4067, 4080], [4055, 4065],[4083, 4098]],  
            '[SII]6717': [6716.44, [6714, 6720], [6680, 6700], [6735, 6750]],
            '[SII]6731': [6730.82, [6728, 6733], [6684, 6702], [6735, 6750]],
##
            #
            '[NII]5755': [5754.64, [5752, 5758], [5720, 5745], [5760, 5775]],
            '[NII]6548': [6548.04, [6546, 6551], [6525, 6540], [6590, 6605]],  
            '[NII]6584': [6583.46, [6581, 6586], [6535, 6545], [6590, 6605]],
##
            '[OIII]4363': [4363.2,[4359, 4367], [4346, 4355],  [4370, 4385]],
            '[OIII]4959': [4958.91,[4955, 4962], [4830, 4850],  [5020, 5035]],
            '[OIII]5007': [5006.84,[5004, 5010], [4830, 4850],  [5020, 5035]],
##
            #
            #'[SIII]6312': [6312.1, [6310, 6315], [6260, 6290], [6320, 6330]],
            #'[SIII]9069': [9068.6, [9066, 9072], [9025, 9035], [9080, 9100]],
            #'[SIII]9531': [9530.6, [9528, 9535], [9480, 9510], [9555, 9580]],    
##
            #'[OII]3726': [3726.03, [3723, 3734], [3650, 3675], [3755, 3765]],
            #'[OII]7320': [7319.99, [7317, 7324], [7300, 7315], [7345, 7360]],
            #'[OII]7330': [7330, [7327, 7335], [7300, 7315], [7345, 7360]]

            }

linename = []
lines = []
line_mask_arr = []
left_mask_arr = []
right_mask_arr = []

# Iterating over the above line_fitting keys and values to append values to lists
for key, values in zip(line_fitting.keys(), line_fitting.values()):
    linename.append(key)
    lines.append(values[0])
    line_mask_arr.append(values[1]) 
    left_mask_arr.append(values[2]) 
    right_mask_arr.append(values[3]) 

linefitdict = {'fiberid': [], 'RA': [], 'Dec': []}  # Dictionary to store data for each line in each fiber from each tile

num = np.linspace(1, 10, 10, dtype = int)

for i, j in enumerate(linename):
    

    if j  == 'NII_OII':

        for k in num:  # Assuming up to 10 components 
            linefitdict[f'{j}_flux{k-1}'] = []
            linefitdict[f'{j}_flux{k-1}_err'] = []
        linefitdict[f'{j}_wave0'] = []
        linefitdict[f'{j}_wave0_err'] = []
        linefitdict[f'{j}_sigma0'] = []
        linefitdict[f'{j}_cont0'] = []
        linefitdict[f'{j}_cont0_err'] = []
        linefitdict[f'{j}_cont1'] = []
        linefitdict[f'{j}_cont1_err'] = []
        linefitdict[f'{j}_cont2'] = []
        linefitdict[f'{j}_cont2_err'] = []

    else:
        linefitdict[f'{j}_flux0'] = []
        linefitdict[f'{j}_flux0_err'] = []
        linefitdict[f'{j}_wave0'] = []
        linefitdict[f'{j}_wave0_err'] = []
        linefitdict[f'{j}_sigma0'] = []
        linefitdict[f'{j}_cont'] = []
        linefitdict[f'{j}_cont_err'] = []


# Reading the fits file    

with fits.open('/Users/amritasingh/amrita/LVM/LVM_M16_output/lvmSFrame-00038276_w_res_spectra.fits') as hdu:

    header = hdu[0].header
    data = hdu[1].data
    table = hdu['SLITMAP'].data
    mask = hdu['MASK'].data
    wave = hdu['WAVE'].data
    flux1 = hdu['RES_SPECTRA'].data/978.18
    ivar = hdu['IVAR'].data
    error = np.sqrt(1/ivar)/978.18

    # Making a selection on science and good quality fibers
    selsci = table['targettype'] == 'science'
    selfib = table['fibstatus'] == 0
    sel = selsci * selfib


    error1 = error[sel]
    fiberid = table['fiberid'][sel]

    
    RA = table['ra'][sel]
    Dec = table['dec'][sel]

    spaxel_wave = wave
    print(spaxel_wave.shape)

    # Reading flux and error in each fiber
    for i, j in enumerate(fiberid):

        spaxel_flux = (flux1[i]* 1e17)
        spaxel_error =(error1[i]* 1e17)
        RA_fib = RA[i]
        Dec_fib = Dec[i]
        spaxel_fiberid = fiberid[i]

        # Store the fiber number and tile id for each iteration
        linefitdict['RA'].append(RA_fib)
        linefitdict['Dec'].append(Dec_fib)
        linefitdict['fiberid'].append(spaxel_fiberid)
        
        # Iterating over each line and masks to select emission line ranges with their underlying left and right continuum masks
        for lineid, line_data in line_fitting.items():
            
            # Extracting data for line fitting
            line, line_mask, left_mask, right_mask = line_data
        

            if lineid == 'NII_OII':

                total_mask = mklinemask(spaxel_wave, line_mask, left_mask, right_mask, line)

                #print(f"fiber no. {fibid_fib}, tile id: {tileid_fib}")

                popt, pcov = fit_ten_gauss_quad(spaxel_wave[total_mask], spaxel_flux[total_mask], spaxel_error[total_mask], line, plot = False, plotout=f'individual_fib_{lineid}')

                for i in range(16):

                    if popt[i]<0:
                        popt[i] = np.nan
       
                #print(f"fiber no. {fibid_fib}, tile id: {tileid_fib}, popt shape: {popt.shape}, popt 10: {popt[10]}")
                
                # Store data for 1st component
                linefitdict[lineid+'_flux0'].append(popt[0]*1e-17)
                linefitdict[lineid+'_flux0_err'].append(np.sqrt(pcov[0,0])*1e-17)
                linefitdict[lineid+'_wave0'].append(popt[1])
                linefitdict[lineid+'_wave0_err'].append(np.sqrt(pcov[1,1]))
                linefitdict[lineid+'_sigma0'].append(popt[2])
                # Store data for 2nd component
                linefitdict[lineid+'_flux1'].append(popt[4]*1e-17)
                linefitdict[lineid+'_flux1_err'].append(np.sqrt(pcov[4, 4])*1e-17)
                # Store data for 3rd component
                linefitdict[lineid+'_flux2'].append(popt[5]*1e-17)
                linefitdict[lineid+'_flux2_err'].append(np.sqrt(pcov[5, 5])*1e-17)
                # Store data for 4th component
                linefitdict[lineid+'_flux3'].append(popt[6]*1e-17)
                linefitdict[lineid+'_flux3_err'].append(np.sqrt(pcov[6, 6])*1e-17)
                # Store data for 5th component
                linefitdict[lineid+'_flux4'].append(popt[7]*1e-17)
                linefitdict[lineid+'_flux4_err'].append(np.sqrt(pcov[7, 7])*1e-17)
                # Store data for 6th component
                linefitdict[lineid+'_flux5'].append(popt[8]*1e-17)
                linefitdict[lineid+'_flux5_err'].append(np.sqrt(pcov[8, 8])*1e-17)
                # Store data for 7th component
                linefitdict[lineid+'_flux6'].append(popt[9]*1e-17)
                linefitdict[lineid+'_flux6_err'].append(np.sqrt(pcov[9, 9])*1e-17)
                # Store data for 8th component
                linefitdict[lineid+'_flux7'].append(popt[10]*1e-17)
                linefitdict[lineid+'_flux7_err'].append(np.sqrt(pcov[10, 10])*1e-17)
                # Store data for 9th component
                linefitdict[lineid+'_flux8'].append(popt[11]*1e-17)
                linefitdict[lineid+'_flux8_err'].append(np.sqrt(pcov[11, 11])*1e-17)
                # Store data for 10th component
                linefitdict[lineid+'_flux9'].append(popt[12]*1e-17)
                linefitdict[lineid+'_flux9_err'].append(np.sqrt(pcov[12, 12])*1e-17)
                # Store continuum data
                linefitdict[lineid+'_cont0'].append(popt[13]*1e-17)
                linefitdict[lineid+'_cont0_err'].append(np.sqrt(pcov[13, 13])*1e-17)
                linefitdict[lineid+'_cont1'].append(popt[14]*1e-17)
                linefitdict[lineid+'_cont1_err'].append(np.sqrt(pcov[14,14])*1e-17)
                linefitdict[lineid+'_cont2'].append(popt[15]*1e-17)
                linefitdict[lineid+'_cont2_err'].append(np.sqrt(pcov[15, 15])*1e-17)

            else:

                total_mask = mklinemask(wave, line_mask, left_mask, right_mask, line)

                # Single Gaussian fitting
                popt, pcov = fit_gauss(spaxel_wave[total_mask], spaxel_flux[total_mask], spaxel_error[total_mask], line, plot = False, plotout=f'individual_fib_{lineid}_{j}')
                print(f"line: {lineid} fiber no. {spaxel_fiberid}, flux: {popt[0]:.2f}e-17, pcov: {np.sqrt(pcov[0,0]):.3f}e-17")

                # Store fitting parameters
                linefitdict[lineid+'_flux0'].append(popt[0]*1e-17)
                linefitdict[lineid+'_flux0_err'].append(np.sqrt(pcov[0, 0])*1e-17)
                linefitdict[lineid+'_wave0'].append(popt[1])
                linefitdict[lineid+'_wave0_err'].append(np.sqrt(pcov[1, 1]))
                linefitdict[lineid+'_sigma0'].append(popt[2])
                linefitdict[lineid+'_cont'].append(popt[3]*1e-17)
                linefitdict[lineid+'_cont_err'].append(np.sqrt(pcov[3, 3])*1e-17)

           
# Ensuring all lists have the same length by filling missing values with zeros
max_length = max(len(lst) for lst in linefitdict.values())
for key in linefitdict:
    if len(linefitdict[key]) < max_length:
        linefitdict[key].extend([0] * (max_length - len(linefitdict[key])))
 
linefitdict = {key: np.array(value) for key, value in linefitdict.items()}

# Create a table
table = Table(linefitdict)

# converting table to a FITS file
table.write('/Users/amritasingh/amrita/LVM/LVM_M16_output/M16_obs_flux.fits', overwrite=True)


(12401,)
line: HI3750 fiber no. 3, flux: 1.81e-17, pcov: 0.921e-17
line: HI3771 fiber no. 3, flux: 2.45e-17, pcov: 0.650e-17
line: HI3835 fiber no. 3, flux: 4.56e-17, pcov: 0.769e-17
line: HI4102 fiber no. 3, flux: 15.43e-17, pcov: 0.537e-17
line: HI8863 fiber no. 3, flux: 3.34e-17, pcov: 0.169e-17
line: HI9015 fiber no. 3, flux: 5.40e-17, pcov: 0.188e-17
line: Ha6563 fiber no. 3, flux: 453.64e-17, pcov: 1.510e-17
line: Hb4861 fiber no. 3, flux: 81.89e-17, pcov: 0.850e-17
line: Hgm4340 fiber no. 3, flux: 28.44e-17, pcov: 0.583e-17
line: [SII]6717 fiber no. 3, flux: 27.29e-17, pcov: 0.372e-17
line: [SII]6731 fiber no. 3, flux: 19.54e-17, pcov: 0.325e-17
line: [NII]5755 fiber no. 3, flux: 1.28e-17, pcov: 0.301e-17
line: [NII]6548 fiber no. 3, flux: 41.04e-17, pcov: 0.451e-17
line: [NII]6584 fiber no. 3, flux: 128.30e-17, pcov: 0.762e-17
line: [OIII]4363 fiber no. 3, flux: 0.29e-17, pcov: 0.226e-17
line: [OIII]4959 fiber no. 3, flux: 22.83e-17, pcov: 0.460e-17
line: [OIII]5007 fiber no. 3

In [ ]:
with fits.open('/Users/amritasingh/LVM/W28_SNR_for_knox/w28_lvmSFrame-00003470_flux_table.fits') as hdul:
    data = hdul[1].data
    header = hdul[1].header
    print(repr(header))

XTENSION= 'BINTABLE'           / binary table extension                         
BITPIX  =                    8 / array data type                                
NAXIS   =                    2 / number of array dimensions                     
NAXIS1  =                  728 / length of dimension 1                          
NAXIS2  =                 1754 / length of dimension 2                          
PCOUNT  =                    0 / number of group parameters                     
GCOUNT  =                    1 / number of groups                               
TFIELDS =                   91 / number of table fields                         
TTYPE1  = 'fiberid '                                                            
TFORM1  = 'K       '                                                            
TTYPE2  = 'RA      '                                                            
TFORM2  = 'D       '                                                            
TTYPE3  = 'Dec     '        

In [ ]:

# This v2 version has all header keywords removed except for wave, flux, error, ivar and mask 
with fits.open('/Volumes/amrita/linefitplots/lagoon_combined_sframe10_flux_table_wo_mjd_60236.fits') as hdu:
    data = hdu[1].data
    header = hdu[1].header

print(repr(header), data.shape, data['Hb4861_flux0'])

XTENSION= 'BINTABLE'           / binary table extension                         
BITPIX  =                    8 / array data type                                
NAXIS   =                    2 / number of array dimensions                     
NAXIS1  =                  792 / length of dimension 1                          
NAXIS2  =                 1754 / length of dimension 2                          
PCOUNT  =                    0 / number of group parameters                     
GCOUNT  =                    1 / number of groups                               
TFIELDS =                   99 / number of table fields                         
TTYPE1  = 'fiberid '                                                            
TFORM1  = 'K       '                                                            
TTYPE2  = 'RA      '                                                            
TFORM2  = 'D       '                                                            
TTYPE3  = 'Dec     '        

In [ ]:
# with all 4 exps of tile 10 (1754,) [2.05401845e-16 2.30695981e-16 2.84976591e-16 ... 2.23380173e-14 1.86676381e-14 1.52226594e-14]
# with only 2 exps of tile 2 (1754,) [2.44350306e-16 2.51609757e-16 3.30366798e-16 ... 2.68533748e-14 1.84356196e-14 1.53279279e-14]

In [ ]:
'''
# Define your line fitting dictionary and other variables as before
line_fitting = {
    
            'NII4621' : [4621.35, [4619.8, 4645], [4610, 4615], [4693, 4700]],
            '[SII]4069': [4068.6, [4067, 4080], [4055, 4063], [4087, 4097]], 
            '[SII]6717': [6716.44, [6714, 6720], [6690, 6710], [6735, 6750]],
            '[SII]6731': [6730.82, [6728, 6734], [6684, 6702], [6740, 6760]],
#
            '[NII]5755': [5754.64, [5752, 5758], [5730, 5745], [5760, 5770]],
            '[NII]6548': [6548.04, [6546, 6551], [6525, 6540], [6590, 6605]],  
            '[NII]6584': [6583.46, [6581, 6586], [6535, 6545], [6590, 6605]],
#
            '[OIII]4363': [4363.2,[4359, 4366.5], [4347, 4353],  [4372, 4381]],
            '[OIII]4959': [4958.91,[4955, 4962], [4830, 4850],  [5020, 5035]],
            '[OIII]5007': [5006.84,[5004, 5010], [4830, 4850],  [5020, 5035]],
#
            #'[SIII]6312': [6312.1, [6310, 6315], [6260, 6290], [6320, 6330]],
            #'[SIII]9069': [9068.6, [9066, 9072], [9025, 9035], [9080, 9100]],
            #'[SIII]9531': [9530.6, [9528, 9535], [9480, 9510], [9555, 9580]],  
#
            '[OII]3726': [3726.03, [3724, 3734], [3650, 3675], [3755, 3765]],
            '[OII]7320': [7319.99, [7317, 7324], [7300, 7315], [7345, 7360]],
            '[OII]7330': [7330, [7327, 7335], [7300, 7315], [7345, 7360]],
#
            '[ClIII]5517':[5517.71, [5515, 5520], [5490, 5510], [5522, 5535]],
            '[ClIII]5537':[5537.88, [5536, 5541], [5522, 5533], [5546, 5570]],
#
            'Ha6563': [6562.8, [6560, 6566], [6535, 6545], [6590, 6605]],
            'Hb4861': [4861.32, [4858, 4866], [4820, 4850], [4870, 4900]],
            'Hgm4340': [4340.463, [4338, 4344], [4310, 4335], [4345, 4355]],

            'OII4638': [4638.86, [4636.5, 4679], [4610, 4620], [4684, 4694]]
}

linename = []
lines = []
line_mask_arr = []
left_mask_arr = []
right_mask_arr = []

# Iterating over the above line_fitting keys and values to append values to lists
for key, values in zip(line_fitting.keys(), line_fitting.values()):
    linename.append(key)
    lines.append(values[0])
    line_mask_arr.append(values[1]) 
    left_mask_arr.append(values[2]) 
    right_mask_arr.append(values[3]) 

linefitdict = {'fiberid': [], 'tileid': [], 'RA': [], 'Dec': []}  # Dictionary to store data for each line in each fiber from each tile

# Initialize the dictionary to store fitting results
num = np.linspace(1, 7, 7, dtype = int)

for i, j in enumerate(linename):
    for k in num:  # Assuming up to 7 components 
        linefitdict[f'{j}_flux{k-1}'] = []
        linefitdict[f'{j}_flux{k-1}_err'] = []
        linefitdict[f'{j}_wave{k-1}'] = []
        linefitdict[f'{j}_wave{k-1}_err'] = []
        linefitdict[f'{j}_sigma{k-1}'] = []
        linefitdict[f'{j}_sigma{k-1}_err'] = []
    linefitdict[f'{j}_cont'] = []
    linefitdict[f'{j}_cont_err'] = []

    
# Reading fits file
with fits.open('/Volumes/amrita/LVM/lagoon_outputs/condition2_cont.fits') as hdu:  # reading the fits file

    data = hdu[1].data

    wavelength = data['wave']
    flux = data['flux']* 1e17
    error = data['error']* 1e17

    fiberid = data['fiberid']
    tileid = data['tileid']
    RA = data['RA']
    Dec = data['Dec']

    # Reading flux and error in each fiber
    for i, j in enumerate(fiberid):

        int_flux = flux[i]/978.18
        int_error = error[i]/978.18
        RA_fib = RA[i]
        Dec_fib = Dec[i]
        spaxel_tileid = tileid[i]
        spaxel_fiberid = fiberid[i]
        wave = wavelength[i]
        
        # Store the fiber number and tile id for each iteration
        
        linefitdict['RA'].append(RA_fib)
        linefitdict['Dec'].append(Dec_fib)
        linefitdict['fiberid'].append(spaxel_fiberid)
        linefitdict['tileid'].append(spaxel_tileid)
        
        # Iterating over each line and masks to select emission line ranges with their underlying left and right continuum masks
        for lineid, line_data in line_fitting.items():
            
            # Extracting data for line fitting
            line, line_mask, left_mask, right_mask = line_data

            if lineid == '[OII]3726':

                total_mask = mklinemask(wave, line_mask, left_mask, right_mask, line)
                #Fit 2 gaussians to [OII] doublets 3726 and 3729 \lambda (3726 is written as line+2.8)

                popt, pcov = fit_double_gauss(wave[total_mask], int_flux[total_mask], int_error[total_mask], line, line+2.8, plot = False, plotout=f'{lineid}_{j}_tile{spaxel_tileid}')

                # Store data for 1st component
                linefitdict[lineid+'_flux0'].append(popt[0]*1e-17)                 
                linefitdict[lineid+'_flux0_err'].append(np.sqrt(pcov[0,0])*1e-17)
                linefitdict[lineid+'_wave0'].append(popt[1])
                linefitdict[lineid+'_wave0_err'].append(np.sqrt(pcov[1,1]))
                linefitdict[lineid+'_sigma0'].append(popt[2])
                linefitdict[lineid+'_sigma0_err'].append(np.sqrt(pcov[2,2]))


                # Store data for 2nd component
                linefitdict[lineid+'_flux1'].append(popt[3]*1e-17)
                linefitdict[lineid+'_flux1_err'].append(np.sqrt(pcov[3,3])*1e-17)
                linefitdict[lineid+'_wave1'].append(popt[4])
                linefitdict[lineid+'_wave1_err'].append(np.sqrt(pcov[4,4]))
                linefitdict[lineid+'_sigma1'].append(popt[5])
                linefitdict[lineid+'_sigma1_err'].append(np.sqrt(pcov[5,5]))


                # Store continuum data
                linefitdict[lineid+'_cont'].append(popt[6]*1e-17)
                linefitdict[lineid+'_cont_err'].append(np.sqrt(pcov[6,6]*1e-17))

                num = np.linspace(2, 6, 4, dtype =int)
                for i in num:
                    i = str(i)

                    linefitdict[lineid+'_flux'+i].append(0)
                    linefitdict[lineid+'_flux'+i+'_err'].append(0)
                    linefitdict[lineid+'_wave'+i].append(0)
                    linefitdict[lineid+'_wave'+i+'_err'].append(0)
                    linefitdict[lineid+'_sigma'+i].append(0)

            elif lineid == '[OIII]4363':

                total_mask = mklinemask(wave, line_mask, left_mask, right_mask, line)

                #Fit 2 gaussians to [FeII]4360 and [OIII]4363 \lambda lines (4360 \lambda is written as line-3.2)
                popt, pcov = fit_double_gauss(wave[total_mask], int_flux[total_mask], int_error[total_mask], line-3.2, line, plot = False, plotout=f'{lineid:2}_{j}_tile{spaxel_tileid}')

                # Store data for 1st component       
                linefitdict[lineid+'_flux0'].append(popt[0]*1e-17)
                linefitdict[lineid+'_flux0_err'].append(np.sqrt(pcov[0,0])*1e-17)
                linefitdict[lineid+'_wave0'].append(popt[1])
                linefitdict[lineid+'_wave0_err'].append(np.sqrt(pcov[1,1]))
                linefitdict[lineid+'_sigma0'].append(popt[2])
                linefitdict[lineid+'_sigma0_err'].append(np.sqrt(pcov[2,2]))

                # Store data for 2nd component
                linefitdict[lineid+'_flux1'].append(popt[3]*1e-17)
                linefitdict[lineid+'_flux1_err'].append(np.sqrt(pcov[3,3])*1e-17)
                linefitdict[lineid+'_wave1'].append(popt[4])
                linefitdict[lineid+'_wave1_err'].append(np.sqrt(pcov[4,4]))
                linefitdict[lineid+'_sigma1'].append(popt[5])
                linefitdict[lineid+'_sigma1_err'].append(np.sqrt(pcov[5,5]))

                # Store continuum data
                linefitdict[lineid+'_cont'].append(popt[6]*1e-17)
                linefitdict[lineid+'_cont_err'].append(np.sqrt(pcov[6,6])*1e-17)

                num = np.linspace(2, 6, 4, dtype =int)
                for i in num:
                    i = str(i)

                    linefitdict[lineid+'_flux'+i].append(0)
                    linefitdict[lineid+'_flux'+i+'_err'].append(0)
                    linefitdict[lineid+'_wave'+i].append(0)
                    linefitdict[lineid+'_wave'+i+'_err'].append(0)
                    linefitdict[lineid+'_sigma'+i].append(0)

            elif lineid == 'OII4638':

                total_mask = mklinemask(wave, line_mask, left_mask, right_mask, line)

                popt, pcov = fit_siete_gauss(wave[total_mask], int_flux[total_mask], int_error[total_mask], line, plot = True, plotout=f'{lineid:2}_{j}_tile{spaxel_tileid}')

                #print(len(popt), popt[3])

                # Store data for 1st component
                linefitdict[lineid+'_flux0'].append(popt[0]*1e-17)
                linefitdict[lineid+'_flux0_err'].append(np.sqrt(pcov[0,0])*1e-17)
                linefitdict[lineid+'_wave0'].append(popt[1])
                linefitdict[lineid+'_wave0_err'].append(np.sqrt(pcov[1,1]))
                linefitdict[lineid+'_sigma0'].append(popt[2])
                linefitdict[lineid+'_sigma0_err'].append(np.sqrt(pcov[2,2]))

                # popt[3] is velocity shift, not storing it

                # Store data for 2nd component
                linefitdict[lineid+'_flux1'].append(popt[4]*1e-17)
                linefitdict[lineid+'_flux1_err'].append(np.sqrt(pcov[4, 4])*1e-17)
                linefitdict[lineid+'_wave1'].append(0)
                linefitdict[lineid+'_wave1_err'].append(0)
                linefitdict[lineid+'_sigma1'].append(0)
                linefitdict[lineid+'_sigma1_err'].append(0)

                # Store data for 3rd component
                linefitdict[lineid+'_flux2'].append(popt[5]*1e-17)
                linefitdict[lineid+'_flux2_err'].append(np.sqrt(pcov[5, 5])*1e-17)
                linefitdict[lineid+'_wave2'].append(0)
                linefitdict[lineid+'_wave2_err'].append(0)     ############## 
                linefitdict[lineid+'_sigma2'].append(0)
                linefitdict[lineid+'_sigma2_err'].append(0)

                # Store data for 4th component
                linefitdict[lineid+'_flux3'].append(popt[6]*1e-17)
                linefitdict[lineid+'_flux3_err'].append(np.sqrt(pcov[6, 6])*1e-17)
                linefitdict[lineid+'_wave3'].append(0)
                linefitdict[lineid+'_wave3_err'].append(np.sqrt(0))     ############## 
                linefitdict[lineid+'_sigma3'].append(0)
                linefitdict[lineid+'_sigma3_err'].append(0)

                # Store data for 5th component
                linefitdict[lineid+'_flux4'].append(popt[7]*1e-17)
                linefitdict[lineid+'_flux4_err'].append(np.sqrt(pcov[7, 7])*1e-17)
                linefitdict[lineid+'_wave4'].append(0)
                linefitdict[lineid+'_wave4_err'].append(0)     ############## 
                linefitdict[lineid+'_sigma4'].append(0)
                linefitdict[lineid+'_sigma4_err'].append(0)

                # Store data for 6th component
                linefitdict[lineid+'_flux5'].append(popt[8]*1e-17)
                linefitdict[lineid+'_flux5_err'].append(np.sqrt(pcov[8, 8])*1e-17)
                linefitdict[lineid+'_wave5'].append(0)
                linefitdict[lineid+'_wave5_err'].append(0)     ############## 
                linefitdict[lineid+'_sigma5'].append(0)
                linefitdict[lineid+'_sigma5_err'].append(0)

                # Store data for 6th component
                linefitdict[lineid+'_flux6'].append(popt[9]*1e-17)
                linefitdict[lineid+'_flux6_err'].append(np.sqrt(pcov[9, 9])*1e-17)
                linefitdict[lineid+'_wave6'].append(0)
                linefitdict[lineid+'_wave6_err'].append(0)     ############## 
                linefitdict[lineid+'_sigma6'].append(0)
                linefitdict[lineid+'_sigma6_err'].append(0)

                # Store continuum data
                linefitdict[lineid+'_cont'].append(popt[10]*1e-17)
                linefitdict[lineid+'_cont_err'].append(np.sqrt(pcov[10, 10])*1e-17)


            elif lineid == '[SII]4069':
                
                total_mask = mklinemask(wave, line_mask, left_mask, right_mask, line)

                #Fit 3 Gaussians to 4069, 4073 and 4076 \lambda lines, the rest frame lambda of other two lines are defined as line+c (c is a const)
                popt, pcov = fit_triple_gauss(wave[total_mask], int_flux[total_mask], int_error[total_mask], line, line+5, line+7.5, plot = False, plotout=f'{lineid}_{j}_tile{spaxel_tileid}')

                # Store data for 1st component
                linefitdict[lineid+'_flux0'].append(popt[0]* 1e-17)
                linefitdict[lineid+'_flux0_err'].append(np.sqrt(pcov[0,0])* 1e-17)
                linefitdict[lineid+'_wave0'].append(popt[1])
                linefitdict[lineid+'_wave0_err'].append(np.sqrt(pcov[1,1]))
                linefitdict[lineid+'_sigma0'].append(popt[2])
                linefitdict[lineid+'_sigma0_err'].append(np.sqrt(pcov[2,2]))

                # Store data for 2nd component
                linefitdict[lineid+'_flux1'].append(popt[3]* 1e-17)
                linefitdict[lineid+'_flux1_err'].append(np.sqrt(pcov[3,3])* 1e-17)
                linefitdict[lineid+'_wave1'].append(popt[4])
                linefitdict[lineid+'_wave1_err'].append(np.sqrt(pcov[4,4]))
                linefitdict[lineid+'_sigma1'].append(popt[5])
                linefitdict[lineid+'_sigma1_err'].append(np.sqrt(pcov[5,5]))

                # Store data for 3rd component
                linefitdict[lineid+'_flux2'].append(popt[6]* 1e-17)
                linefitdict[lineid+'_flux2_err'].append(np.sqrt(pcov[6, 6])* 1e-17)
                linefitdict[lineid+'_wave2'].append(popt[7])
                linefitdict[lineid+'_wave2_err'].append(np.sqrt(pcov[7,7]))     ############## 
                linefitdict[lineid+'_sigma2'].append(popt[8])
                linefitdict[lineid+'_sigma2_err'].append(np.sqrt(pcov[8,8]))

                # Store continuum data
                linefitdict[lineid+'_cont'].append(popt[9]* 1e-17)
                linefitdict[lineid+'_cont_err'].append(np.sqrt(pcov[9,9])* 1e-17)


                num = np.linspace(3, 6, 3, dtype =int)
                for i in num:
                    i = str(i)

                    linefitdict[lineid+'_flux'+i].append(0)
                    linefitdict[lineid+'_flux'+i+'_err'].append(0)
                    linefitdict[lineid+'_wave'+i].append(0)
                    linefitdict[lineid+'_wave'+i+'_err'].append(0)
                    linefitdict[lineid+'_sigma'+i].append(0)

            
            else:

                total_mask = mklinemask(wave, line_mask, left_mask, right_mask, line)

                # Single Gaussian fitting
                popt, pcov = fit_gauss(wave[total_mask], int_flux[total_mask], int_error[total_mask], line, plot = False, plotout=f'{lineid:2}_{j}_tile{spaxel_tileid}')

                # Store fitting parameters
                linefitdict[lineid+'_flux0'].append(popt[0]*1e-17)
                linefitdict[lineid+'_flux0_err'].append(np.sqrt(pcov[0, 0])*1e-17)
                linefitdict[lineid+'_wave0'].append(popt[1])
                linefitdict[lineid+'_wave0_err'].append(np.sqrt(pcov[1, 1]))
                linefitdict[lineid+'_sigma0'].append(popt[2])
                linefitdict[lineid+'_sigma0_err'].append(np.sqrt(pcov[2, 2]))

                linefitdict[lineid+'_cont'].append(popt[3]*1e-17)
                linefitdict[lineid+'_cont_err'].append(np.sqrt(pcov[3, 3])*1e-17)

                # Other componets store 0 for single gaussian fit

                num = np.linspace(1, 6, 6, dtype =int)
                for i in num:
                    i = str(i)

                    linefitdict[lineid+'_flux'+i].append(0)
                    linefitdict[lineid+'_flux'+i+'_err'].append(0)
                    linefitdict[lineid+'_wave'+i].append(0)
                    linefitdict[lineid+'_wave'+i+'_err'].append(0)
                    linefitdict[lineid+'_sigma'+i].append(0)
           

# Ensuring all lists have the same length by filling missing values with zeros
#max_length = max(len(lst) for lst in linefitdict.values())
#for key in linefitdict:
#    if len(linefitdict[key]) < max_length:
#        linefitdict[key].extend([0] * (max_length - len(linefitdict[key])))
# 
#linefitdict = {key: np.array(value) for key, value in linefitdict.items()}
#
## Create an Astropy Table
#table = Table(linefitdict)
#
## Write the table to a FITS file
#table.write('/Users/amritasingh/LVM_lagoon_outputs/lagoon_obs_table.fits', overwrite=True)
                    
'''

"\n# Define your line fitting dictionary and other variables as before\nline_fitting = {\n    \n            'NII4621' : [4621.35, [4619.8, 4645], [4610, 4615], [4693, 4700]],\n            '[SII]4069': [4068.6, [4067, 4080], [4055, 4063], [4087, 4097]], \n            '[SII]6717': [6716.44, [6714, 6720], [6690, 6710], [6735, 6750]],\n            '[SII]6731': [6730.82, [6728, 6734], [6684, 6702], [6740, 6760]],\n#\n            '[NII]5755': [5754.64, [5752, 5758], [5730, 5745], [5760, 5770]],\n            '[NII]6548': [6548.04, [6546, 6551], [6525, 6540], [6590, 6605]],  \n            '[NII]6584': [6583.46, [6581, 6586], [6535, 6545], [6590, 6605]],\n#\n            '[OIII]4363': [4363.2,[4359, 4366.5], [4347, 4353],  [4372, 4381]],\n            '[OIII]4959': [4958.91,[4955, 4962], [4830, 4850],  [5020, 5035]],\n            '[OIII]5007': [5006.84,[5004, 5010], [4830, 4850],  [5020, 5035]],\n#\n            #'[SIII]6312': [6312.1, [6310, 6315], [6260, 6290], [6320, 6330]],\n            #'[SII

In [ ]:
####################### Dust correction ###################################

In [ ]:
def pyneb_ratio(Te, ne):

    atom = pn.RecAtom('H', 1)

    wavelengths = [3750.15, 3835.38, 4101.74, 4340.463, 4861.32, 9229.01] #8862.78, 9014.91, 

    # Initialize arrays for emissivities and ratios
    emis = np.zeros_like(wavelengths)  
    ratios = np.zeros_like(wavelengths)

    # Calculate emissivity for each wavelength and store it in the emis list
    for k, wl in enumerate(wavelengths):
        emis[k] = atom.getEmissivity(tem=Te, den=ne, wave=wl)

    # Calculate the ratio of each emissivity w.r.t. the first one (emis[0])
    for n in range(0, len(wavelengths)):  
        ratios[n] = emis[n] /emis[4]
    ratios[4] = 1
    
    return ratios

pyneb_ratio(7650, 600)

# theoretical Balmer decrements (from Osterbrock_2006)
balmer_theoretical = {

    'Hk':   [],
    #'Hi':   [],
    'He':   [],
    'Hd':   [],
    'Hg':   [],
    'Hb':   [],
    #'P11':  [],
    #'P10'':  [],
    'P9' :  []
                }

ratios = pyneb_ratio(7650, 650)

# Assign ratios to the balmer_theoretical dictionary
keys = list(balmer_theoretical.keys())
for i, key in enumerate(keys):
    balmer_theoretical[key] = ratios[i]

print(balmer_theoretical)

# Opening integrated spectrum fits file to read observed fluxes
#with fits.open('/Users/amritasingh/LVM_lagoon_outputs/integrated_spectrum/SB_lagoon_integrated_obs_flux_table.fits') as hdu: # cframes

with fits.open('/Users/amritasingh/LVM_lagoon_outputs/condition2_cont/lagoon_sframe_obs_flux_table_dec11_newdrp.fits') as hdu: #sframes
    #print(repr(hdu[1].header))
    data = hdu[1].data

# Wavelengths of the Balmer lines 
wavelengths = {
    'Hk': data['HI3750_wave0'][0],
    #'Hi': data['HI3771_wave0'][0],
    'He': data['HI3835_wave0'][0],
    'Hd': data['HI4102_wave0'][0], 
    'Hg': data['Hgm4340_wave0'][0], 
    'Hb': data['Hb4861_wave0'][0], 
    #'P11': data['HI8863_wave0'][0],
    #'P10': data['HI9015_wave0'][0],
    'P9' : data['HI9229_wave0'][0] 
                }

# Observed Balmer line fluxes 
observed_ratios = {

    'Hk':  data['HI3750_flux0']/(data['Hb4861_flux0']  ), # basically the line flux is normalized by theoretical value
    #'Hi':  data['HI3771_flux0']/(data['Hb4861_flux0']  ),
    'He':  data['HI3835_flux0']/(data['Hb4861_flux0']  ),
    'Hd':  data['HI4102_flux0']/(data['Hb4861_flux0']  ),
    'Hg':  data['Hgm4340_flux0']/(data['Hb4861_flux0'] ),
    'Hb':  data['Hb4861_flux0']/(data['Hb4861_flux0']  ),
    #'P11': data['HI8863_flux0']/(data['Hb4861_flux0']  ),
    #'P10': data['HI9015_flux0']/(data['Hb4861_flux0']  ),
    'P9' : data['HI9229_flux0']/(data['Hb4861_flux0']  )
                  
                  }

# divide obs_ratios and modeled ratios with the theoretical values at 0 extinction
# calculate err on obs_ratios and plot it using errorbar func, overplot it with modeled obs_ratios
error_obs_ratios = {

    'Hk' : observed_ratios['Hk' ] * np.sqrt((data['HI3750_flux0_err']/data['HI3750_flux0'])**2 + (data['Hb4861_flux0_err']/data['Hb4861_flux0'])**2),
    #'Hi' : observed_ratios['Hi' ] * np.sqrt((data['HI3771_flux0_err']/data['HI3771_flux0'])**2 + (data['Hb4861_flux0_err']/data['Hb4861_flux0'])**2),
    'He' : observed_ratios['He' ] * np.sqrt((data['HI3835_flux0_err']/data['HI3835_flux0'])**2 + (data['Hb4861_flux0_err']/data['Hb4861_flux0'])**2),
    'Hd' : observed_ratios['Hd' ] * np.sqrt((data['HI4102_flux0_err']/data['HI4102_flux0'])**2 + (data['Hb4861_flux0_err']/data['Hb4861_flux0'])**2),
    'Hg' : observed_ratios['Hg' ] * np.sqrt((data['Hgm4340_flux0_err']/data['Hgm4340_flux0'])**2 + (data['Hb4861_flux0_err']/data['Hb4861_flux0'])**2),
    'Hb' : observed_ratios['Hb' ] * np.sqrt(2) * (data['Hb4861_flux0_err'] / data['Hb4861_flux0']),
    #'P11': observed_ratios[#'P11'] * np.sqrt((data['HI8863_flux0_err']/data['HI8863_flux0'])**2 + (data['Hb4861_flux0_err']/data['Hb4861_flux0'])**2),
    #'P10': observed_ratios['P10'] * np.sqrt((data['HI9015_flux0_err']/data['HI9015_flux0'])**2 + (data['Hb4861_flux0_err']/data['Hb4861_flux0'])**2),
    'P9' : observed_ratios['P9' ] * np.sqrt((data['HI9229_flux0_err']/data['HI9229_flux0'])**2 + (data['Hb4861_flux0_err']/data['Hb4861_flux0'])**2)

            }


{'Hk': 0.030326510978472904, 'He': 0.0724182180088562, 'Hd': 0.25642072973776003, 'Hg': 0.46493464670595047, 'Hb': 1.0, 'P9': 0.026078352040148762}


In [ ]:
def dust_corr_all_ratios(filename, lines_to_correct, balmer_theoretical, observed_ratios, error_obs_ratios, wavelengths):
    """
    Correct fluxes using multiple Balmer and Paschen line ratios, saving separate files for each correction.
    """
    with fits.open(filename) as hdul:
        table = hdul[1].data

    extinction_model = F99(Rv=3.1)

    for ratio_label, theo_ratio in balmer_theoretical.items():
        if ratio_label == "Hb":  # Skip Hβ itself
            continue

        obs_ratio = observed_ratios[ratio_label]
        err_obs_ratio = error_obs_ratios[ratio_label]
        wave_ratio = wavelengths[ratio_label] * u.AA

        f_lambda = extinction_model(wave_ratio)
        f_hb = extinction_model(wavelengths['Hb'] * u.AA)

        if np.isclose(f_hb, f_lambda):
            print(f"Skipping ratio {ratio_label} due to zero division risk (f_hb = f_lambda).")
            continue

        try:
            E_B_V = 2.5 * np.log10(obs_ratio / theo_ratio) / (f_hb - f_lambda)
            log_err_term = (1 / np.log(10)) * (err_obs_ratio / obs_ratio)
            E_B_V_err = np.abs((2.5 / (f_hb - f_lambda)) * log_err_term)
        except RuntimeError:
            print(f"Error during E(B-V) calculation for ratio: {ratio_label}")
            continue

        # Prepare new columns
        new_columns = []

        for key in table.columns.names:
            if "flux" in key or "RA" in key or "Dec" in key:
                obs_col_name = f"obs_{key}"
                new_columns.append(fits.Column(name=obs_col_name, format="D", array=table[key]))

        for line_label in lines_to_correct:
            for i in range(10):  # Correcting up to 10 components
                flux_column = f"{line_label}_flux{i}"
                flux_err_column = f"{line_label}_flux{i}_err"
                wave_column = f"{line_label}_wave{i}"

                if flux_column in table.columns.names:
                    line_wavelength = (
                        table[wave_column][0] if wave_column in table.columns.names else wavelengths[line_label]
                    )
                    if line_wavelength == 0:
                        int_flux_col_name = f"int_{flux_column}"
                        int_flux_err_col_name = f"int_{flux_err_column}"
                        new_columns.append(fits.Column(name=int_flux_col_name, format="D", array=table[flux_column]))
                        new_columns.append(fits.Column(name=int_flux_err_col_name, format="D", array=table[flux_err_column]))
                    else:
                        f_line = extinction_model(line_wavelength * u.AA)
                        A_line = E_B_V * f_line
                        A_line_err = E_B_V_err * f_line

                        observed_flux = table[flux_column]
                        observed_flux_err = table[flux_err_column]
                        intrinsic_flux = observed_flux * 10 ** (0.4 * A_line)
                        intrinsic_flux_err = intrinsic_flux * np.sqrt(
                            (observed_flux_err / observed_flux) ** 2
                            + (0.4 * A_line_err * np.log(10)) ** 2
                        )

                        int_flux_col_name = f"int_{flux_column}"
                        int_flux_err_col_name = f"int_{flux_err_column}"
                        new_columns.append(fits.Column(name=int_flux_col_name, format="D", array=intrinsic_flux))
                        new_columns.append(fits.Column(name=int_flux_err_col_name, format="D", array=intrinsic_flux_err))

        # Add E(B-V) columns
        new_columns.append(fits.Column(name="E_B_V", format="D", array=E_B_V * np.ones(len(table))))
        new_columns.append(fits.Column(name="E_B_V_err", format="D", array=E_B_V_err * np.ones(len(table))))

        print('mean , median:', np.nanmean(E_B_V), np.nanmedian(E_B_V), 'err:', np.nanmean(E_B_V_err), np.nanmedian(E_B_V_err))

        # Write to a new FITS file
        cols = fits.ColDefs(new_columns)
        hdu = fits.BinTableHDU.from_columns(cols)
        output_filename = f"/Users/amritasingh/LVM_lagoon_outputs/condition2_cont/lagoon_sframe_obs_corr_flux_table_dec11_newdrp_{ratio_label.replace(' ', '_')}.fits"
        hdu.writeto(output_filename, overwrite=True)
        print(f"File saved: {output_filename}")

# Input
filename = "/Users/amritasingh/LVM_lagoon_outputs/condition2_cont/lagoon_sframe_obs_flux_table_dec11_newdrp.fits"

lines_to_correct = {

    'NII_OII' : [4607.16],
    #'[SII]4069': [4068.6], 
    '[SII]6717': [6716.44],
    '[SII]6731': [6730.82],

    #'[NII]5755': [5754.64],
    ##'[NII]6548': [6548.04],  
    #'[NII]6584': [6583.46],
#
    '[OIII]4363': [4363.2],
    '[OIII]4959': [4958.91],
    '[OIII]5007': [5006.84],
#
    '[OII]3726': [3726.03],
    '[OII]7320': [7319.99],
    '[OII]7330': [7330],
#
    #'Ha6563': [6562.8],
    'Hb4861': [4861.32],
    'Hgm4340': [4340.46],
    'HI3750': [3750.15],
    #'HI3771': [3770.63],
    'HI3835': [3835.38],
    'HI4102': [4101.74],
    'HI8863': [8862.78],
    'HI9015': [9014.91],
    'HI9229': [9229.01],
    ##'OII4638': [4638.86],
    #'[ClIII]5517':[5517.71],
    #'[ClIII]5537':[5537.88] 

    }

dust_corr_all_ratios(filename, lines_to_correct, balmer_theoretical, observed_ratios, error_obs_ratios, wavelengths)


/var/folders/01/0zj4nff14gx4hd80g0nl5gbr0000gn/T/ipykernel_901/1989659135.py:65: RuntimeWarning: divide by zero encountered in divide
  (observed_flux_err / observed_flux) ** 2
/var/folders/01/0zj4nff14gx4hd80g0nl5gbr0000gn/T/ipykernel_901/1989659135.py:65: RuntimeWarning: overflow encountered in square
  (observed_flux_err / observed_flux) ** 2
/var/folders/01/0zj4nff14gx4hd80g0nl5gbr0000gn/T/ipykernel_901/1989659135.py:64: RuntimeWarning: invalid value encountered in multiply
  intrinsic_flux_err = intrinsic_flux * np.sqrt(


mean , median: 1.6340792775668167 0.2536398897867044 err: 3.771629114617543e+17 2.6397601658481618
File saved: /Users/amritasingh/LVM_lagoon_outputs/condition2_cont/lagoon_sframe_obs_corr_flux_table_dec11_newdrp_Hk.fits


/var/folders/01/0zj4nff14gx4hd80g0nl5gbr0000gn/T/ipykernel_901/1989659135.py:26: RuntimeWarning: invalid value encountered in log10
  E_B_V = 2.5 * np.log10(obs_ratio / theo_ratio) / (f_hb - f_lambda)


mean , median: 2.17036250034956 0.4360265266466984 err: 2.7430882845048864e+17 1.1591165323299935
File saved: /Users/amritasingh/LVM_lagoon_outputs/condition2_cont/lagoon_sframe_obs_corr_flux_table_dec11_newdrp_He.fits
mean , median: 1.8445594743798017 0.9350285726972001 err: 1.4272737280172938e+18 0.40317122700330155
File saved: /Users/amritasingh/LVM_lagoon_outputs/condition2_cont/lagoon_sframe_obs_corr_flux_table_dec11_newdrp_Hd.fits
mean , median: 1.8479277007580803 1.178895928119482 err: 1.5744506370703243e+24 0.3590260058011158
File saved: /Users/amritasingh/LVM_lagoon_outputs/condition2_cont/lagoon_sframe_obs_corr_flux_table_dec11_newdrp_Hg.fits
mean , median: 0.7661569923895153 1.1486074054436575 err: 2.646573853758561e+16 0.296830153537878
File saved: /Users/amritasingh/LVM_lagoon_outputs/condition2_cont/lagoon_sframe_obs_corr_flux_table_dec11_newdrp_P11.fits
mean , median: 0.9805455504457264 1.083214207727498 err: 5.432586783700055e+16 0.20919861824191383
File saved: /Users/a

In [ ]:
################################ Last modified date: Dec 1st, 24 ################################
obs_ratio = np.array([observed_ratios[line] for line in observed_ratios])
err_obs_ratio = np.array([error_obs_ratios[line] for line in error_obs_ratios])
theo_ratio = np.array([balmer_theoretical[line] for line in balmer_theoretical])
wave = np.array([wavelengths[line] for line in wavelengths])* u.AA

def dust_corr(filename, lines_to_correct):
    with fits.open(filename ) as hdul:
        table = hdul[1].data

    # Initializing the F99 extinction model
    extinction_model = F99(Rv=3.1)

    for obs_ratios, err_obs_ratios, theo_ratios, wave_array in zip (obs_ratio, err_obs_ratio, theo_ratio, wave):

        #print('wave_array, obs_ratios, err_obs_ratios, theo_ratios', wave_array, obs_ratios, err_obs_ratios, theo_ratios)

        # Intrinsic Ha/Hb ratio
        I_intrinsic = theo_ratios

        # Observed Ha/Hb ratio
        I_observed = obs_ratios
        I_observed_err = err_obs_ratios

        # Color excess
        f_lambda = extinction_model(wave_array)
        f_hb = extinction_model(wave[5])

        # Check if (f_hb - f_lambda) is zero
        if np.isclose(f_hb, f_lambda):
            print(f"Skipping iteration for wavelength {wave_array} due to zero division risk (f_hb = f_lambda).")
            continue

        try:
            E_B_V = 2.5 * np.log10(I_observed / I_intrinsic) / (f_hb - f_lambda)

            # Error on E(B-V)
            log_err_term = (1 / np.log(10)) * (I_observed_err / I_observed)
            E_B_V_err = np.abs((2.5 / (f_hb - f_lambda)) * log_err_term)

        except RuntimeError:
            print('Error during E(B-V) calculation for wavelength:', wave_array)
            continue

        print(f'wavelength: {wave_array:.3f}, E_B_V: {E_B_V.flatten()}, E_B_V_err: {E_B_V_err.flatten()}')
        print()
    
    # Create lists to store new columns
    new_columns = []

    # Copy and rename observed columns
    for key in table.columns.names:
        if 'flux' in key or 'RA' in key or 'Dec' in key:  #'wave' in key or 
            obs_col_name = f"obs_{key}"
            new_columns.append(fits.Column(name=obs_col_name, format='D', array=table[key]))

    # Loop over lines to correct
    offsets = [0, 14.19, 23.38, 31.69, 34.65, 41.97, 43.68, 51.01, 54.47, 69.07]

    for line_label in lines_to_correct:
        for i in range(10):  # Correcting up to 10 components (0 to 10)

            flux_column = f'{line_label}_flux{i}'
            flux_err_column = f'{line_label}_flux{i}_err'
            wave_column = f'{line_label}_wave{i}'

            if flux_column in table.columns.names:

                if line_label == 'NII_OII':
                    line_wavelength = 4607.16 + offsets[i]  # Using the above provided offsets

                else:
                    line_wavelength = table[wave_column][0]  # Using the observed wavelength from the table
                
                if line_wavelength == 0:  # Checkin for zero wavelength
                    # Copying the column without correction
                    int_flux_col_name = f"int_{flux_column}"
                    int_flux_err_col_name = f"int_{flux_err_column}"
                    new_columns.append(fits.Column(name=int_flux_col_name, format='D', array=table[flux_column]))
                    new_columns.append(fits.Column(name=int_flux_err_col_name, format='D', array=table[flux_err_column]))

                else:

                    f_line = extinction_model(line_wavelength * u.AA)
                    A_line = E_B_V * f_line
                    A_line_err = E_B_V_err * f_line

                    observed_flux = table[flux_column]
                    observed_flux_err = table[flux_err_column]
                    intrinsic_flux = observed_flux * 10 ** (0.4 * A_line)
                    intrinsic_flux_err = intrinsic_flux * np.sqrt((observed_flux_err / observed_flux) ** 2 + (0.4 * A_line_err * np.log(10)) ** 2)

                    # Corrected fluxes and errors
                    int_flux_col_name = f"int_{flux_column}"
                    int_flux_err_col_name = f"int_{flux_err_column}"
                    new_columns.append(fits.Column(name=int_flux_col_name, format='D', array=intrinsic_flux))
                    new_columns.append(fits.Column(name=int_flux_err_col_name, format='D', array=intrinsic_flux_err))

    
    # Adding E(B-V) and its error to the new columns
    new_columns.append(fits.Column(name='E_B_V', format='D', array=E_B_V))
    new_columns.append(fits.Column(name='E_B_V_err', format='D', array=E_B_V_err))

    # columns for the new data
    cols = fits.ColDefs(new_columns)

    # BinTableHDU object with the columns and write it to a FITS file
    hdu = fits.BinTableHDU.from_columns(cols)
    hdu.writeto(f'/Users/amritasingh/LVM_lagoon_outputs/condition2_cont/lagoon_sframe_obs_corr_flux_table_nov26_theo_ratio_{I_intrinsic:.4f}.fits', overwrite=True)

# Variable radial bins
filename = '/Users/amritasingh/LVM_lagoon_outputs/condition2_cont/lagoon_sframe_obs_flux_table_nov26.fits'

lines_to_correct = {

    'NII_OII' : [4607.16],
    #'[SII]4069': [4068.6], 
    '[SII]6717': [6716.44],
    '[SII]6731': [6730.82],

    #'[NII]5755': [5754.64],
    ##'[NII]6548': [6548.04],  
    #'[NII]6584': [6583.46],
#
    '[OIII]4363': [4363.2],
    '[OIII]4959': [4958.91],
    '[OIII]5007': [5006.84],
#
    '[OII]3726': [3726.03],
    '[OII]7320': [7319.99],
    '[OII]7330': [7330],
#
    #'Ha6563': [6562.8],
    'Hb4861': [4861.32],
    'Hgm4340': [4340.46],
    'HI3750': [3750.15],
    'HI3771': [3770.63],
    'HI3835': [3835.38],
    'HI4102': [4101.74],
    'HI8863': [8862.78],
    'HI9015': [9014.91],
    'HI9229': [9229.01],
    ##'OII4638': [4638.86],
    #
    #'[ClIII]5517':[5517.71],
    #'[ClIII]5537':[5537.88] 
    }

dust_corr(filename, lines_to_correct)


wavelength: 3750.551 Angstrom, E_B_V: [1.58202026], E_B_V_err: [0.01155372]

wavelength: 3771.097 Angstrom, E_B_V: [1.70093458], E_B_V_err: [0.0089717]

wavelength: 3836.038 Angstrom, E_B_V: [1.2586895], E_B_V_err: [0.00445559]

wavelength: 4102.763 Angstrom, E_B_V: [1.13963453], E_B_V_err: [0.00149648]

wavelength: 4341.415 Angstrom, E_B_V: [1.15469074], E_B_V_err: [0.00133203]

Skipping iteration for wavelength 4861.523461580584 Angstrom due to zero division risk (f_hb = f_lambda).
wavelength: 8863.276 Angstrom, E_B_V: [1.01538979], E_B_V_err: [0.00103137]

wavelength: 9015.356 Angstrom, E_B_V: [1.01289938], E_B_V_err: [0.00073821]

wavelength: 9229.579 Angstrom, E_B_V: [1.07643551], E_B_V_err: [0.00059008]



/var/folders/01/0zj4nff14gx4hd80g0nl5gbr0000gn/T/ipykernel_21934/737235494.py:91: RuntimeWarning: divide by zero encountered in divide
  intrinsic_flux_err = intrinsic_flux * np.sqrt((observed_flux_err / observed_flux) ** 2 + (0.4 * A_line_err * np.log(10)) ** 2)
/var/folders/01/0zj4nff14gx4hd80g0nl5gbr0000gn/T/ipykernel_21934/737235494.py:91: RuntimeWarning: overflow encountered in square
  intrinsic_flux_err = intrinsic_flux * np.sqrt((observed_flux_err / observed_flux) ** 2 + (0.4 * A_line_err * np.log(10)) ** 2)
/var/folders/01/0zj4nff14gx4hd80g0nl5gbr0000gn/T/ipykernel_21934/737235494.py:91: RuntimeWarning: invalid value encountered in multiply
  intrinsic_flux_err = intrinsic_flux * np.sqrt((observed_flux_err / observed_flux) ** 2 + (0.4 * A_line_err * np.log(10)) ** 2)


In [ ]:
################################ Last modified date: Sept 30th, 24 ################################

def dust_corr(filename, lines_to_correct):
    with fits.open(filename ) as hdul:
        table = hdul[1].data

    # Initializing the F99 extinction model
    extinction_model = F99(Rv=3.1)

    # Retrieving Hα and Hβ line fluxes and their errors
    Hgm_flux = table['Hgm4340_flux0']
    Hb_flux =  table['Hb4861_flux0']
    Hgm_flux_err = table['Hgm4340_flux0_err']
    Hb_flux_err = table['Hb4861_flux0_err']

    # Observed wavelengths of Hα and Hβ
    Hgm_wavelength = table['Hgm4340_wave0']* u.AA
    Hb_wavelength = table['Hb4861_wave0'] * u.AA

    # Intrinsic Hgm/Hβ ratio
    I_intrinsic = 0.46

    # Calculate observed Hα/Hβ ratio
    I_observed = Hgm_flux / Hb_flux
    I_observed_err = I_observed * np.sqrt((Hgm_flux_err / Hgm_flux) ** 2 + (Hb_flux_err / Hb_flux) ** 2)

    # Calculate color excess
    fHgm = extinction_model(Hgm_wavelength)
    fHb = extinction_model(Hb_wavelength)
    
    E_B_V = 2.5 * np.log10(I_observed / I_intrinsic) / (fHb - fHgm)

    # Error on color excess
    log_err_term = (1 / np.log(10)) * (I_observed_err / I_observed)
    E_B_V_err = np.abs((2.5 / (fHb - fHgm)) * log_err_term)

    E_B_V[E_B_V < 0] = 0

    print(np.nanmin(E_B_V), np.nanmin(E_B_V_err))
    
    # Create lists to store new columns
    new_columns = []

    # Copy and rename observed columns
    for key in table.columns.names:
        if 'flux' in key or 'RA' in key or 'Dec' in key:  #'wave' in key or 
            obs_col_name = f"obs_{key}"
            new_columns.append(fits.Column(name=obs_col_name, format='D', array=table[key]))

    # Loop over lines to correct
    offsets = [0, 14.19, 23.38, 31.69, 34.65, 41.97, 43.68, 51.01, 54.47, 69.07]

    for line_label in lines_to_correct:
        for i in range(10):  # Correcting up to 10 components (0 to 10)

            flux_column = f'{line_label}_flux{i}'
            flux_err_column = f'{line_label}_flux{i}_err'
            wave_column = f'{line_label}_wave{i}'

            if flux_column in table.columns.names:

                if line_label == 'NII_OII':
                    line_wavelength = 4607.16 + offsets[i]  # Using the above provided offsets

                else:
                    line_wavelength = table[wave_column][0]  # Using the observed wavelength from the table
                
                if line_wavelength == 0:  # Checkin for zero wavelength
                    # Copying the column without correction
                    int_flux_col_name = f"int_{flux_column}"
                    int_flux_err_col_name = f"int_{flux_err_column}"
                    new_columns.append(fits.Column(name=int_flux_col_name, format='D', array=table[flux_column]))
                    new_columns.append(fits.Column(name=int_flux_err_col_name, format='D', array=table[flux_err_column]))

                else:

                    f_line = extinction_model(line_wavelength * u.AA)
                    A_line = E_B_V * f_line
                    A_line_err = E_B_V_err * f_line

                    observed_flux = table[flux_column]
                    observed_flux_err = table[flux_err_column]
                    intrinsic_flux = observed_flux * 10 ** (0.4 * A_line)
                    intrinsic_flux_err = intrinsic_flux * np.sqrt((observed_flux_err / observed_flux) ** 2 + (0.4 * A_line_err * np.log(10)) ** 2)

                    # Corrected fluxes and errors
                    int_flux_col_name = f"int_{flux_column}"
                    int_flux_err_col_name = f"int_{flux_err_column}"
                    new_columns.append(fits.Column(name=int_flux_col_name, format='D', array=intrinsic_flux))
                    new_columns.append(fits.Column(name=int_flux_err_col_name, format='D', array=intrinsic_flux_err))

    
    # Adding E(B-V) and its error to the new columns
    new_columns.append(fits.Column(name='E_B_V', format='D', array=E_B_V))
    new_columns.append(fits.Column(name='E_B_V_err', format='D', array=E_B_V_err))

    # columns for the new data
    cols = fits.ColDefs(new_columns)

    # BinTableHDU object with the columns and write it to a FITS file
    hdu = fits.BinTableHDU.from_columns(cols)
    hdu.writeto(f'/Users/amritasingh/LVM_lagoon_outputs/condition2_cont/lagoon_sframe_obs_corr_flux_table_nov26_theo_ratio_{I_intrinsic}.fits', overwrite=True)

# Variable radial bins
filename = '/Users/amritasingh/LVM_lagoon_outputs/condition2_cont/lagoon_sframe_obs_flux_table_nov26.fits'

lines_to_correct = {

    'NII_OII' : [4607.16],
    #'[SII]4069': [4068.6], 
    '[SII]6717': [6716.44],
    '[SII]6731': [6730.82],

    #'[NII]5755': [5754.64],
    ##'[NII]6548': [6548.04],  
    #'[NII]6584': [6583.46],
#
    '[OIII]4363': [4363.2],
    '[OIII]4959': [4958.91],
    '[OIII]5007': [5006.84],
#
    '[OII]3726': [3726.03],
    '[OII]7320': [7319.99],
    '[OII]7330': [7330],
#
    #'Ha6563': [6562.8],
    'Hb4861': [4861.32],
    'Hgm4340': [4340.46],
    'HI3750': [3750.15],
    'HI3771': [3770.63],
    'HI3835': [3835.38],
    'HI4102': [4101.74],
    'HI8863': [8862.78],
    'HI9015': [9014.91],
    'HI9229': [9229.01],

    ##'OII4638': [4638.86],
    #
    #'[ClIII]5517':[5517.71],
    #'[ClIII]5537':[5537.88] 
    }

dust_corr(filename, lines_to_correct)


0.0 0.02538577937580335


/var/folders/01/0zj4nff14gx4hd80g0nl5gbr0000gn/T/ipykernel_9018/106236498.py:84: RuntimeWarning: divide by zero encountered in divide
  intrinsic_flux_err = intrinsic_flux * np.sqrt((observed_flux_err / observed_flux) ** 2 + (0.4 * A_line_err * np.log(10)) ** 2)
/var/folders/01/0zj4nff14gx4hd80g0nl5gbr0000gn/T/ipykernel_9018/106236498.py:84: RuntimeWarning: overflow encountered in square
  intrinsic_flux_err = intrinsic_flux * np.sqrt((observed_flux_err / observed_flux) ** 2 + (0.4 * A_line_err * np.log(10)) ** 2)
/var/folders/01/0zj4nff14gx4hd80g0nl5gbr0000gn/T/ipykernel_9018/106236498.py:84: RuntimeWarning: invalid value encountered in multiply
  intrinsic_flux_err = intrinsic_flux * np.sqrt((observed_flux_err / observed_flux) ** 2 + (0.4 * A_line_err * np.log(10)) ** 2)


In [ ]:
hdu = fits.open('/Users/amritasingh/LVM_lagoon_outputs/condition2_cont/Oct_3_lagoon_FOV_obs_corr_flux_table.fits')

data = hdu[1].data

print(np.nanmin(data['E_B_V']), np.nanmin(data['E_B_V_err']))

-8.001937396545742 0.01730229652049911


In [ ]:
################################ Test it for spatially resolved spetra  ################################

def dust_corr(filename, lines_to_correct):
    with fits.open(filename + '.fits') as hdul:
        table = hdul[1].data

    # Ensure the number of entries matches the original table
    num_entries = len(table)

    # Wavelength range (in Angstroms), this fits file is opened just to read the wavelength, no other use
    with fits.open('/Users/amritasingh/LVM_lagoon_outputs/integrated_spectrum/lagoon_integrated_spectrum.fits') as hdu:
        wave = hdu[1].data['wave'].flatten()
    wavelengths = wave * u.AA

    # Initializing the F99 extinction model
    extinction_model = F99(Rv=3.1)

    # Retrieving Hα and Hβ line fluxes and their errors
    Ha_flux = table['Ha6563_flux0']
    Hb_flux = table['Hb4861_flux0']
    
    Ha_flux_err = table['Ha6563_flux0_err']
    Hb_flux_err = table['Hb4861_flux0_err']

    # Observed wavelengths of Hα and Hβ
    Ha_wavelength = 6562.8 * u.AA
    Hb_wavelength = 4861.32 * u.AA

    # Intrinsic Hα/Hβ ratio
    I_intrinsic = 2.86

    # Calculate observed Hα/Hβ ratio
    I_observed = Ha_flux / Hb_flux
    I_observed_err = I_observed * np.sqrt((Ha_flux_err / Ha_flux) ** 2 + (Hb_flux_err / Hb_flux) ** 2)

    # Calculate color excess
    fHa = extinction_model(Ha_wavelength)
    fHb = extinction_model(Hb_wavelength)
    E_B_V = 2.5 * np.log10(I_observed / I_intrinsic) / (fHb - fHa)
    E_B_V_err = 2.5 / (I_observed * np.log(10) * (fHb - fHa)) * I_observed_err

    # Create lists to store new columns
    new_columns = []

    # Copy and rename observed columns
    for key in table.columns.names:
        if 'flux' in key or 'wave' in key or 'RA' in key or 'Dec' in key:
            obs_col_name = f"obs_{key}"
            new_columns.append(fits.Column(name=obs_col_name, format='D', array=table[key]))

    # Loop over lines to correct
    offsets = [0, 14.19, 23.38, 31.69, 34.65, 41.97, 43.68, 51.01, 54.47, 69.07]

    for line_label, wavelengths in lines_to_correct.items():
        for i in range(10):  # Correcting up to 10 components (0 to 10)

            flux_column = f'{line_label}_flux{i}'
            flux_err_column = f'{line_label}_flux{i}_err'
            wave_column = f'{line_label}_wave{i}'

            if flux_column in table.columns.names:

                if line_label == 'NII_OII':
                    line_wavelength = 4607.16 + offsets[i]  # Using the above provided offsets
                else:
                    line_wavelength = table[wave_column][0]  # Using the observed wavelength from the table
                
                if line_wavelength == 0:  # Check for zero wavelength
                    # Copying the column without correction
                    int_flux_col_name = f"int_{flux_column}"
                    int_flux_err_col_name = f"int_{flux_err_column}"
                    new_columns.append(fits.Column(name=int_flux_col_name, format='D', array=table[flux_column]))
                    new_columns.append(fits.Column(name=int_flux_err_col_name, format='D', array=table[flux_err_column]))

                else:
                    f_line = extinction_model(line_wavelength * u.AA)
                    A_line = E_B_V * f_line
                    A_line_err = E_B_V_err * f_line

                    observed_flux = table[flux_column]
                    observed_flux_err = table[flux_err_column]
                    intrinsic_flux = observed_flux * 10 ** (0.4 * A_line)
                    intrinsic_flux_err = intrinsic_flux * np.sqrt((observed_flux_err / observed_flux) ** 2 + (0.4 * A_line_err * np.log(10)) ** 2)

                    # Store corrected fluxes and errors
                    int_flux_col_name = f"int_{flux_column}"
                    int_flux_err_col_name = f"int_{flux_err_column}"
                    new_columns.append(fits.Column(name=int_flux_col_name, format='D', array=intrinsic_flux))
                    new_columns.append(fits.Column(name=int_flux_err_col_name, format='D', array=intrinsic_flux_err))


    # Add E(B-V) and its error to the new columns
    new_columns.append(fits.Column(name='E_B_V', format='D', array=E_B_V))
    new_columns.append(fits.Column(name='E_B_V_err', format='D', array=E_B_V_err))

    # Create FITS columns for the new data
    cols = fits.ColDefs(new_columns)

    # Create a BinTableHDU object with the columns and write it to a FITS file
    hdu = fits.BinTableHDU.from_columns(cols)
    hdu.writeto('/Users/amritasingh/LVM_lagoon_outputs/condition2_cont/annular_binned/region_masks/v1_unmasked_lagoon_obs_corr_table.fits', overwrite=True)

# Variable radial bins
filename = '/Users/amritasingh/LVM_lagoon_outputs/condition2_cont/annular_binned/region_masks/v1_unmasked_lagoon_obs_table'

lines_to_correct = {
    #'[SII]4069': [4068.6], 
    '[SII]6717': [6716.44],
    '[SII]6731': [6730.82],
    #'[NII]5755': [5754.64],
    #'[NII]6548': [6548.04],  
    '[NII]6584': [6583.46],
    #'[OIII]4363': [4363.2],
    #'[OIII]4959': [4958.91],
    '[OIII]5007': [5006.84],
    #'[SIII]6312': [6312.1],
    #'[SIII]9069': [9068.6],
    #'[SIII]9531': [9530.6],  
    '[OII]3726': [3726.03],
    '[OII]7320': [7319.99],
    '[OII]7330': [7330],
    #'[ClIII]5517': [5517.71],
    #'[ClIII]5537': [5537.88],
    'Ha6563': [6562.8],
    'Hb4861': [4861.32],
    #'Hgm4340': [4340.46],
    #'OII4638': [4638.86]
}

dust_corr(filename, lines_to_correct)


In [ ]:
with fits.open('/Users/amritasingh/LVM_lagoon_outputs/condition2_cont/annular_binned/region_masks/v1_unmasked_lagoon_obs_corr_table.fits') as hdu:

    print(repr(hdu[1].header))

XTENSION= 'BINTABLE'           / binary table extension                         
BITPIX  =                    8 / array data type                                
NAXIS   =                    2 / number of array dimensions                     
NAXIS1  =                 3344 / length of dimension 1                          
NAXIS2  =                 3591 / length of dimension 2                          
PCOUNT  =                    0 / number of group parameters                     
GCOUNT  =                    1 / number of groups                               
TFIELDS =                  418 / number of table fields                         
TTYPE1  = 'obs_RA  '                                                            
TFORM1  = 'D       '                                                            
TTYPE2  = 'obs_Dec '                                                            
TFORM2  = 'D       '                                                            
TTYPE3  = 'obs_Ha6563_flux0'